# Virtual Hit Finding for the Adenosine A<sub>2A</sub> Receptor from an Enamine GPCR-Targeted Library

**An end-to-end, reproducible ligand-based virtual screening workflow**

Author: Babak Mamnoon, Ph.D. — Computational Chemistry / Cheminformatics / AI-ML for Drug Discovery

---

## Project rationale

The adenosine A<sub>2A</sub> receptor (`ADORA2A`, UniProt **P29274**, ChEMBL **CHEMBL251**) is a
class-A GPCR and a validated immuno-oncology target: tumour-derived extracellular adenosine
engages A<sub>2A</sub> on tumour-infiltrating lymphocytes and suppresses effector T-cell function,
so A<sub>2A</sub> antagonism is pursued as a checkpoint-adjacent mechanism. It is also one of the
best-characterised GPCRs structurally, with a 1.8 Å thermostabilised structure
(**PDB 4EIY**) bound to the high-affinity antagonist ZM-241385 — which gives this project a
credible structure-based arm and a redocking control.

Two datasets play two distinct roles in this workflow, and they are never pooled:

| Dataset | Role | Labels? |
|---|---|---|
| **Enamine GPCR-targeted library** (53,440 plated compounds) | the *screening deck* — the compounds we rank and prioritise | none |
| **ChEMBL A<sub>2A</sub> bioactivities** | the *training/validation set* — supervised model, reference actives for similarity search, and the known-active spike used for enrichment metrics | Ki / IC50 / EC50 / Kd |

The Enamine library carries no measured A<sub>2A</sub> activity, so every predictive
score in this project is a **prospective** prediction from a model trained on ChEMBL and then
applied, frozen, to the curated library.

### A note on library–target compatibility

A profiling pass on this library (see §1.3) shows a tight, lead-like distribution
(median MW ≈ 338, cLogP ≈ 2.7, TPSA ≈ 59) dominated by neutral amide/heteroaryl chemotypes,
with only ~32% of compounds carrying an aliphatic basic nitrogen. That composition is a poor
match for aminergic receptors, whose ligands depend on a protonated amine forming a salt bridge
to Asp3.32 — but a good match for A<sub>2A</sub>, whose antagonist pharmacophore is a neutral
fused/bicyclic heteroaromatic core with an exocyclic H-bond donor–acceptor pair engaging
Asn253<sup>6.55</sup>. This is the main reason A<sub>2A</sub> was selected.

---

## Phase roadmap

| Phase | Contents | Status |
|---|---|---|
| **1** | Environment, configuration, data ingest, chemical curation & standardisation engine, drug-/lead-likeness filtering, ChEMBL A<sub>2A</sub> retrieval & bioactivity curation, leakage audit, curation visualisations, persisted artefacts | **this phase** |
| **2** | Descriptor & fingerprint generation, physicochemical/ADMET profiling, Butina & k-means clustering, PCA/UMAP chemical-space mapping, library-vs-actives space comparison | pending |
| **3** | Similarity search, supervised model (RF / gradient boosting on fingerprints + descriptors), scaffold-aware splitting, class-imbalance handling, ROC-AUC / PR-AUC / EF / BEDROC, bootstrap confidence intervals, prospective scoring of the library | pending |
| **4** | Optional AutoDock Vina structure-based arm (PDB 4EIY, ZM-241385 redocking control), transparent composite ranking, diversity-aware hit selection, final prioritised table + 2D structure grids | pending |

Each phase appends to **this same notebook**, preserving all configuration, variables, and
on-disk artefacts established earlier. By the end of Phase 4 the notebook runs top-to-bottom as
one coherent pipeline.

## How to run

* **Locally** — Python ≥ 3.9. The setup cell installs anything missing.
* **Google Colab** — runs as-is; the setup cell detects Colab and adapts. Phase 3 is CPU-viable;
  Phase 2's UMAP and Phase 4's docking benefit from more cores but do not require a GPU.
  GPU-awareness is introduced in Phase 3 only if a neural model is justified there.

Set `CFG.library_path` in the configuration cell to point at your Enamine export. CSV, `.smiles`,
and `.sdf` inputs are all accepted.

---
# Phase 1 — Data acquisition, chemical curation, and dataset construction

Phase 1 produces the two clean, documented, reproducible datasets that everything downstream
depends on. Nothing here is modelling; the goal is that no molecule reaching Phase 2 is invalid,
duplicated, mis-protonated, a salt form, a non-drug-like outlier, or — critically — a known A<sub>2A</sub>
ligand hiding in the screening deck.

**Sections**

1. **1.0** Environment, imports, configuration, reproducibility stamp
2. **1.1** Library ingest (CSV / SMILES / SDF auto-detection)
3. **1.2** Chemical curation & standardisation engine, with a per-step audit
4. **1.3** Descriptor recomputation and drug-likeness / lead-likeness filtering
5. **1.4** ChEMBL A<sub>2A</sub> bioactivity retrieval and curation
6. **1.5** Train/screen leakage audit
7. **1.6** Curation and property visualisations
8. **1.7** Persisted artefacts and Phase 1 validation gate

### Design principles applied throughout

* **Every filter is counted.** A `CurationAudit` object records molecules in, molecules out, and
  the reason for every removal at every step. Nothing is dropped silently.
* **One curation engine, two datasets.** The library and the ChEMBL compounds pass through the
  *identical* standardisation function. If they were standardised differently, an InChIKey-based
  leakage check and any fingerprint comparison between them would be meaningless.
* **Vendor descriptors are QC'd, not trusted.** Enamine ships MW/cLogP/TPSA; we recompute with
  RDKit and plot the agreement rather than assuming it.
* **Flag first, filter second.** Drug-likeness verdicts are stored as boolean columns so a filter
  can be relaxed in Phase 3 without re-running curation.

## 1.0 Environment, imports, and configuration

In [ ]:
# --- Environment setup -------------------------------------------------------
# Idempotent: installs only what is missing. Safe to re-run.
import importlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

REQUIREMENTS = {
    "rdkit":    "rdkit",                       # cheminformatics toolkit
    "pandas":   "pandas",
    "numpy":    "numpy",
    "matplotlib": "matplotlib",
    "tqdm":     "tqdm",
    "chembl_webresource_client": "chembl_webresource_client",   # ChEMBL REST client
}

missing = [pkg for mod, pkg in REQUIREMENTS.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    if not IN_COLAB:
        # Some managed/system Pythons (Debian PEP 668) need this flag; harmless elsewhere.
        try:
            subprocess.check_call(cmd)
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd + ["--break-system-packages"])
    else:
        subprocess.check_call(cmd)
    print("Install complete.")
else:
    print("All requirements already satisfied.")

print(f"Running in Google Colab: {IN_COLAB}")

In [ ]:
# --- Imports -----------------------------------------------------------------
import json, os, platform, random, textwrap, time, warnings
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger, rdBase
from rdkit.Chem import Descriptors, Draw, FilterCatalog, PandasTools
from rdkit.Chem import rdMolDescriptors as rdMD
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.QED import qed
from rdkit.Chem.Scaffolds import MurckoScaffold
from tqdm.auto import tqdm

# RDKit is extremely chatty about the sanitisation failures we are deliberately
# hunting for; we silence the global logger and capture failures ourselves.
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=FutureWarning)

VERSIONS = {
    "python":     sys.version.split()[0],
    "platform":   platform.platform(),
    "rdkit":      rdBase.rdkitVersion,
    "pandas":     pd.__version__,
    "numpy":      np.__version__,
    "matplotlib": mpl.__version__,
}
print("Environment")
print("-" * 60)
for k, v in VERSIONS.items():
    print(f"  {k:<11} {v}")

In [ ]:
# --- Configuration: the single source of truth for the whole project ---------
# Every threshold, path, and switch used in Phases 1-4 lives here. Later phases
# extend this object rather than introducing loose constants.

@dataclass
class Config:
    # ---- identity -----------------------------------------------------------
    project:        str = "A2A_Enamine_HitFinding"
    target_name:    str = "Adenosine A2A receptor"
    target_gene:    str = "ADORA2A"
    uniprot:        str = "P29274"
    chembl_target:  str = "CHEMBL251"      # verified at runtime against UniProt
    pdb_id:         str = "4EIY"           # 1.8 A, ZM-241385 bound (Phase 4)
    reference_ligand: str = "ZM-241385"

    # ---- reproducibility ----------------------------------------------------
    random_seed:    int = 42

    # ---- input --------------------------------------------------------------
    # Point this at your Enamine export. CSV / .smiles / .sdf all accepted.
    library_path:   str = "Enamine_GPCR_Library_plated_53440cmpds_20260503.csv"
    # Optional: a manually downloaded ChEMBL activity export, used if the REST
    # API is unreachable (offline, firewalled, or blocked egress).
    chembl_csv_fallback: str = "ChEMBL_A2A.csv"

    # ---- output tree --------------------------------------------------------
    out_root:       str = "outputs"

    # ---- curation switches --------------------------------------------------
    keep_largest_fragment: bool = True
    neutralise_charges:    bool = True
    canonical_tautomer:    bool = False   # correct but slow (~10x); off for 53k
    remove_isotopes:       bool = True
    # Organic-chemistry element whitelist. Anything outside this is rejected as
    # unsuitable for a small-molecule screening deck (metals, metalloids, etc.).
    allowed_elements: tuple = (
        "H", "B", "C", "N", "O", "F", "Si", "P", "S", "Cl", "Se", "Br", "I",
    )
    min_heavy_atoms: int = 12
    max_heavy_atoms: int = 60
    apply_pains:     bool = True
    apply_brenk:     bool = False   # very aggressive on lead-like space; flag only

    # ---- lead-likeness (applied to the screening deck) ----------------------
    # Deliberately looser than Lipinski: this is a lead-finding campaign, so we
    # want room for downstream optimisation rather than drug-like end-points.
    ll_mw:      tuple = (200.0, 450.0)
    ll_clogp:   tuple = (-1.0, 4.5)
    ll_tpsa:    tuple = (20.0, 140.0)
    ll_hbd_max: int = 5
    ll_hba_max: int = 10
    ll_rotb_max: int = 8
    ll_rings_min: int = 1

    # ---- ChEMBL bioactivity curation ---------------------------------------
    activity_types:   tuple = ("Ki", "IC50", "EC50", "Kd")
    allowed_relations: tuple = ("=",)          # drop >, <, ~ for regression-grade data
    assay_types:      tuple = ("B", "F")       # Binding, Functional
    min_confidence:   int = 8                  # ChEMBL assay confidence score
    # Class boundaries on pActivity (-log10 M). The 5.5-7.0 gap is discarded to
    # avoid training on molecules whose class assignment is ambiguous.
    active_cutoff:    float = 7.0              # >= 100 nM  -> active
    inactive_cutoff:  float = 5.5              # <= ~3 uM   -> inactive
    max_replicate_spread: float = 1.0          # log units; drop inconsistent duplicates

    def __post_init__(self):
        self.paths = {}
        root = Path(self.out_root)
        for key, rel in {
            "root":     ".",
            "raw":      "data/raw",
            "curated":  "data/curated",
            "figures":  "results/figures",
            "tables":   "results/tables",
            "metrics":  "results/metrics",
            "models":   "results/models",
        }.items():
            self.paths[key] = root / rel

CFG = Config()

# Deterministic seeding for everything Phase 1-4 touches.
random.seed(CFG.random_seed)
np.random.seed(CFG.random_seed)
os.environ["PYTHONHASHSEED"] = str(CFG.random_seed)

for p in CFG.paths.values():
    p.mkdir(parents=True, exist_ok=True)

print(f"Project     : {CFG.project}")
print(f"Target      : {CFG.target_name}  ({CFG.target_gene} / {CFG.uniprot} / {CFG.chembl_target})")
print(f"Seed        : {CFG.random_seed}")
print(f"Output root : {Path(CFG.out_root).resolve()}")
print("\nOutput tree:")
for k, v in CFG.paths.items():
    print(f"  {k:<8} {v}")

In [ ]:
# --- Plot style --------------------------------------------------------------
# One validated categorical palette used consistently across all four phases.
# Slots 1-3 of a colourblind-safe ordering (validated all-pairs: worst CVD
# dE 9.2, worst normal-vision dE 24.0). Legends and direct labels are always
# present, so identity is never carried by colour alone.

PALETTE = {
    "library":  "#2a78d6",   # slot 1 - blue   : the Enamine screening deck
    "active":   "#eb6834",   # slot 2 - orange : ChEMBL actives
    "inactive": "#1baf7a",   # slot 3 - aqua   : ChEMBL inactives
}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#256abf",
            "#1c5cab", "#184f95", "#104281", "#0d366b"]
INK      = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8a87"}
SURFACE  = "#fcfcfb"

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "600",
    "axes.labelsize": 9.5, "axes.labelcolor": INK["secondary"],
    "text.color": INK["primary"],
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "axes.edgecolor": "#d8d7d3", "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#ebeae6", "grid.linewidth": 0.8, "axes.grid": True,
    "axes.grid.axis": "y", "legend.frameon": False, "legend.fontsize": 8.5,
    "lines.linewidth": 2.0,
})

def save_fig(fig, name, subdir="figures"):
    """Persist a figure under the configured output tree and report the path."""
    path = CFG.paths[subdir] / f"{name}.png"
    fig.savefig(path)
    print(f"  saved  {path}")
    return path

print("Plot style configured. Palette:")
for k, v in PALETTE.items():
    print(f"  {k:<9} {v}")

In [ ]:
# --- Reproducibility stamp ---------------------------------------------------
# Written once per run so any artefact can be traced back to the exact
# environment and configuration that produced it.

RUN_STAMP = {
    "project":     CFG.project,
    "run_utc":     datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "phase":       1,
    "seed":        CFG.random_seed,
    "in_colab":    IN_COLAB,
    "versions":    VERSIONS,
    "config":      {k: (list(v) if isinstance(v, tuple) else v)
                    for k, v in asdict(CFG).items() if k != "paths"},
}
stamp_path = CFG.paths["metrics"] / "run_manifest.json"
stamp_path.write_text(json.dumps(RUN_STAMP, indent=2))
print(f"Run manifest -> {stamp_path}")
print(f"Run UTC      : {RUN_STAMP['run_utc']}")

## 1.1 Library ingest

Enamine exports are inconsistent across formats, and each quirk below is a real one observed in
the supplied files rather than defensive boilerplate:

* The **CSV** carries an Excel `sep=,` directive on line 1, which `pandas` otherwise reads as a
  one-column header. The loader sniffs for it and skips it.
* The **`.smiles`** file is the same table but **tab**-delimited, despite the extension implying a
  bare SMILES list.
* The **SDF** (≈124 MB here) holds the same molecules with properties as SD tags. It is supported
  but is not needed for Phase 1: the CSV columns are identical, and parsing 124 MB of connection
  tables to obtain data already present in an 8 MB CSV is wasted effort. The SDF becomes useful in
  Phase 4 if you want vendor-supplied conformers.

The loader normalises heterogeneous column spellings to a canonical internal schema so the rest of
the notebook never touches a vendor-specific name.

In [ ]:
# --- Canonical internal schema ----------------------------------------------
# Vendor column name (lower-cased, stripped) -> internal name.
COLUMN_ALIASES = {
    "smiles": "smiles", "canonical_smiles": "smiles", "structure": "smiles",
    "catalog id": "compound_id", "catalog_id": "compound_id", "id": "compound_id",
    "idnumber": "compound_id", "molecule_chembl_id": "compound_id",
    "mw": "vendor_mw", "mw (desalted)": "vendor_mw_desalted",
    "clogp": "vendor_clogp", "logp": "vendor_clogp",
    "logs": "vendor_logs",
    "hbd": "vendor_hbd", "hba": "vendor_hba", "tpsa": "vendor_tpsa",
    "rotbonds": "vendor_rotb",
    "analogsfromreal": "real_analogs_url",
}

def _normalise_columns(df):
    """Map vendor column spellings onto the canonical internal schema."""
    renamed = {}
    for col in df.columns:
        key = str(col).strip().lower()
        renamed[col] = COLUMN_ALIASES.get(key, str(col).strip().lower().replace(" ", "_"))
    return df.rename(columns=renamed)


def load_library(path):
    """
    Load an Enamine export from CSV, .smiles (tab-delimited), or .sdf.

    Returns a DataFrame with at minimum `smiles` and `compound_id` columns,
    plus any vendor descriptors present in the source.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Library not found: {path}\n"
            f"Set CFG.library_path to your Enamine export "
            f"(.csv, .smiles, or .sdf) and re-run this cell."
        )
    suffix = path.suffix.lower()
    print(f"Reading {path.name}  ({path.stat().st_size / 1e6:.1f} MB, format: {suffix})")

    if suffix in (".csv", ".txt", ".smi", ".smiles", ".tsv"):
        # Detect an Excel 'sep=' directive on the first line.
        with open(path, "r", errors="replace") as fh:
            first = fh.readline()
        skiprows = 1 if first.strip().lower().startswith("sep=") else 0
        if skiprows:
            print("  note: Excel 'sep=' directive on line 1 -> skipping it")
        # Sniff the delimiter from the header line.
        with open(path, "r", errors="replace") as fh:
            for _ in range(skiprows):
                fh.readline()
            header = fh.readline()
        sep = "\t" if header.count("\t") > header.count(",") else ","
        print(f"  delimiter: {'TAB' if sep == chr(9) else 'comma'}")
        df = pd.read_csv(path, sep=sep, skiprows=skiprows)

    elif suffix == ".sdf":
        print("  note: SDF parsing is slower and memory-hungry; the CSV export "
              "carries identical fields if available.")
        df = PandasTools.LoadSDF(path, smilesName="SMILES",
                                molColName=None, includeFingerprints=False)
    else:
        raise ValueError(f"Unsupported input format: {suffix}")

    df = _normalise_columns(df)

    if "smiles" not in df.columns:
        raise ValueError(f"No SMILES column found. Columns present: {list(df.columns)}")
    if "compound_id" not in df.columns:
        print("  note: no catalogue ID column -> generating sequential IDs")
        df["compound_id"] = [f"LIB{i:07d}" for i in range(len(df))]

    df["compound_id"] = df["compound_id"].astype(str)
    return df

In [ ]:
# --- Load the screening deck -------------------------------------------------
lib_raw = load_library(CFG.library_path)

print(f"\nLoaded {len(lib_raw):,} records x {lib_raw.shape[1]} columns")
print(f"Columns: {list(lib_raw.columns)}")
print("\nFirst 5 records:")
display(lib_raw.head())

print("\nMissing values per column:")
miss = lib_raw.isna().sum()
print(miss[miss > 0].to_string() if miss.any() else "  none")

# Vendor descriptor summary, for comparison against RDKit values in section 1.3.
vendor_cols = [c for c in lib_raw.columns if c.startswith("vendor_")]
if vendor_cols:
    print("\nVendor-supplied descriptors (as shipped):")
    display(lib_raw[vendor_cols].describe().T.round(2))

## 1.2 Chemical curation and standardisation

Curation is the step that most often silently determines whether a virtual screen produces
anything real. Two molecules that differ only in tautomer, protonation state, salt counter-ion, or
an isotope label will receive different fingerprints and therefore different model scores, despite
being the same compound. Worse, an undetected duplicate that appears in both a training and a test
split inflates every performance metric reported afterwards.

The pipeline below is applied **identically** to the Enamine library and to the ChEMBL compounds:

| # | Step | Rationale |
|---|---|---|
| 1 | **Parse & sanitise** | Rejects syntactically invalid SMILES, bad valences, and un-kekulisable aromatic systems. |
| 2 | **`Cleanup`** | RDKit's normalisation of functional-group representation — nitro groups, N-oxides, azides, and similar groups have multiple legal encodings that must be unified. |
| 3 | **Largest organic fragment** | Discards salt counter-ions and solvates; the pharmacologically relevant species is the largest covalent unit. |
| 4 | **Element whitelist** | Rejects metals/metalloids outside organic-chemistry space, which are not tractable for fingerprints, docking, or synthesis from a plated library. |
| 5 | **Isotope stripping** | Normalises deuterated/¹³C-labelled analogues onto their parent graph. |
| 6 | **Normalise** | Applies RDKit's transform catalogue so equivalent functional groups converge on one representation. |
| 7 | **Uncharge** | Neutralises where chemically reasonable, so a carboxylate and its acid are one entity. Formal charges that cannot be neutralised (quaternary ammonium) are preserved. |
| 8 | **Canonical tautomer** *(optional)* | The most rigorous de-duplication, but ~10× slower. Off by default for 53k compounds; enable via `CFG.canonical_tautomer`. |
| 9 | **Heavy-atom bounds** | Removes fragments too small to be a hit and molecules too large for a lead-finding deck. |
| 10 | **Structural alerts** | PAINS filters remove frequent-hitter chemotypes — promiscuous redox cyclers, Michael acceptors, and metal chelators that produce assay artefacts rather than pharmacology. |
| 11 | **InChIKey de-duplication** | The final identity check, performed on the *standardised* graph so it catches duplicates that raw SMILES comparison misses. |

**Audit discipline.** Every removal is attributed to exactly one step and counted. The resulting
waterfall is reported as a table and plotted in §1.6, so the reviewer can see precisely where any
compound was lost.

In [ ]:
# --- The curation audit ------------------------------------------------------
class CurationAudit:
    """
    Records the molecule count before and after every curation step, together
    with the reason for each removal. Produces the waterfall table and figure.
    """

    def __init__(self, name, n_start):
        self.name = name
        self.n_start = n_start
        self.steps = [{"step": "Input records", "n_removed": 0, "n_retained": n_start}]
        self.rejections = []          # (identifier, step, detail)

    def record(self, step, n_retained, detail=""):
        prev = self.steps[-1]["n_retained"]
        self.steps.append({
            "step": step,
            "n_removed": prev - n_retained,
            "n_retained": n_retained,
            "detail": detail,
        })
        return self

    def reject(self, identifier, step, detail=""):
        self.rejections.append({"compound_id": identifier, "step": step, "detail": detail})

    def table(self):
        df = pd.DataFrame(self.steps)
        df["pct_of_input"] = (100 * df["n_retained"] / max(self.n_start, 1)).round(2)
        df["pct_removed"]  = (100 * df["n_removed"] / max(self.n_start, 1)).round(2)
        return df[["step", "n_removed", "pct_removed", "n_retained", "pct_of_input", "detail"]] \
               if "detail" in df else df

    def rejection_table(self):
        return pd.DataFrame(self.rejections) if self.rejections else pd.DataFrame(
            columns=["compound_id", "step", "detail"])

    def summary(self):
        end = self.steps[-1]["n_retained"]
        print(f"\nCuration audit — {self.name}")
        print("=" * 92)
        print(self.table().to_string(index=False))
        print("-" * 92)
        print(f"Retained {end:,} of {self.n_start:,} "
              f"({100 * end / max(self.n_start, 1):.2f}%)  |  "
              f"removed {self.n_start - end:,}")
        return self


print("CurationAudit ready.")

In [ ]:
# --- Structural-alert catalogues --------------------------------------------
def build_alert_catalog(pains=True, brenk=False):
    """Assemble an RDKit FilterCatalog from the requested alert sets."""
    params = FilterCatalog.FilterCatalogParams()
    sets = []
    if pains:
        for cat in ("PAINS_A", "PAINS_B", "PAINS_C"):
            params.AddCatalog(getattr(FilterCatalog.FilterCatalogParams.FilterCatalogs, cat))
            sets.append(cat)
    if brenk:
        params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
        sets.append("BRENK")
    catalog = FilterCatalog.FilterCatalog(params)
    print(f"Alert catalogue: {', '.join(sets) if sets else 'none'} "
          f"({catalog.GetNumEntries()} substructure filters)")
    return catalog

ALERT_CATALOG = build_alert_catalog(pains=CFG.apply_pains, brenk=CFG.apply_brenk)

# A separate flag-only catalogue: BRENK hits are annotated but never dropped,
# because BRENK is aggressive enough to delete legitimate lead-like matter.
BRENK_CATALOG = build_alert_catalog(pains=False, brenk=True)

In [ ]:
# --- Standardisation primitives ---------------------------------------------
_LFC       = rdMolStandardize.LargestFragmentChooser()
_UNCHARGER = rdMolStandardize.Uncharger()
_NORMALIZER = rdMolStandardize.Normalizer()
_TAUTOMER  = rdMolStandardize.TautomerEnumerator()
_ALLOWED   = set(CFG.allowed_elements)


def standardize_smiles(smiles, cfg=CFG):
    """
    Apply the full standardisation cascade to one SMILES string.

    Returns
    -------
    (canonical_smiles, inchikey, reason)
        On success `reason` is None. On rejection the first two are None and
        `reason` names the step that rejected the molecule, so the audit can
        attribute every loss.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None, None, "empty_smiles"

    # 1. parse + sanitise
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, "unparseable_smiles"
    if mol.GetNumAtoms() == 0:
        return None, None, "empty_molecule"

    try:
        # 2. functional-group cleanup
        mol = rdMolStandardize.Cleanup(mol)

        # 3. largest covalent fragment (salt / solvate removal)
        if cfg.keep_largest_fragment:
            mol = _LFC.choose(mol)
        if mol is None or mol.GetNumAtoms() == 0:
            return None, None, "no_fragment_after_salt_strip"

        # 4. element whitelist
        for atom in mol.GetAtoms():
            if atom.GetSymbol() not in _ALLOWED:
                return None, None, f"disallowed_element:{atom.GetSymbol()}"

        # 5. isotope stripping
        if cfg.remove_isotopes:
            for atom in mol.GetAtoms():
                atom.SetIsotope(0)

        # 6. normalise representation
        mol = _NORMALIZER.normalize(mol)

        # 7. neutralise where chemically reasonable
        if cfg.neutralise_charges:
            mol = _UNCHARGER.uncharge(mol)

        # 8. canonical tautomer (optional, slow)
        if cfg.canonical_tautomer:
            mol = _TAUTOMER.Canonicalize(mol)

        Chem.SanitizeMol(mol)

        # 9. size bounds
        n_heavy = mol.GetNumHeavyAtoms()
        if n_heavy < cfg.min_heavy_atoms:
            return None, None, f"too_few_heavy_atoms:{n_heavy}"
        if n_heavy > cfg.max_heavy_atoms:
            return None, None, f"too_many_heavy_atoms:{n_heavy}"

        # 10. structural alerts
        if ALERT_CATALOG.GetNumEntries() and ALERT_CATALOG.HasMatch(mol):
            entry = ALERT_CATALOG.GetFirstMatch(mol)
            name = entry.GetDescription() if entry is not None else "unspecified"
            return None, None, f"structural_alert:{name}"

        # 11. identity
        canon = Chem.MolToSmiles(mol)
        key = Chem.MolToInchiKey(mol)
        if not key:
            return None, None, "inchikey_generation_failed"
        return canon, key, None

    except Exception as exc:                      # noqa: BLE001 - any RDKit failure
        return None, None, f"standardisation_error:{type(exc).__name__}"


# Quick self-test on molecules with known, deliberate defects.
_TESTS = [
    ("CC(=O)Oc1ccccc1C(=O)O",                  "aspirin, clean"),
    ("CC(=O)Oc1ccccc1C(=O)[O-].[Na+]",         "sodium salt -> should neutralise + desalt"),
    ("[13CH3]C(=O)Oc1ccccc1C(=O)O",            "13C label -> should strip isotope"),
    ("c1ccccc1(C)(C)",                          "bad valence -> should reject"),
    ("not_a_smiles",                            "garbage -> should reject"),
    ("CCO",                                     "ethanol -> too few heavy atoms"),
    ("[Pt](Cl)(Cl)(N)N",                        "cisplatin -> disallowed element"),
]
print("Self-test of standardize_smiles()")
print("-" * 78)
for smi, note in _TESTS:
    c, k, r = standardize_smiles(smi)
    verdict = f"OK   {c}" if c else f"REJECT  ({r})"
    print(f"  {note:<45} {verdict}")

In [ ]:
# --- Batch curation ----------------------------------------------------------
def curate_dataframe(df, name, id_col="compound_id", smiles_col="smiles",
                     cfg=CFG, progress=True):
    """
    Run the full curation cascade over a DataFrame of molecules.

    Returns
    -------
    (curated_df, audit)
        `curated_df` gains `std_smiles` and `inchikey` columns and carries only
        molecules that survived every step, de-duplicated on InChIKey.
    """
    audit = CurationAudit(name, len(df))
    work = df.copy()

    # Pre-pass: drop records with no SMILES at all.
    work = work[work[smiles_col].notna() & (work[smiles_col].astype(str).str.strip() != "")]
    audit.record("Non-empty SMILES", len(work))

    canon, keys, reasons = [], [], []
    it = tqdm(work[smiles_col].astype(str), desc=f"Standardising {name}",
              unit="mol", disable=not progress)
    for smi in it:
        c, k, r = standardize_smiles(smi, cfg=cfg)
        canon.append(c); keys.append(k); reasons.append(r)

    work["std_smiles"] = canon
    work["inchikey"]   = keys
    work["_reason"]    = reasons

    # Attribute every rejection to its step, in cascade order, so the waterfall
    # reflects the true sequence rather than lumping all failures together.
    STEP_ORDER = [
        ("Valid, parseable structure",
         ("empty_smiles", "unparseable_smiles", "empty_molecule")),
        ("Salt / solvate stripped",
         ("no_fragment_after_salt_strip",)),
        ("Element whitelist",
         ("disallowed_element",)),
        ("Heavy-atom bounds",
         ("too_few_heavy_atoms", "too_many_heavy_atoms")),
        ("Structural alerts (PAINS)",
         ("structural_alert",)),
        ("Standardisation completed",
         ("standardisation_error", "inchikey_generation_failed")),
    ]
    for label, prefixes in STEP_ORDER:
        mask = work["_reason"].notna() & work["_reason"].astype(str).str.startswith(prefixes)
        for cid, rsn in zip(work.loc[mask, id_col], work.loc[mask, "_reason"]):
            audit.reject(cid, label, rsn)
        work = work[~mask]
        audit.record(label, len(work))

    # Anything left with a reason is an unclassified failure; surface it loudly
    # rather than letting it pass silently.
    leftover = work["_reason"].notna()
    if leftover.any():
        print(f"  warning: {leftover.sum()} unclassified rejection reason(s): "
              f"{work.loc[leftover, '_reason'].unique()[:5]}")
        work = work[~leftover]
        audit.record("Unclassified failures", len(work))

    # InChIKey de-duplication on the standardised graph.
    n_before = len(work)
    dupe_mask = work.duplicated(subset="inchikey", keep="first")
    for cid in work.loc[dupe_mask, id_col]:
        audit.reject(cid, "InChIKey de-duplication", "duplicate_structure")
    work = work[~dupe_mask]
    audit.record("Unique structures (InChIKey)", len(work),
                 f"{n_before - len(work)} duplicate structures collapsed")

    work = work.drop(columns=["_reason"]).reset_index(drop=True)
    return work, audit


print("curate_dataframe() ready.")

In [ ]:
# --- Curate the screening deck ----------------------------------------------
t0 = time.time()
lib_curated, lib_audit = curate_dataframe(lib_raw, "Enamine GPCR library")
print(f"\nElapsed: {time.time() - t0:.1f} s")

lib_audit.summary()

# What exactly was removed, and why — grouped so the reasons are inspectable.
rej = lib_audit.rejection_table()
if len(rej):
    print("\nRejections by step:")
    print(rej["step"].value_counts().to_string())
    print("\nTop 15 specific reasons:")
    print(rej["detail"].str.split(":").str[0].value_counts().head(15).to_string())
    print("\nMost frequent PAINS classes triggered:")
    pains = rej.loc[rej["detail"].str.startswith("structural_alert"), "detail"]
    if len(pains):
        print(pains.str.replace("structural_alert:", "", regex=False)
                   .value_counts().head(10).to_string())
else:
    print("\nNo molecules were rejected.")

## 1.3 Descriptor recomputation and lead-likeness filtering

Two things happen here.

**Descriptors are recomputed from the standardised structures.** Vendor descriptors are computed on
the vendor's own representation, before our desalting and neutralisation, and with a different
cLogP implementation. Carrying them forward would mean filtering on values that do not correspond
to the molecules we are actually screening. We recompute with RDKit and then *plot the agreement*
(§1.6) as a QC check — the correlation should be tight, and a systematic offset in cLogP is
expected and informative rather than alarming.

**Filters are lead-like, not drug-like.** This is deliberate. A hit-finding campaign should deliver
matter with room to grow during optimisation: medicinal-chemistry elaboration typically adds
50–150 Da and 1–2 log units of lipophilicity between hit and candidate, so a hit sitting at the
Lipinski ceiling is already a dead end. The bounds applied are therefore tighter on the upper end
than Ro5 (MW ≤ 450, cLogP ≤ 4.5) while remaining permissive enough not to bias against the
A<sub>2A</sub> antagonist chemotype.

An A<sub>2A</sub>-specific note: the canonical antagonist pharmacophore is a neutral, planar,
fused bicyclic heteroaromatic system (triazolotriazine, triazoloquinazoline, aminopyrimidine)
bearing an exocyclic amine that donates to Asn253<sup>6.55</sup>. Filtering hard on HBD would
discard exactly the compounds we want, so `ll_hbd_max` is set permissively at 5.

Every verdict is stored as a boolean column. The hard filter applied at the end of this section
uses `passes_leadlike`, but Phase 3 can relax any individual criterion without re-running §1.2.

In [ ]:
# --- Descriptor computation --------------------------------------------------
DESCRIPTORS = {
    "mw":          Descriptors.MolWt,
    "clogp":       Descriptors.MolLogP,
    "tpsa":        Descriptors.TPSA,
    "hbd":         Descriptors.NumHDonors,
    "hba":         Descriptors.NumHAcceptors,
    "rotb":        Descriptors.NumRotatableBonds,
    "heavy_atoms": lambda m: m.GetNumHeavyAtoms(),
    "rings":       rdMD.CalcNumRings,
    "aromatic_rings": rdMD.CalcNumAromaticRings,
    "heteroatoms": rdMD.CalcNumHeteroatoms,
    "fsp3":        rdMD.CalcFractionCSP3,
    "stereocentres": lambda m: rdMD.CalcNumAtomStereoCenters(m),
    "formal_charge": lambda m: Chem.GetFormalCharge(m),
    "qed":         qed,
}

# SMARTS for an aliphatic basic nitrogen: neutral sp3 N that is not an amide,
# nitrile, aromatic, or aniline nitrogen. Used for the library-target
# compatibility argument in the project rationale.
BASIC_N = Chem.MolFromSmarts(
    "[$([NX3;H2,H1,H0;!$(N[#6]=[O,N,S]);!$(N#*);!$(N=*);!$(Nc);!$(N[S,P]=O)]);!$([N+])]"
)

def compute_descriptors(df, smiles_col="std_smiles", progress=True):
    """Compute RDKit descriptors, Murcko scaffold, and alert flags."""
    rows = []
    it = tqdm(df[smiles_col], desc="Descriptors", unit="mol", disable=not progress)
    for smi in it:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:                       # cannot happen post-curation
            rows.append({k: np.nan for k in DESCRIPTORS})
            continue
        rec = {}
        for name, fn in DESCRIPTORS.items():
            try:
                rec[name] = fn(mol)
            except Exception:                 # noqa: BLE001
                rec[name] = np.nan
        rec["has_basic_amine"] = bool(mol.HasSubstructMatch(BASIC_N))
        rec["brenk_alert"] = bool(BRENK_CATALOG.HasMatch(mol))
        try:
            rec["murcko_scaffold"] = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        except Exception:                     # noqa: BLE001
            rec["murcko_scaffold"] = ""
        rows.append(rec)
    return pd.concat([df.reset_index(drop=True),
                      pd.DataFrame(rows)], axis=1)


t0 = time.time()
lib_desc = compute_descriptors(lib_curated)
print(f"\nComputed {len(DESCRIPTORS)} descriptors for {len(lib_desc):,} molecules "
      f"in {time.time() - t0:.1f} s")
display(lib_desc[list(DESCRIPTORS)].describe().T.round(2))

print(f"\nAliphatic basic nitrogen present : "
      f"{100 * lib_desc['has_basic_amine'].mean():.1f}% of the library")
print(f"BRENK alert (flagged, not removed): "
      f"{100 * lib_desc['brenk_alert'].mean():.1f}%")
print(f"Unique Murcko scaffolds           : "
      f"{lib_desc['murcko_scaffold'].nunique():,} "
      f"({lib_desc['murcko_scaffold'].nunique() / len(lib_desc):.2f} per molecule)")

In [ ]:
# --- Lead-likeness assessment -----------------------------------------------
def assess_druglikeness(df, cfg=CFG):
    """
    Add boolean verdict columns. Nothing is removed here — the hard filter is
    applied separately so criteria stay independently adjustable downstream.
    """
    out = df.copy()

    # Lipinski Ro5 (reported for reference, not used as the screening filter).
    ro5 = pd.concat([
        out["mw"] <= 500, out["clogp"] <= 5,
        out["hbd"] <= 5,  out["hba"] <= 10,
    ], axis=1)
    out["ro5_violations"] = (~ro5).sum(axis=1)
    out["passes_ro5"] = out["ro5_violations"] <= 1

    # Veber oral-bioavailability criteria.
    out["passes_veber"] = (out["rotb"] <= 10) & (out["tpsa"] <= 140)

    # The operative lead-likeness filter for this campaign.
    criteria = {
        "ll_mw":    out["mw"].between(*cfg.ll_mw),
        "ll_clogp": out["clogp"].between(*cfg.ll_clogp),
        "ll_tpsa":  out["tpsa"].between(*cfg.ll_tpsa),
        "ll_hbd":   out["hbd"] <= cfg.ll_hbd_max,
        "ll_hba":   out["hba"] <= cfg.ll_hba_max,
        "ll_rotb":  out["rotb"] <= cfg.ll_rotb_max,
        "ll_rings": out["rings"] >= cfg.ll_rings_min,
    }
    for name, mask in criteria.items():
        out[name] = mask
    out["passes_leadlike"] = pd.concat(criteria.values(), axis=1).all(axis=1)

    # Per-criterion attrition, so it is obvious which bound is doing the work.
    print("Lead-likeness criteria (evaluated independently on the curated deck)")
    print("-" * 72)
    bounds = {
        "ll_mw":    f"MW in {cfg.ll_mw}",
        "ll_clogp": f"cLogP in {cfg.ll_clogp}",
        "ll_tpsa":  f"TPSA in {cfg.ll_tpsa}",
        "ll_hbd":   f"HBD <= {cfg.ll_hbd_max}",
        "ll_hba":   f"HBA <= {cfg.ll_hba_max}",
        "ll_rotb":  f"RotB <= {cfg.ll_rotb_max}",
        "ll_rings": f"rings >= {cfg.ll_rings_min}",
    }
    for name, mask in criteria.items():
        print(f"  {bounds[name]:<26} pass {mask.sum():>7,}  "
              f"fail {(~mask).sum():>7,}  ({100 * mask.mean():5.1f}% pass)")
    print("-" * 72)
    print(f"  {'ALL criteria':<26} pass {out['passes_leadlike'].sum():>7,}  "
          f"fail {(~out['passes_leadlike']).sum():>7,}  "
          f"({100 * out['passes_leadlike'].mean():5.1f}% pass)")
    print(f"\n  Lipinski Ro5 (<=1 violation): {100 * out['passes_ro5'].mean():.1f}% pass")
    print(f"  Veber:                        {100 * out['passes_veber'].mean():.1f}% pass")
    return out


lib_desc = assess_druglikeness(lib_desc)

# Apply the hard filter and close out the library audit.
n_before = len(lib_desc)
LIBRARY = lib_desc[lib_desc["passes_leadlike"]].reset_index(drop=True)
lib_audit.record("Lead-likeness filter", len(LIBRARY),
                 f"MW/cLogP/TPSA/HBD/HBA/RotB/rings bounds")
print(f"\nScreening deck: {len(LIBRARY):,} compounds "
      f"(removed {n_before - len(LIBRARY):,} at the lead-likeness gate)")
lib_audit.summary()

## 1.4 ChEMBL A<sub>2A</sub> bioactivity retrieval and curation

This section builds the labelled dataset that trains and validates the Phase 3 model.

**Target resolution is done by UniProt accession, not by a hard-coded ID.** `CHEMBL251` is the
expected answer for human `ADORA2A`, but resolving `P29274` at runtime and asserting the match
means the notebook fails loudly if ChEMBL re-maps an identifier, rather than silently training on
the wrong receptor.

**The raw pull is cached to disk.** ChEMBL is versioned and grows between releases, so an
uncached notebook is not reproducible: re-running it six months later silently changes the training
set and every metric derived from it. The raw response is written to `data/raw/` with the run
timestamp and reused on subsequent runs. Delete that file to force a refresh.

**Bioactivity curation decisions**, each of which is a defensible choice rather than a default:

* **Activity types** — Ki, IC50, EC50, Kd only. Percent-inhibition and unnormalised readouts are
  not convertible to a common scale.
* **Relation `=` only.** Censored values (`>10000`) are genuine information that a compound is
  inactive, but they cannot be placed on a continuous pActivity axis without inventing a number.
  They are excluded here; Phase 3 discusses recovering them as qualitative inactives.
* **Assay confidence ≥ 8** restricts to assays ChEMBL maps confidently to a single protein target,
  excluding whole-organism and homologous-family assays.
* **Replicate aggregation by median**, with duplicates whose spread exceeds 1 log unit discarded
  entirely. A compound measured at both 5 nM and 5 µM across two labs is not usefully labelled,
  and averaging it manufactures a false middle value.
* **A discarded activity gap.** Actives are pActivity ≥ 7.0 (≤ 100 nM), inactives ≤ 5.5 (≥ ~3 µM),
  and the band between them is *excluded from training entirely*. Assigning a hard label to a
  300 nM compound teaches the model a boundary that the data does not support.

In [ ]:
# --- Target resolution -------------------------------------------------------
def resolve_chembl_target(uniprot, expected=None):
    """
    Resolve a UniProt accession to a ChEMBL target ID, preferring SINGLE PROTEIN.
    Returns (target_chembl_id, pref_name) or (None, None) if unreachable.
    """
    try:
        from chembl_webresource_client.new_client import new_client
        hits = list(new_client.target.filter(
            target_components__accession=uniprot, organism="Homo sapiens"))
    except Exception as exc:                       # noqa: BLE001
        print(f"  ChEMBL target lookup failed: {type(exc).__name__}: {exc}")
        return None, None

    if not hits:
        print(f"  no ChEMBL target found for {uniprot}")
        return None, None

    print(f"  ChEMBL targets mapped to {uniprot}:")
    for h in hits:
        print(f"    {h['target_chembl_id']:<14} {h.get('target_type','?'):<22} "
              f"{h.get('pref_name','?')}")

    single = [h for h in hits if h.get("target_type") == "SINGLE PROTEIN"]
    chosen = (single or hits)[0]
    tid, name = chosen["target_chembl_id"], chosen.get("pref_name", "")

    if expected and tid != expected:
        print(f"  WARNING: resolved {tid} but configuration expects {expected}. "
              f"Verify before proceeding.")
    else:
        print(f"  resolved -> {tid}  ({name})")
    return tid, name


print(f"Resolving {CFG.target_gene} / {CFG.uniprot} ...")
resolved_id, resolved_name = resolve_chembl_target(CFG.uniprot, expected=CFG.chembl_target)
TARGET_ID = resolved_id or CFG.chembl_target
print(f"\nUsing target: {TARGET_ID}")

In [ ]:
# --- Manual ChEMBL export loader --------------------------------------------
# The ChEMBL web interface emits a SEMICOLON-delimited, quoted CSV whose column
# names differ from the REST API's. It also has quirks that will silently
# destroy a dataset if unhandled -- see normalise_chembl_fields() below.
WEB_EXPORT_ALIASES = {
    "molecule_chembl_id": "molecule_chembl_id",
    "smiles": "canonical_smiles", "canonical_smiles": "canonical_smiles",
    "standard_type": "standard_type", "standard_relation": "standard_relation",
    "standard_value": "standard_value", "standard_units": "standard_units",
    "pchembl_value": "pchembl_value", "assay_chembl_id": "assay_chembl_id",
    "assay_type": "assay_type", "assay_description": "assay_description",
    "assay_organism": "assay_organism", "document_chembl_id": "document_chembl_id",
    "data_validity_comment": "data_validity_comment",
    "comment": "activity_comment", "activity_comment": "activity_comment",
    "potential_duplicate": "potential_duplicate",
    "assay_confidence_score": "confidence_score",
    "confidence_score": "confidence_score",
    "target_chembl_id": "target_chembl_id", "target_organism": "target_organism",
    "target_type": "target_type", "molecule_name": "molecule_name",
    "molecule_max_phase": "max_phase", "document_year": "document_year",
}


def load_chembl_export(path):
    """Load a ChEMBL activity export downloaded from the web interface."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"ChEMBL export not found: {path}")

    # Sniff the delimiter from the header rather than assuming: REST-derived
    # CSVs are comma-delimited, web-interface exports are semicolon-delimited.
    with open(path, "r", errors="replace") as fh:
        header = fh.readline()
    sep = max([",", ";", "\t"], key=header.count)
    sep_name = {",": "comma", ";": "semicolon", "\t": "TAB"}[sep]
    print(f"  delimiter sniffed: {sep_name}")
    # NOTE: `low_memory` is invalid with engine='python', so the delimiter is
    # sniffed above and the fast C engine is used here instead.
    df = pd.read_csv(path, sep=sep, low_memory=False, quotechar='"',
                     on_bad_lines="warn")
    n_orig_cols = len(df.columns)
    df.columns = [str(c).strip().strip('"').lower().replace(" ", "_")
                   .replace("-", "_").replace("#", "num_")
                  for c in df.columns]
    df = df.rename(columns={c: WEB_EXPORT_ALIASES[c]
                            for c in df.columns if c in WEB_EXPORT_ALIASES})

    print(f"Loaded {len(df):,} rows x {n_orig_cols} columns from {path.name}")
    required = ["molecule_chembl_id", "canonical_smiles", "standard_type",
                "standard_value", "standard_units"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Export missing required column(s): {missing}\n"
                         f"Columns found: {sorted(df.columns)[:30]}")
    return df


def normalise_chembl_fields(df):
    """
    Repair the web export's encoding quirks BEFORE any filter touches the data.

    Each of these has been observed in a real CHEMBL251 export and each would
    corrupt the dataset silently if left alone:

    1. `standard_relation` arrives as "'='" -- the operator wrapped in LITERAL
       single quotes. A filter for "=" then matches zero rows and deletes the
       entire dataset while reporting a perfectly normal-looking audit. This is
       the single most dangerous quirk in the export.
    2. Empty strings and the literal text "None" appear where NULL is meant.
    3. `data_validity_comment` mixes WARNINGS ("Outside typical range") with a
       QUALITY MARK ("Manually validated"). Dropping every non-null comment
       therefore discards the highest-confidence records in the file.
    """
    d = df.copy()
    d.columns = [c.strip().lower() for c in d.columns]

    # (2) normalise null-ish strings across all object columns
    for c in d.select_dtypes(include="object").columns:
        d[c] = d[c].replace({"": np.nan, "None": np.nan, "none": np.nan,
                             "NaN": np.nan, "null": np.nan})

    # (1) strip the literal quoting from the relation operator
    if "standard_relation" in d:
        before = set(d["standard_relation"].dropna().unique())
        d["standard_relation"] = (d["standard_relation"].astype(str)
                                  .str.strip().str.strip("'\"").str.strip())
        d.loc[d["standard_relation"].isin(["nan", ""]), "standard_relation"] = np.nan
        after = set(d["standard_relation"].dropna().unique())
        if before != after:
            print(f"  repaired standard_relation quoting: "
                  f"{sorted(before)} -> {sorted(after)}")

    # (3) separate genuine validity WARNINGS from the quality mark
    VALIDITY_WARNINGS = {
        "outside typical range", "potential transcription error",
        "potential author error", "non standard unit for type",
        "author confirmed error", "potential missing data",
    }
    if "data_validity_comment" in d:
        dvc = d["data_validity_comment"].astype(str).str.strip().str.lower()
        d["_validity_warning"] = dvc.isin(VALIDITY_WARNINGS)
        kept = d["data_validity_comment"].notna() & ~d["_validity_warning"]
        if kept.any():
            print(f"  retained {int(kept.sum())} record(s) flagged "
                  f"'Manually validated' (a quality mark, not a warning)")
    else:
        d["_validity_warning"] = False

    for col in ("standard_value", "pchembl_value", "confidence_score",
                "potential_duplicate"):
        if col in d:
            d[col] = pd.to_numeric(d[col], errors="coerce")
    return d


print("load_chembl_export() and normalise_chembl_fields() ready.")

In [ ]:
# --- Activity retrieval, with on-disk cache ---------------------------------
ACTIVITY_FIELDS = [
    "molecule_chembl_id", "canonical_smiles", "standard_type", "standard_relation",
    "standard_value", "standard_units", "pchembl_value", "assay_chembl_id",
    "assay_type", "assay_description", "confidence_score", "target_chembl_id",
    "document_chembl_id", "data_validity_comment", "activity_comment",
]
RAW_CHEMBL = CFG.paths["raw"] / f"chembl_activities_{TARGET_ID}.csv"


def fetch_chembl_activities(target_id, cache_path, activity_types=CFG.activity_types):
    """
    Download A2A bioactivities from the ChEMBL REST API, caching the raw result.

    Returns (DataFrame, source) where source is 'cache', 'api', 'user_csv',
    or 'synthetic'.
    """
    if cache_path.exists():
        df = pd.read_csv(cache_path, low_memory=False)
        print(f"Loaded {len(df):,} cached activity records from {cache_path}")
        print("  (delete this file to force a fresh download)")
        return df, "cache"

    # A configured manual export takes precedence over the network.
    if CFG.chembl_csv_fallback and Path(CFG.chembl_csv_fallback).exists():
        print(f"Using the configured manual export "
              f"({CFG.chembl_csv_fallback}) in preference to the API.")
        return load_chembl_export(CFG.chembl_csv_fallback), "user_csv"

    print(f"Querying ChEMBL for {target_id} activities ({', '.join(activity_types)}) ...")
    try:
        from chembl_webresource_client.new_client import new_client
        res = new_client.activity.filter(
            target_chembl_id=target_id,
            standard_type__in=list(activity_types),
        ).only(ACTIVITY_FIELDS)
        records = list(tqdm(res, desc="Downloading", unit="rec"))
        df = pd.DataFrame.from_records(records)
        if len(df):
            df.to_csv(cache_path, index=False)
            print(f"Downloaded {len(df):,} records -> cached at {cache_path}")
            return df, "api"
        print("  query returned no records")
    except Exception as exc:                       # noqa: BLE001
        print(f"  ChEMBL API unavailable: {type(exc).__name__}: {exc}")

    # Fallback 1: a user-supplied ChEMBL export.
    if CFG.chembl_csv_fallback and Path(CFG.chembl_csv_fallback).exists():
        return load_chembl_export(CFG.chembl_csv_fallback), "user_csv"

    return pd.DataFrame(), "unavailable"


chembl_raw, CHEMBL_SOURCE = fetch_chembl_activities(TARGET_ID, RAW_CHEMBL)
print(f"\nSource: {CHEMBL_SOURCE}   shape: {chembl_raw.shape}")
if len(chembl_raw):
    display(chembl_raw.head(3))

In [ ]:
# --- Offline smoke-test fallback --------------------------------------------
# If ChEMBL could not be reached AND no manual export was supplied, build a
# clearly-labelled SYNTHETIC stand-in so the notebook remains executable
# end-to-end. This exists to exercise the plumbing only.
#
#   *** ANY MODEL METRIC COMPUTED ON SYNTHETIC DATA IS MEANINGLESS. ***
#
# Every downstream cell checks the SYNTHETIC_MODE flag and banners its output.

SYNTHETIC_MODE = False

if not len(chembl_raw):
    SYNTHETIC_MODE = True
    print("!" * 78)
    print("!!  ChEMBL UNAVAILABLE - ENTERING SYNTHETIC SMOKE-TEST MODE")
    print("!!")
    print("!!  No real A2A bioactivity data was retrieved. A synthetic stand-in")
    print("!!  is being generated from the screening library with RANDOM labels,")
    print("!!  purely so the pipeline can be executed and validated.")
    print("!!")
    print("!!  Results produced in this mode carry NO scientific meaning.")
    print("!!  To obtain real results, run with network access to ebi.ac.uk, or")
    print("!!  set CFG.chembl_csv_fallback to a manual ChEMBL export.")
    print("!" * 78)

    rng = np.random.default_rng(CFG.random_seed)
    n_syn = 2000
    pool = lib_curated.sample(min(n_syn, len(lib_curated)),
                              random_state=CFG.random_seed).reset_index(drop=True)
    chembl_raw = pd.DataFrame({
        "molecule_chembl_id": [f"SYNTH{i:06d}" for i in range(len(pool))],
        "canonical_smiles":   pool["std_smiles"].values,
        "standard_type":      rng.choice(["Ki", "IC50"], len(pool)),
        "standard_relation":  "=",
        "standard_value":     10 ** rng.normal(2.6, 1.1, len(pool)),   # nM
        "standard_units":     "nM",
        "pchembl_value":      np.nan,
        "assay_chembl_id":    "SYNTH_ASSAY",
        "assay_type":         "B",
        "assay_description":  "SYNTHETIC - not real data",
        "confidence_score":   9,
        "target_chembl_id":   TARGET_ID,
        "document_chembl_id": "SYNTH_DOC",
        "data_validity_comment": np.nan,
        "activity_comment":   np.nan,
    })
    print(f"\nSynthetic stand-in: {len(chembl_raw):,} records")
else:
    print("Real ChEMBL data present - synthetic mode NOT engaged.")

RUN_STAMP["synthetic_mode"] = SYNTHETIC_MODE
RUN_STAMP["chembl_source"] = CHEMBL_SOURCE

In [ ]:
# --- Bioactivity curation ----------------------------------------------------
UNIT_TO_NM = {"nM": 1.0, "uM": 1e3, "µM": 1e3, "mM": 1e6, "M": 1e9,
              "pM": 1e-3, "nmol/L": 1.0, "umol/L": 1e3}


def curate_bioactivities(df, cfg=CFG):
    """
    Filter, normalise, aggregate, and label ChEMBL bioactivity records.

    Returns (per_compound_df, audit).
    """
    audit = CurationAudit("ChEMBL bioactivities", len(df))

    # Repair the export's encoding quirks BEFORE any filter runs. Skipping this
    # is how a curation pipeline deletes an entire dataset and still prints a
    # plausible audit table.
    d = normalise_chembl_fields(df)

    # 0. Target sanity — a report-card export should be single-target, but an
    #    assembled or concatenated file may not be. Verify rather than trust.
    if "target_chembl_id" in d and d["target_chembl_id"].notna().any():
        wrong = d["target_chembl_id"] != TARGET_ID
        if wrong.any():
            print(f"  dropping {int(wrong.sum())} record(s) against a target "
                  f"other than {TARGET_ID}: "
                  f"{d.loc[wrong, 'target_chembl_id'].unique()[:5]}")
        d = d[~wrong]
    audit.record(f"Target is {TARGET_ID}", len(d))

    # 1. Genuine data-validity WARNINGS only. 'Manually validated' is a quality
    #    mark and its records are the best in the file — they are kept.
    d = d[~d["_validity_warning"]]
    audit.record("No data-validity warning", len(d))

    # 1b. ChEMBL's own potential-duplicate flag.
    if "potential_duplicate" in d:
        dup = d["potential_duplicate"].fillna(0) == 1
        if dup.any():
            print(f"  dropping {int(dup.sum())} record(s) flagged by ChEMBL as "
                  f"potential duplicates")
        d = d[~dup]
    audit.record("Not flagged as a potential duplicate", len(d))

    # 1c. Assay organism. A human target can carry assays run in bovine or
    #     porcine tissue; those measure a different receptor sequence and do not
    #     belong in a human-A2A training set. NaN is retained (the export often
    #     omits it for recombinant human assays).
    if "assay_organism" in d:
        non_human = d["assay_organism"].notna() & (d["assay_organism"] != "Homo sapiens")
        if non_human.any():
            print(f"  dropping {int(non_human.sum())} non-human assay record(s): "
                  f"{sorted(d.loc[non_human, 'assay_organism'].unique())[:6]}")
        d = d[~non_human]
    audit.record("Human or unspecified assay organism", len(d))

    # 2. Activity type
    d = d[d["standard_type"].isin(cfg.activity_types)]
    audit.record(f"Activity type in {list(cfg.activity_types)}", len(d))

    # 3. Relation — '=' only, so every value sits on a real continuous axis
    if "standard_relation" in d:
        d = d[d["standard_relation"].isin(cfg.allowed_relations)]
    audit.record(f"Relation in {list(cfg.allowed_relations)}", len(d))

    # 4. Assay type
    if "assay_type" in d:
        d = d[d["assay_type"].isin(cfg.assay_types)]
    audit.record(f"Assay type in {list(cfg.assay_types)}", len(d))

    # 5. Assay confidence. Web-interface exports frequently omit this column;
    #    when absent the filter is SKIPPED, and that is stated rather than
    #    passed over, because the target/organism/assay-type filters above are
    #    doing the equivalent work in its place.
    if "confidence_score" in d and d["confidence_score"].notna().any():
        d = d[d["confidence_score"].fillna(0) >= cfg.min_confidence]
        audit.record(f"Confidence score >= {cfg.min_confidence}", len(d))
    else:
        print(f"  note: no confidence-score column in this export — filter "
              f"SKIPPED.\n        Target/organism/assay-type filters above "
              f"substitute for it.")
        audit.record("Confidence score (not available — skipped)", len(d))

    # 6. Usable value + structure
    d = d[d["standard_value"].notna() & (d["standard_value"] > 0)]
    d = d[d["canonical_smiles"].notna()]
    audit.record("Positive value and SMILES present", len(d))

    # 7. Unit normalisation to nM
    d["_factor"] = d["standard_units"].map(UNIT_TO_NM)
    unmapped = d["_factor"].isna()
    if unmapped.any():
        print(f"  dropping {unmapped.sum()} record(s) with unconvertible units: "
              f"{d.loc[unmapped, 'standard_units'].unique()[:8]}")
    d = d[~unmapped]
    d["value_nm"] = d["standard_value"] * d["_factor"]
    audit.record("Units convertible to nM", len(d))

    # 8. Physically plausible range: 0.01 nM to 1 mM. Values outside this are
    #    almost always unit or transcription errors rather than real affinities.
    d = d[(d["value_nm"] >= 0.01) & (d["value_nm"] <= 1e6)]
    audit.record("Value in 0.01 nM - 1 mM", len(d))

    # 9. pActivity
    d["pactivity"] = -np.log10(d["value_nm"] * 1e-9)

    # 10. Aggregate replicates per compound, rejecting inconsistent ones
    grp = d.groupby("molecule_chembl_id")
    agg = grp.agg(
        canonical_smiles=("canonical_smiles", "first"),
        pactivity=("pactivity", "median"),
        pactivity_std=("pactivity", "std"),
        pactivity_min=("pactivity", "min"),
        pactivity_max=("pactivity", "max"),
        n_measurements=("pactivity", "size"),
        activity_types=("standard_type", lambda s: "|".join(sorted(set(s)))),
        n_documents=("document_chembl_id", "nunique"),
    ).reset_index()
    audit.record("Aggregated to unique compounds", len(agg),
                 f"{len(d):,} measurements -> {len(agg):,} compounds")

    spread = (agg["pactivity_max"] - agg["pactivity_min"]).fillna(0)
    inconsistent = spread > cfg.max_replicate_spread
    if inconsistent.any():
        print(f"  discarding {inconsistent.sum()} compound(s) whose replicate "
              f"spread exceeds {cfg.max_replicate_spread} log units")
    agg = agg[~inconsistent]
    audit.record(f"Replicate spread <= {cfg.max_replicate_spread} log", len(agg))

    # 11. Class assignment, with the ambiguous middle band discarded
    agg["activity_class"] = np.select(
        [agg["pactivity"] >= cfg.active_cutoff, agg["pactivity"] <= cfg.inactive_cutoff],
        ["active", "inactive"], default="intermediate")
    n_mid = (agg["activity_class"] == "intermediate").sum()
    print(f"  excluding {n_mid} compound(s) in the ambiguous "
          f"{cfg.inactive_cutoff}-{cfg.active_cutoff} pActivity band")
    labelled = agg[agg["activity_class"] != "intermediate"].copy()
    labelled["label"] = (labelled["activity_class"] == "active").astype(int)
    audit.record("Unambiguous class label", len(labelled),
                 f"{n_mid} intermediate compounds excluded")

    return labelled.reset_index(drop=True), audit


chembl_act, chembl_audit = curate_bioactivities(chembl_raw)
chembl_audit.summary()

if len(chembl_act):
    print("\nClass balance:")
    vc = chembl_act["activity_class"].value_counts()
    for cls, n in vc.items():
        print(f"  {cls:<10} {n:>6,}  ({100 * n / len(chembl_act):5.1f}%)")
    if {"active", "inactive"} <= set(vc.index):
        print(f"  imbalance ratio (inactive:active) = "
              f"{vc.get('inactive', 0) / max(vc.get('active', 1), 1):.2f} : 1")
    print("\npActivity distribution:")
    print(chembl_act["pactivity"].describe().round(2).to_string())

In [ ]:
# --- Structural curation of the ChEMBL compounds ----------------------------
# Deliberately the SAME engine as the library. Without this, the InChIKey
# leakage check in 1.5 and every library-vs-actives comparison in Phases 2-3
# would be comparing differently-standardised graphs.

chembl_act = chembl_act.rename(columns={"canonical_smiles": "smiles",
                                        "molecule_chembl_id": "compound_id"})

ACTIVES, chembl_struct_audit = curate_dataframe(
    chembl_act, "ChEMBL A2A compounds (structural)")
chembl_struct_audit.summary()

# Descriptors on the same footing as the library, for the Phase 2 comparison.
ACTIVES = compute_descriptors(ACTIVES)
ACTIVES = assess_druglikeness(ACTIVES)

print(f"\nLabelled training set: {len(ACTIVES):,} compounds")
if len(ACTIVES):
    print(ACTIVES["activity_class"].value_counts().to_string())

## 1.5 Train/screen leakage audit

A question that is easy to skip and expensive to get wrong: **are any compounds in the Enamine
screening deck already known A<sub>2A</sub> ligands in ChEMBL?**

It matters in two directions:

* If a library compound is in the training set, its Phase 3 prediction is memorisation, not
  prospective inference. Ranking it as a top hit would be circular.
* Known actives that appear in the library are, on the other hand, exactly what is needed as a
  **spiked positive control** for enrichment factor and BEDROC in Phase 3 — provided they are held
  out of training.

Because both datasets went through the identical standardisation cascade, an InChIKey join is a
valid identity test. We also compare Murcko scaffolds, which catches the softer case of a library
compound that is a close analogue rather than an exact match — relevant to interpreting Phase 3
similarity scores.

In [ ]:
# --- Exact-structure overlap -------------------------------------------------
lib_keys = set(LIBRARY["inchikey"])
act_keys = set(ACTIVES["inchikey"])
overlap_keys = lib_keys & act_keys

print(f"Screening deck        : {len(lib_keys):,} unique structures")
print(f"ChEMBL labelled set   : {len(act_keys):,} unique structures")
print(f"Exact structure overlap: {len(overlap_keys):,}")

OVERLAP = pd.DataFrame(columns=["compound_id", "inchikey", "pactivity", "activity_class"])
if overlap_keys:
    OVERLAP = (LIBRARY[LIBRARY["inchikey"].isin(overlap_keys)]
               [["compound_id", "inchikey", "std_smiles"]]
               .merge(ACTIVES[["inchikey", "compound_id", "pactivity", "activity_class"]],
                      on="inchikey", suffixes=("_lib", "_chembl")))
    print("\nOverlapping compounds (these must be held out of Phase 3 training "
          "and can serve as spiked positives):")
    display(OVERLAP.head(20))
    print(f"\n  of which active: {(OVERLAP['activity_class'] == 'active').sum()}")
else:
    print("\nNo exact overlap. The screen is fully prospective — which also means")
    print("Phase 3 enrichment metrics must be computed on a held-out ChEMBL split")
    print("rather than on spiked library actives.")

# Mark the library so downstream phases cannot accidentally treat these as novel.
LIBRARY["in_chembl_trainset"] = LIBRARY["inchikey"].isin(act_keys)

# --- Scaffold-level overlap (analogues, not identities) ---------------------
lib_scaf = set(LIBRARY["murcko_scaffold"]) - {""}
act_scaf = set(ACTIVES["murcko_scaffold"]) - {""}
shared_scaf = lib_scaf & act_scaf
print(f"\nMurcko scaffolds — library {len(lib_scaf):,} | "
      f"ChEMBL {len(act_scaf):,} | shared {len(shared_scaf):,}")
if act_scaf:
    print(f"  {100 * len(shared_scaf) / len(act_scaf):.1f}% of ChEMBL scaffolds "
          f"are represented in the library")
LIBRARY["scaffold_in_chembl"] = LIBRARY["murcko_scaffold"].isin(act_scaf)
print(f"  {LIBRARY['scaffold_in_chembl'].sum():,} library compounds "
      f"({100 * LIBRARY['scaffold_in_chembl'].mean():.1f}%) share a scaffold "
      f"with a known A2A ligand")

## 1.6 Curation and property visualisations

Five figures, each answering a specific question:

1. **Curation waterfalls** — where were compounds lost, and to which step?
2. **Vendor vs RDKit descriptor agreement** — is the recomputation consistent with the vendor's,
   and where does it systematically differ?
3. **Property distributions, library vs ChEMBL actives** — does the screening deck actually cover
   the property space that A<sub>2A</sub> ligands occupy? A library that does not overlap the
   actives in MW/cLogP/TPSA cannot produce hits regardless of model quality.
4. **pActivity distribution** — where do the class boundaries fall relative to the data, and how
   much is discarded in the ambiguous band?
5. **Lead-likeness attrition** — which single criterion removes the most matter?

All figures use one validated, colourblind-safe palette (worst-pair CVD ΔE 9.2, normal-vision
ΔE 24.0) with legends and direct labels throughout, so no comparison depends on colour alone.

In [ ]:
# --- Figure 1: curation waterfalls ------------------------------------------
def plot_waterfall(audit, ax, color_ramp=SEQ_BLUE):
    """Horizontal retention waterfall with direct value labels."""
    t = audit.table()
    labels = t["step"].tolist()
    retained = t["n_retained"].tolist()
    removed = t["n_removed"].tolist()
    y = np.arange(len(labels))[::-1]

    # Ordinal ramp: never lighter than step 250 on a light surface.
    cols = [color_ramp[min(2 + i, len(color_ramp) - 1)] for i in range(len(labels))]

    ax.barh(y, retained, color=cols, height=0.62, edgecolor=SURFACE, linewidth=2)
    for yi, r, rem in zip(y, retained, removed):
        ax.text(r + max(retained) * 0.012, yi, f"{r:,}", va="center", ha="left",
                fontsize=8.5, color=INK["primary"], fontweight="600")
        if rem:
            ax.text(r - max(retained) * 0.012, yi, f"-{rem:,}", va="center", ha="right",
                    fontsize=7.5, color="#ffffff", fontweight="600")
    ax.set_yticks(y)
    ax.set_yticklabels([textwrap.fill(l, 30) for l in labels], fontsize=8)
    ax.set_xlabel("Compounds retained")
    ax.set_xlim(0, max(retained) * 1.16)
    ax.grid(axis="x", alpha=0.7); ax.grid(axis="y", visible=False)
    ax.set_title(f"{audit.name}\n{retained[-1]:,} of {retained[0]:,} retained "
                 f"({100 * retained[-1] / max(retained[0], 1):.1f}%)", loc="left")
    return ax


fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6))
plot_waterfall(lib_audit, axes[0])
plot_waterfall(chembl_struct_audit, axes[1])
fig.suptitle("Curation waterfall — every removal attributed to a single step",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig01_curation_waterfall")
plt.show()

In [ ]:
# --- Figure 2: vendor vs RDKit descriptor agreement -------------------------
pairs = [("vendor_mw", "mw", "Molecular weight (Da)"),
         ("vendor_clogp", "clogp", "cLogP"),
         ("vendor_tpsa", "tpsa", "TPSA (Å²)")]
pairs = [p for p in pairs if p[0] in LIBRARY.columns]

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(4.4 * len(pairs), 4.2))
    axes = np.atleast_1d(axes)
    samp = LIBRARY.sample(min(6000, len(LIBRARY)), random_state=CFG.random_seed)
    for ax, (vcol, rcol, label) in zip(axes, pairs):
        x = pd.to_numeric(samp[vcol], errors="coerce")
        y = samp[rcol]
        ok = x.notna() & y.notna()
        x, y = x[ok], y[ok]
        ax.scatter(x, y, s=6, alpha=0.18, color=PALETTE["library"],
                   edgecolors="none", rasterized=True)
        lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], color=INK["secondary"], lw=1.2, ls="--",
                label="identity")
        r = np.corrcoef(x, y)[0, 1]
        bias = (y - x).mean()
        ax.set_xlabel(f"Vendor {label}"); ax.set_ylabel(f"RDKit {label}")
        ax.set_title(f"{label}\nr = {r:.4f}   mean offset = {bias:+.2f}", loc="left")
        ax.legend(loc="lower right")
        ax.grid(alpha=0.7)
    fig.suptitle("Descriptor QC — vendor values vs RDKit recomputation on "
                 "standardised structures",
                 x=0.01, ha="left", fontsize=12, fontweight="600")
    fig.tight_layout()
    save_fig(fig, "fig02_vendor_vs_rdkit_descriptors")
    plt.show()
else:
    print("No vendor descriptor columns present — skipping QC scatter.")

In [ ]:
# --- Figure 3: property distributions, library vs ChEMBL actives ------------
PROPS = [("mw", "Molecular weight (Da)"), ("clogp", "cLogP"),
         ("tpsa", "TPSA (Å²)"), ("hbd", "H-bond donors"),
         ("hba", "H-bond acceptors"), ("rotb", "Rotatable bonds"),
         ("fsp3", "Fraction sp³ carbon"), ("qed", "QED drug-likeness")]

groups = [("Enamine library", LIBRARY, PALETTE["library"])]
if len(ACTIVES):
    a = ACTIVES[ACTIVES["activity_class"] == "active"]
    i = ACTIVES[ACTIVES["activity_class"] == "inactive"]
    if len(a): groups.append(("ChEMBL actives", a, PALETTE["active"]))
    if len(i): groups.append(("ChEMBL inactives", i, PALETTE["inactive"]))

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, (col, label) in zip(axes.ravel(), PROPS):
    allv = pd.concat([g[col].dropna() for _, g, _ in groups])
    if allv.empty:
        ax.set_visible(False); continue
    lo, hi = np.nanpercentile(allv, [0.5, 99.5])
    bins = np.linspace(lo, hi, 36)
    for name, g, colr in groups:
        v = g[col].dropna()
        if v.empty: continue
        ax.hist(v, bins=bins, density=True, histtype="step", lw=2,
                color=colr, label=f"{name} (n={len(v):,})")
        ax.axvline(v.median(), color=colr, lw=1.0, ls=":", alpha=0.85)
    ax.set_xlabel(label); ax.set_ylabel("Density")
    ax.set_title(label, loc="left")
    ax.grid(alpha=0.7)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(groups),
           bbox_to_anchor=(0.5, -0.03), fontsize=9.5)
fig.suptitle("Physicochemical property coverage — does the deck span the space "
             "A2A ligands occupy?  (dotted line = median)",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig03_property_distributions")
plt.show()

In [ ]:
# --- Figure 4: pActivity distribution with class boundaries ----------------
if len(ACTIVES) and ACTIVES["pactivity"].notna().any():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.4),
                                  gridspec_kw={"width_ratios": [1.6, 1]})

    # Full distribution BEFORE the intermediate band was dropped, so the reader
    # can see how much was discarded and why.
    allp = chembl_act["pactivity"] if "pactivity" in chembl_act else ACTIVES["pactivity"]
    bins = np.linspace(np.floor(allp.min()), np.ceil(allp.max()), 46)
    ax.hist(allp, bins=bins, color=SEQ_BLUE[3], edgecolor=SURFACE, linewidth=1.2)
    ax.axvspan(CFG.inactive_cutoff, CFG.active_cutoff, color="#ebeae6",
               zorder=0, label=f"discarded band ({CFG.inactive_cutoff}–{CFG.active_cutoff})")
    ax.axvline(CFG.active_cutoff, color=PALETTE["active"], lw=2)
    ax.axvline(CFG.inactive_cutoff, color=PALETTE["inactive"], lw=2)
    ymax = ax.get_ylim()[1]
    ax.text(CFG.active_cutoff + 0.08, ymax * 0.92, f"active\n≥ {CFG.active_cutoff}",
            color=PALETTE["active"], fontsize=9, fontweight="600", va="top")
    ax.text(CFG.inactive_cutoff - 0.08, ymax * 0.92, f"inactive\n≤ {CFG.inactive_cutoff}",
            color="#0f7a55", fontsize=9, fontweight="600", va="top", ha="right")
    ax.set_xlabel("pActivity  (−log₁₀ M)"); ax.set_ylabel("Compounds")
    ax.set_title(f"A2A bioactivity distribution (n = {len(allp):,} compounds)", loc="left")
    ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.78))
    ax.grid(alpha=0.7)

    # Retained class balance
    vc = ACTIVES["activity_class"].value_counts()
    names = [n for n in ("active", "inactive") if n in vc.index]
    vals = [vc[n] for n in names]
    cols = [PALETTE["active"] if n == "active" else PALETTE["inactive"] for n in names]
    bars = ax2.bar(names, vals, color=cols, width=0.55, edgecolor=SURFACE, linewidth=2)
    for b, v in zip(bars, vals):
        ax2.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom",
                 fontsize=10, fontweight="600", color=INK["primary"])
    ax2.set_ylabel("Compounds")
    ax2.set_ylim(0, max(vals) * 1.16)
    ax2.set_title(f"Retained class balance  "
                  f"({vals[0] / max(vals[-1], 1):.2f} : 1)" if len(vals) > 1
                  else "Retained class balance", loc="left")
    ax2.grid(alpha=0.7)

    fig.suptitle("Activity labelling — boundaries, discarded band, and class balance",
                 x=0.01, ha="left", fontsize=12, fontweight="600")
    fig.tight_layout()
    save_fig(fig, "fig04_activity_distribution")
    plt.show()
else:
    print("No pActivity data available — skipping activity figure.")

In [ ]:
# --- Figure 5: lead-likeness attrition --------------------------------------
crit_labels = {
    "ll_mw":    f"MW {CFG.ll_mw[0]:.0f}–{CFG.ll_mw[1]:.0f}",
    "ll_clogp": f"cLogP {CFG.ll_clogp[0]:.1f}–{CFG.ll_clogp[1]:.1f}",
    "ll_tpsa":  f"TPSA {CFG.ll_tpsa[0]:.0f}–{CFG.ll_tpsa[1]:.0f}",
    "ll_hbd":   f"HBD ≤ {CFG.ll_hbd_max}",
    "ll_hba":   f"HBA ≤ {CFG.ll_hba_max}",
    "ll_rotb":  f"RotB ≤ {CFG.ll_rotb_max}",
    "ll_rings": f"rings ≥ {CFG.ll_rings_min}",
}
fails = {crit_labels[c]: int((~lib_desc[c]).sum()) for c in crit_labels}
fails = dict(sorted(fails.items(), key=lambda kv: kv[1]))

fig, ax = plt.subplots(figsize=(8.4, 4.4))
y = np.arange(len(fails))
vals = list(fails.values())
ax.barh(y, vals, color=PALETTE["library"], height=0.6,
        edgecolor=SURFACE, linewidth=2)
for yi, v in zip(y, vals):
    ax.text(v + max(vals) * 0.015, yi, f"{v:,}  ({100 * v / len(lib_desc):.1f}%)",
            va="center", fontsize=8.5, color=INK["primary"], fontweight="600")
ax.set_yticks(y); ax.set_yticklabels(list(fails))
ax.set_xlabel("Compounds failing this criterion (evaluated independently)")
ax.set_xlim(0, max(vals) * 1.30 if max(vals) else 1)
ax.grid(axis="x", alpha=0.7); ax.grid(axis="y", visible=False)
ax.set_title(f"Lead-likeness attrition — which bound does the work?\n"
             f"{lib_desc['passes_leadlike'].sum():,} of {len(lib_desc):,} "
             f"pass all criteria simultaneously", loc="left")
fig.tight_layout()
save_fig(fig, "fig05_leadlikeness_attrition")
plt.show()

In [ ]:
# --- Figure 6: a look at the actual molecules -------------------------------
# A sanity check no statistic replaces: draw a random sample of the curated deck
# and confirm they look like plausible, synthesisable screening compounds.
samp = LIBRARY.sample(min(12, len(LIBRARY)), random_state=CFG.random_seed)
mols = [Chem.MolFromSmiles(s) for s in samp["std_smiles"]]
legends = [f"{r.compound_id}\nMW {r.mw:.0f} · cLogP {r.clogp:.1f} · QED {r.qed:.2f}"
           for r in samp.itertuples()]
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(260, 200),
                           legends=legends, useSVG=False)
grid_path = CFG.paths["figures"] / "fig06_curated_library_sample.png"
with open(grid_path, "wb") as fh:
    fh.write(img.data if hasattr(img, "data") else img)
print(f"  saved  {grid_path}")
img

## 1.7 Persisted artefacts and the Phase 1 validation gate

Everything Phase 2 needs is written to disk, so the notebook can be resumed from here without
re-running curation. The validation gate that follows is a real set of assertions, not a summary:
if any invariant that later phases depend on is violated, Phase 1 fails loudly here rather than
producing a quietly wrong answer three phases later.

In [ ]:
# --- Persist artefacts -------------------------------------------------------
LIB_OUT   = CFG.paths["curated"] / "library_curated.csv"
ACT_OUT   = CFG.paths["curated"] / "chembl_a2a_labelled.csv"
OVER_OUT  = CFG.paths["curated"] / "library_chembl_overlap.csv"

LIBRARY.to_csv(LIB_OUT, index=False)
ACTIVES.to_csv(ACT_OUT, index=False)
OVERLAP.to_csv(OVER_OUT, index=False)

# Audit trails
lib_audit.table().to_csv(CFG.paths["tables"] / "audit_library.csv", index=False)
chembl_audit.table().to_csv(CFG.paths["tables"] / "audit_chembl_bioactivity.csv", index=False)
chembl_struct_audit.table().to_csv(CFG.paths["tables"] / "audit_chembl_structural.csv",
                                   index=False)
lib_audit.rejection_table().to_csv(CFG.paths["tables"] / "rejections_library.csv", index=False)

# Phase 1 metrics, machine-readable
PHASE1_METRICS = {
    "library_input":          int(lib_audit.n_start),
    "library_curated":        int(len(lib_curated)),
    "library_final":          int(len(LIBRARY)),
    "library_retention_pct":  round(100 * len(LIBRARY) / max(lib_audit.n_start, 1), 2),
    "library_unique_scaffolds": int(LIBRARY["murcko_scaffold"].nunique()),
    "library_pct_basic_amine": round(100 * LIBRARY["has_basic_amine"].mean(), 1),
    "chembl_source":          CHEMBL_SOURCE,
    "chembl_synthetic_mode":  bool(SYNTHETIC_MODE),
    "chembl_measurements_in": int(chembl_audit.n_start),
    "chembl_labelled":        int(len(ACTIVES)),
    "chembl_actives":         int((ACTIVES["activity_class"] == "active").sum()),
    "chembl_inactives":       int((ACTIVES["activity_class"] == "inactive").sum()),
    "train_screen_overlap":   int(len(OVERLAP)),
    "shared_scaffolds":       int(len(shared_scaf)),
}
(CFG.paths["metrics"] / "phase1_metrics.json").write_text(
    json.dumps(PHASE1_METRICS, indent=2))
RUN_STAMP["phase1_metrics"] = PHASE1_METRICS
(CFG.paths["metrics"] / "run_manifest.json").write_text(json.dumps(RUN_STAMP, indent=2))

print("Artefacts written")
print("-" * 70)
for p in sorted(Path(CFG.out_root).rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# --- Phase 1 validation gate ------------------------------------------------
# Hard invariants that Phases 2-4 rely on. Any failure stops the notebook here.

def validate_phase1():
    checks, failures = [], []

    def check(name, condition, detail=""):
        ok = bool(condition)
        checks.append((name, ok, detail))
        if not ok:
            failures.append(f"{name} — {detail}")

    # Structure and identity integrity
    check("Library is non-empty", len(LIBRARY) > 0, f"n={len(LIBRARY)}")
    check("No missing standardised SMILES",
          LIBRARY["std_smiles"].notna().all(),
          f"{LIBRARY['std_smiles'].isna().sum()} missing")
    check("No missing InChIKeys",
          LIBRARY["inchikey"].notna().all(),
          f"{LIBRARY['inchikey'].isna().sum()} missing")
    check("Library InChIKeys unique",
          not LIBRARY["inchikey"].duplicated().any(),
          f"{LIBRARY['inchikey'].duplicated().sum()} duplicates")
    check("Compound IDs unique",
          not LIBRARY["compound_id"].duplicated().any(),
          f"{LIBRARY['compound_id'].duplicated().sum()} duplicates")

    # Every stored SMILES must re-parse — guards against writing a broken deck.
    sub = LIBRARY["std_smiles"].sample(min(2000, len(LIBRARY)),
                                       random_state=CFG.random_seed)
    n_bad = sum(Chem.MolFromSmiles(s) is None for s in sub)
    check("Stored SMILES re-parse (2k sample)", n_bad == 0, f"{n_bad} failed")

    # Filter post-conditions
    check("All library compounds pass lead-likeness",
          LIBRARY["passes_leadlike"].all(),
          f"{(~LIBRARY['passes_leadlike']).sum()} violations")
    check("MW within configured bounds",
          LIBRARY["mw"].between(*CFG.ll_mw).all(), "out-of-range MW present")
    check("No descriptor NaNs in modelling columns",
          LIBRARY[["mw", "clogp", "tpsa", "hbd", "hba", "rotb", "qed"]]
              .notna().all().all(), "NaNs present")

    # Training set
    check("Labelled set is non-empty", len(ACTIVES) > 0, f"n={len(ACTIVES)}")
    check("Both classes present",
          ACTIVES["activity_class"].nunique() >= 2,
          f"classes: {list(ACTIVES['activity_class'].unique())}")
    check("Labels are binary 0/1",
          set(ACTIVES["label"].unique()) <= {0, 1}, "non-binary labels")
    check("pActivity within plausible range",
          ACTIVES["pactivity"].between(3, 13).all(), "implausible pActivity")
    check("ChEMBL InChIKeys unique",
          not ACTIVES["inchikey"].duplicated().any(),
          f"{ACTIVES['inchikey'].duplicated().sum()} duplicates")

    # Leakage is recorded (not necessarily zero — it must simply be known)
    check("Leakage flag present on library",
          "in_chembl_trainset" in LIBRARY.columns, "flag missing")

    # Artefacts on disk
    for p in (LIB_OUT, ACT_OUT, CFG.paths["metrics"] / "phase1_metrics.json"):
        check(f"Artefact exists: {p.name}", p.exists(), str(p))
    n_fig = len(list(CFG.paths["figures"].glob("*.png")))
    check("Figures written", n_fig >= 4, f"{n_fig} PNGs found")

    print("Phase 1 validation gate")
    print("=" * 74)
    for name, ok, detail in checks:
        mark = "PASS" if ok else "FAIL"
        print(f"  [{mark}] {name}" + (f"  ({detail})" if not ok else ""))
    print("=" * 74)
    n_pass = sum(1 for _, ok, _ in checks if ok)
    print(f"{n_pass}/{len(checks)} checks passed")

    if failures:
        raise AssertionError("Phase 1 validation FAILED:\n  - " +
                             "\n  - ".join(failures))
    print("\nPhase 1 validation PASSED — safe to proceed to Phase 2.")
    return True


validate_phase1()

In [ ]:
# --- Phase 1 summary --------------------------------------------------------
m = PHASE1_METRICS
print("=" * 74)
print(f"PHASE 1 COMPLETE — {CFG.target_name} ({TARGET_ID})")
print("=" * 74)
print(f"""
SCREENING DECK (Enamine GPCR library)
  input records .................. {m['library_input']:>9,}
  survived chemical curation ..... {m['library_curated']:>9,}
  survived lead-likeness ......... {m['library_final']:>9,}   <- screened in Phase 2-4
  overall retention .............. {m['library_retention_pct']:>8.1f}%
  unique Murcko scaffolds ........ {m['library_unique_scaffolds']:>9,}
  carrying a basic amine ......... {m['library_pct_basic_amine']:>8.1f}%

TRAINING SET (ChEMBL {TARGET_ID})
  source ......................... {m['chembl_source']:>9}
  measurements retrieved ......... {m['chembl_measurements_in']:>9,}
  labelled compounds ............. {m['chembl_labelled']:>9,}
    actives  (pAct >= {CFG.active_cutoff}) ....... {m['chembl_actives']:>9,}
    inactives (pAct <= {CFG.inactive_cutoff}) ...... {m['chembl_inactives']:>9,}

LEAKAGE AUDIT
  exact train/screen overlap ..... {m['train_screen_overlap']:>9,}
  shared Murcko scaffolds ........ {m['shared_scaffolds']:>9,}
""")
if SYNTHETIC_MODE:
    print("  *** SYNTHETIC MODE: the training set is a random stand-in. ***")
    print("  *** Re-run with ChEMBL access before interpreting any result. ***\n")
print("=" * 74)

---
# Phase 2 — Molecular representation, chemical space, and diversity analysis

Phase 1 produced two curated datasets. Phase 2 turns molecules into the numerical
representations that Phase 3 models and Phase 4 ranking consume, and then uses those
representations to answer one question that determines whether this campaign is viable at all:

> **Does the Enamine deck actually occupy the region of chemical space where A<sub>2A</sub>
> ligands live?**

A library can be perfectly curated, lead-like, and diverse, and still be incapable of producing
hits because it simply does not contain the chemotype. That is a property of the *library*, not of
the model, and no amount of Phase 3 tuning fixes it. It is far cheaper to discover it here than
after docking 48,489 compounds.

**Sections**

1. **2.0** Resuming from Phase 1 artefacts; ingesting a manually downloaded ChEMBL export
2. **2.1** Fingerprint generation — ECFP4/6, FCFP4, RDKit topological, MACCS
3. **2.2** Extended descriptor block and ligand-efficiency denominators
4. **2.3** Chemical-space mapping — PCA and UMAP in a shared embedding
5. **2.4** Clustering and diversity — MiniBatch k-means at scale, Butina for calibration
6. **2.5** Nearest-neighbour analysis and the applicability domain
7. **2.6** Persisted artefacts and the Phase 2 validation gate

### Two decisions made up front, with reasons

**Fingerprints are generated through `rdFingerprintGenerator`, not the deprecated
`GetMorganFingerprintAsBitVect`.** The legacy helpers still work but emit deprecation warnings and
differ subtly in chirality handling; the generator API is what current RDKit intends and makes the
radius/size/invariant choices explicit rather than positional.

**Clustering uses MiniBatch k-means on the full deck and Butina only on a subsample.** Butina is
the better chemistry — it produces clusters with a guaranteed intra-cluster Tanimoto floor rather
than spherical partitions of a projected space — but it requires the full pairwise distance
matrix. At 48,489 compounds that is ~1.18 × 10⁹ pairs, which is neither time- nor memory-tractable
in a notebook. The approach taken is therefore: run Butina on a random subsample to *calibrate* a
defensible Tanimoto cut-off and measure the true cluster-size distribution, then apply MiniBatch
k-means to the full deck with a cluster count informed by that calibration. Both results are
reported; neither is presented as the other.

## 2.0 Resuming from Phase 1, and ingesting a manual ChEMBL export

This section makes the notebook **resumable**: if `LIBRARY` and `ACTIVES` are already in memory
from a Phase 1 run in this kernel, they are used directly; otherwise they are reloaded from the
CSVs Phase 1 wrote. That means Phase 2 can be re-run after a kernel restart without repeating the
~9-minute curation.

It also adds a loader for a **manually downloaded ChEMBL export**. The ChEMBL web interface emits
a *semicolon-delimited* CSV with quoted headers whose column names differ from the REST API's
(`Smiles` rather than `canonical_smiles`, `Molecule ChEMBL ID` rather than `molecule_chembl_id`).
The loader below sniffs the delimiter and maps web-export names onto the canonical schema, so a
file downloaded from the target report card can be dropped in without editing.

**To use a manual export:** set `CFG.chembl_csv_fallback` to its path and re-run this section. The
compounds are pushed through the identical Phase 1 curation engine, so nothing downstream can tell
the difference between an API pull and a manual download.

In [ ]:
# --- Phase 2 imports ---------------------------------------------------------
import importlib, subprocess, sys
for mod, pkg in {"sklearn": "scikit-learn", "scipy": "scipy", "umap": "umap-learn"}.items():
    if importlib.util.find_spec(mod) is None:
        print(f"Installing {pkg} ...")
        cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
        try:
            subprocess.check_call(cmd)
        except subprocess.CalledProcessError:
            subprocess.check_call(cmd + ["--break-system-packages"])

from scipy import sparse
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator as rfg
from rdkit.Chem import MACCSkeys
from rdkit.ML.Cluster import Butina

HAVE_UMAP = importlib.util.find_spec("umap") is not None
if HAVE_UMAP:
    import umap
print(f"UMAP available: {HAVE_UMAP}")

In [ ]:
# --- Resume from Phase 1 artefacts ------------------------------------------
def resume_phase1():
    """Use in-memory Phase 1 objects if present, else reload from disk."""
    g = globals()
    if "LIBRARY" in g and "ACTIVES" in g and len(g["LIBRARY"]):
        print("Using Phase 1 objects already in memory.")
        return g["LIBRARY"], g["ACTIVES"]

    lib_p = CFG.paths["curated"] / "library_curated.csv"
    act_p = CFG.paths["curated"] / "chembl_a2a_labelled.csv"
    if not lib_p.exists():
        raise FileNotFoundError(
            f"{lib_p} not found. Run Phase 1 first, or restore the curated CSVs.")
    lib = pd.read_csv(lib_p, low_memory=False)
    act = pd.read_csv(act_p, low_memory=False) if act_p.exists() else pd.DataFrame()
    print(f"Reloaded from disk: library {len(lib):,} | labelled set {len(act):,}")
    return lib, act


LIBRARY, ACTIVES = resume_phase1()
print(f"\nLIBRARY : {len(LIBRARY):,} compounds x {LIBRARY.shape[1]} columns")
print(f"ACTIVES : {len(ACTIVES):,} compounds")

In [ ]:
# Set CFG.chembl_csv_fallback to a downloaded export to activate this path.
if CFG.chembl_csv_fallback and Path(CFG.chembl_csv_fallback).exists():
    print("Manual ChEMBL export detected — reprocessing the labelled set.\n")
    raw_manual = load_chembl_export(CFG.chembl_csv_fallback)
    chembl_act, chembl_audit = curate_bioactivities(raw_manual)
    chembl_audit.summary()
    chembl_act = chembl_act.rename(columns={"canonical_smiles": "smiles",
                                            "molecule_chembl_id": "compound_id"})
    ACTIVES, chembl_struct_audit = curate_dataframe(chembl_act, "ChEMBL A2A (manual export)")
    ACTIVES = assess_druglikeness(compute_descriptors(ACTIVES))
    ACTIVES.to_csv(CFG.paths["curated"] / "chembl_a2a_labelled.csv", index=False)
    SYNTHETIC_MODE = False
    print(f"\nLabelled set rebuilt from manual export: {len(ACTIVES):,} compounds")
    print(ACTIVES["activity_class"].value_counts().to_string())
else:
    print("No manual ChEMBL export configured "
          "(CFG.chembl_csv_fallback is unset or missing).")
    print(f"Continuing with the Phase 1 labelled set: {len(ACTIVES):,} compounds.")

SYNTHETIC_MODE = bool(globals().get("SYNTHETIC_MODE", False))
if SYNTHETIC_MODE:
    print("\n" + "!" * 74)
    print("!!  SYNTHETIC MODE ACTIVE — the labelled set is a random stand-in.")
    print("!!  Library-only results below (fingerprints, clustering, diversity)")
    print("!!  are REAL. Anything comparing library to 'actives' is NOT.")
    print("!" * 74)

## 2.1 Fingerprint generation

Five representations are computed, because they encode genuinely different things and Phase 3
benchmarks them against each other rather than assuming one is best:

| Fingerprint | Radius / size | What it encodes | Why it is here |
|---|---|---|---|
| **ECFP4** | r=2, 2048 bits | circular atom environments by element/charge/degree | the default for ligand-based VS; captures specific substructure identity |
| **ECFP6** | r=3, 2048 bits | larger environments | more specific, sparser — better when actives share a large common core |
| **FCFP4** | r=2, 2048 bits, feature invariants | circular environments by *pharmacophoric role* (donor, acceptor, aromatic, halogen) rather than element | the scaffold-hopping representation: two different heterocycles with the same H-bonding pattern collide, which is exactly what you want when searching a library for a novel core |
| **RDKit topological** | paths ≤ 7, 2048 bits | linear/branched path substructures | an orthogonal, non-circular view |
| **MACCS** | 166 keys | curated structural-key presence | low-dimensional, interpretable, good baseline |

**FCFP4 deserves emphasis for this project.** The A<sub>2A</sub> antagonist pharmacophore is
defined more by its hydrogen-bonding geometry to Asn253<sup>6.55</sup> than by any specific
heterocycle. An ECFP search will preferentially retrieve compounds containing the *same*
triazolotriazine core as the ChEMBL actives; an FCFP search will also retrieve a different ring
system presenting the same donor–acceptor pair. Since the value of screening a REAL-space library
lies in finding novel cores rather than re-finding known ones, FCFP is likely to be the more
productive representation here — a hypothesis Phase 3 tests rather than assumes.

Fingerprints are stored in two forms: RDKit `ExplicitBitVect` objects for Tanimoto arithmetic, and
a **SciPy sparse matrix** for scikit-learn. Sparse storage matters — 48,489 × 2048 as dense
`float32` is ~400 MB and will exhaust a free Colab instance, while the same data as CSR is a few
tens of MB because only ~50 bits are set per molecule.

In [ ]:
# --- Fingerprint generators --------------------------------------------------
FP_SIZE = 2048

FP_GENERATORS = {
    "ecfp4": rfg.GetMorganGenerator(radius=2, fpSize=FP_SIZE),
    "ecfp6": rfg.GetMorganGenerator(radius=3, fpSize=FP_SIZE),
    "fcfp4": rfg.GetMorganGenerator(
        radius=2, fpSize=FP_SIZE,
        atomInvariantsGenerator=rfg.GetMorganFeatureAtomInvGen()),
    "rdkit_fp": rfg.GetRDKitFPGenerator(maxPath=7, fpSize=FP_SIZE),
}

def compute_fingerprints(df, kinds=("ecfp4", "fcfp4"), smiles_col="std_smiles",
                         progress=True, label=""):
    """
    Generate fingerprints for a DataFrame of standardised molecules.

    Returns
    -------
    dict
        kind -> {"bitvects": [ExplicitBitVect], "matrix": csr_matrix}
    """
    mols = [Chem.MolFromSmiles(s) for s in df[smiles_col]]
    n_bad = sum(m is None for m in mols)
    if n_bad:
        raise ValueError(f"{n_bad} molecules failed to parse — curation invariant broken")

    out = {}
    for kind in kinds:
        bvs = []
        it = tqdm(mols, desc=f"{label}{kind}", unit="mol", disable=not progress)
        if kind == "maccs":
            for m in it:
                bvs.append(MACCSkeys.GenMACCSKeys(m))
            width = 167
        else:
            gen = FP_GENERATORS[kind]
            for m in it:
                bvs.append(gen.GetFingerprint(m))
            width = FP_SIZE

        # Sparse CSR assembly from the on-bit indices — never materialise dense.
        indptr, indices = [0], []
        for bv in bvs:
            on = list(bv.GetOnBits())
            indices.extend(on)
            indptr.append(len(indices))
        mat = sparse.csr_matrix(
            (np.ones(len(indices), dtype=np.uint8), np.array(indices), np.array(indptr)),
            shape=(len(bvs), width))
        out[kind] = {"bitvects": bvs, "matrix": mat}
        density = mat.nnz / (mat.shape[0] * mat.shape[1])
        print(f"  {kind:<9} {mat.shape[0]:>7,} x {mat.shape[1]:<5} "
              f"density {density:.4f}  mean on-bits {mat.nnz / mat.shape[0]:.1f}")
    return out


FP_KINDS = ("ecfp4", "ecfp6", "fcfp4", "rdkit_fp", "maccs")

print("Library fingerprints")
t0 = time.time()
LIB_FP = compute_fingerprints(LIBRARY, FP_KINDS, label="library ")
print(f"  elapsed {time.time() - t0:.1f} s\n")

print("Labelled-set fingerprints")
ACT_FP = compute_fingerprints(ACTIVES, FP_KINDS, label="actives ") if len(ACTIVES) else {}

## 2.2 Extended descriptor block and ligand-efficiency denominators

Phase 1 computed the descriptors needed for filtering. Phase 3 needs a richer, model-ready block,
and Phase 4's ranking needs efficiency denominators.

**Ligand efficiency is included now because it changes what counts as a good hit later.** A
docking score or predicted potency taken alone systematically favours large, greasy molecules —
adding heavy atoms almost always improves a raw score without improving the *quality* of binding.
Normalising by heavy-atom count (LE) and by lipophilicity (LLE) is what prevents Phase 4 from
handing back a ranked list of the largest compounds in the deck. The denominators are computed
here; the efficiencies themselves are assembled in Phase 4 once scores exist.

The descriptor block is deliberately kept to interpretable, fast, well-understood terms rather
than the full ~200-descriptor RDKit set. A large descriptor block on a few thousand training
compounds invites overfitting, and Phase 3's feature importances are only useful if a
medicinal chemist can read them.

In [ ]:
# --- Extended descriptor block ----------------------------------------------
EXTENDED_DESCRIPTORS = {
    "mol_mr":         Descriptors.MolMR,            # Crippen molar refractivity
    "labute_asa":     Descriptors.LabuteASA,        # approximate surface area
    "balaban_j":      Descriptors.BalabanJ,         # topological branching
    "bertz_ct":       Descriptors.BertzCT,          # molecular complexity
    "chi0v":          Descriptors.Chi0v,            # Kier-Hall connectivity
    "chi1v":          Descriptors.Chi1v,
    "kappa1":         Descriptors.Kappa1,           # Kier shape indices
    "kappa2":         Descriptors.Kappa2,
    "hall_kier_alpha": Descriptors.HallKierAlpha,
    "num_amide_bonds": rdMD.CalcNumAmideBonds,
    "num_spiro":      rdMD.CalcNumSpiroAtoms,
    "num_bridgehead": rdMD.CalcNumBridgeheadAtoms,
    "num_sat_rings":  rdMD.CalcNumSaturatedRings,
    "num_ali_rings":  rdMD.CalcNumAliphaticRings,
    "num_het_rings":  rdMD.CalcNumHeterocycles,
    "num_nhoh":       Descriptors.NHOHCount,
    "num_no":         Descriptors.NOCount,
    "max_partial_charge": Descriptors.MaxPartialCharge,
    "min_partial_charge": Descriptors.MinPartialCharge,
}


def compute_extended_descriptors(df, smiles_col="std_smiles", progress=True, label=""):
    """Second-tier descriptors plus derived ratios and LE denominators."""
    rows = []
    for smi in tqdm(df[smiles_col], desc=f"{label}extended desc", unit="mol",
                    disable=not progress):
        mol = Chem.MolFromSmiles(smi)
        rec = {}
        for name, fn in EXTENDED_DESCRIPTORS.items():
            try:
                v = fn(mol)
                rec[name] = float(v) if v is not None and np.isfinite(v) else np.nan
            except Exception:                        # noqa: BLE001
                rec[name] = np.nan
        rows.append(rec)

    ext = pd.DataFrame(rows)
    out = pd.concat([df.reset_index(drop=True), ext], axis=1)

    # Derived, size-normalised terms.
    ha = out["heavy_atoms"].replace(0, np.nan)
    out["tpsa_per_ha"]      = out["tpsa"] / ha
    out["mw_per_ha"]        = out["mw"] / ha
    out["aromatic_frac"]    = out["aromatic_rings"] / out["rings"].replace(0, np.nan)
    out["heteroatom_frac"]  = out["heteroatoms"] / ha
    out["complexity_per_ha"] = out["bertz_ct"] / ha

    # Ligand-efficiency denominators — Phase 4 divides scores by these.
    out["le_denominator"]   = ha                       # LE  = score / heavy atoms
    out["lle_offset"]       = out["clogp"]             # LLE = pAct - cLogP

    # Partial-charge descriptors are NaN for a small number of exotic atom
    # types; impute at the column median so the modelling matrix stays dense.
    for c in ("max_partial_charge", "min_partial_charge"):
        n_nan = out[c].isna().sum()
        if n_nan:
            out[c] = out[c].fillna(out[c].median())
            print(f"  imputed {n_nan} NaN in {c} at the column median")
    return out


t0 = time.time()
LIBRARY = compute_extended_descriptors(LIBRARY, label="library ")
if len(ACTIVES):
    ACTIVES = compute_extended_descriptors(ACTIVES, label="actives ")
print(f"\nElapsed {time.time() - t0:.1f} s")

# The descriptor block Phase 3 will model on.
DESCRIPTOR_COLUMNS = [
    "mw", "clogp", "tpsa", "hbd", "hba", "rotb", "heavy_atoms", "rings",
    "aromatic_rings", "heteroatoms", "fsp3", "stereocentres", "qed",
    *EXTENDED_DESCRIPTORS.keys(),
    "tpsa_per_ha", "mw_per_ha", "aromatic_frac", "heteroatom_frac",
    "complexity_per_ha",
]
DESCRIPTOR_COLUMNS = [c for c in DESCRIPTOR_COLUMNS if c in LIBRARY.columns]
print(f"\nDescriptor block: {len(DESCRIPTOR_COLUMNS)} features")
print(f"NaNs remaining in library block: "
      f"{int(LIBRARY[DESCRIPTOR_COLUMNS].isna().sum().sum())}")
display(LIBRARY[DESCRIPTOR_COLUMNS].describe().T.round(3).head(20))

## 2.3 Chemical-space mapping

Both datasets are projected into **one shared embedding** — the projection is fitted on the
combined data and both are transformed by it. Fitting separate projections and plotting them side
by side would produce two pictures whose axes mean different things, and any apparent overlap
would be an artefact.

Two complementary views:

* **PCA on the descriptor block** — a linear projection of interpretable physicochemical
  properties. Axes have meaning (loadings are reported), and distance is property distance.
* **UMAP on fingerprints** — a non-linear projection of substructure space, run on a TruncatedSVD
  reduction of the sparse fingerprint matrix. Local structure is preserved; global distances are
  not metric and should not be over-read.

A note on reading these plots honestly: **UMAP cluster separation is not evidence of anything by
itself.** UMAP will produce visually distinct islands from continuous data. What is legitimate to
read is whether the ChEMBL actives fall *inside* regions the library populates — that is a
statement about coverage, and §2.5 backs it with a quantitative nearest-neighbour distance rather
than relying on the picture.

In [ ]:
# --- PCA on the descriptor block --------------------------------------------
def build_combined_frame():
    """Stack library and labelled set with a provenance column."""
    lib = LIBRARY[DESCRIPTOR_COLUMNS].copy()
    lib["_group"] = "Enamine library"
    frames = [lib]
    if len(ACTIVES):
        for cls, name in (("active", "ChEMBL actives"), ("inactive", "ChEMBL inactives")):
            sub = ACTIVES[ACTIVES["activity_class"] == cls]
            if len(sub):
                f = sub[DESCRIPTOR_COLUMNS].copy()
                f["_group"] = name
                frames.append(f)
    return pd.concat(frames, ignore_index=True)


combined = build_combined_frame()
X = combined[DESCRIPTOR_COLUMNS].astype(float).values
groups = combined["_group"].values
print(f"Combined matrix: {X.shape[0]:,} molecules x {X.shape[1]} descriptors")
print(combined["_group"].value_counts().to_string())

# Standardise, then fit ONE PCA on everything.
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
pca = PCA(n_components=min(10, Xs.shape[1]), random_state=CFG.random_seed)
PCA_EMB = pca.fit_transform(Xs)

evr = pca.explained_variance_ratio_
print(f"\nExplained variance: PC1 {evr[0]:.1%}, PC2 {evr[1]:.1%}, "
      f"PC1+PC2 {evr[:2].sum():.1%}, first 10 {evr.sum():.1%}")

# Loadings make the axes interpretable — report the dominant terms.
load = pd.DataFrame(pca.components_[:2].T, index=DESCRIPTOR_COLUMNS,
                    columns=["PC1", "PC2"])
print("\nTop PC1 loadings:")
print(load["PC1"].abs().sort_values(ascending=False).head(6).to_string())
print("\nTop PC2 loadings:")
print(load["PC2"].abs().sort_values(ascending=False).head(6).to_string())

In [ ]:
# --- UMAP on fingerprints (via TruncatedSVD of the sparse matrix) -----------
FP_FOR_SPACE = "ecfp4"

mats = [LIB_FP[FP_FOR_SPACE]["matrix"]]
space_groups = ["Enamine library"] * LIB_FP[FP_FOR_SPACE]["matrix"].shape[0]
if ACT_FP:
    for cls, name in (("active", "ChEMBL actives"), ("inactive", "ChEMBL inactives")):
        idx = np.where(ACTIVES["activity_class"].values == cls)[0]
        if len(idx):
            mats.append(ACT_FP[FP_FOR_SPACE]["matrix"][idx])
            space_groups += [name] * len(idx)
FP_COMBINED = sparse.vstack(mats).astype(np.float32)
space_groups = np.array(space_groups)
print(f"Combined fingerprint matrix: {FP_COMBINED.shape} "
      f"(sparse, {FP_COMBINED.nnz / 1e6:.1f}M non-zeros, "
      f"{FP_COMBINED.data.nbytes / 1e6:.1f} MB)")

# TruncatedSVD works directly on sparse input; PCA would densify and blow memory.
t0 = time.time()
svd = TruncatedSVD(n_components=50, random_state=CFG.random_seed)
FP_SVD = svd.fit_transform(FP_COMBINED)
print(f"TruncatedSVD -> {FP_SVD.shape}, "
      f"{svd.explained_variance_ratio_.sum():.1%} variance retained, "
      f"{time.time() - t0:.1f} s")

if HAVE_UMAP:
    t0 = time.time()
    reducer = umap.UMAP(n_neighbors=25, min_dist=0.15, metric="cosine",
                        random_state=CFG.random_seed, verbose=False)
    UMAP_EMB = reducer.fit_transform(FP_SVD)
    print(f"UMAP -> {UMAP_EMB.shape}, {time.time() - t0:.1f} s")
else:
    print("umap-learn unavailable — falling back to the first two SVD components.")
    UMAP_EMB = FP_SVD[:, :2]

In [ ]:
# --- Figure 7: shared chemical-space embeddings -----------------------------
def plot_space(emb, grp, ax, title, xlabel, ylabel, lib_alpha=0.10):
    """Library as a recessive density backdrop; actives drawn on top."""
    order = [("Enamine library", PALETTE["library"], lib_alpha, 3, 0),
             ("ChEMBL inactives", PALETTE["inactive"], 0.55, 10, 2),
             ("ChEMBL actives", PALETTE["active"], 0.80, 16, 3)]
    for name, colr, alpha, size, z in order:
        m = grp == name
        if not m.any():
            continue
        ax.scatter(emb[m, 0], emb[m, 1], s=size, alpha=alpha, c=colr,
                   edgecolors="none", rasterized=True, zorder=z,
                   label=f"{name} (n={m.sum():,})")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")
    leg = ax.legend(loc="best", markerscale=3)
    for h in leg.legend_handles:
        h.set_alpha(1.0)
    ax.grid(alpha=0.5)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_space(PCA_EMB, groups, axes[0],
           f"PCA of the descriptor block  ({evr[:2].sum():.0%} of variance)",
           f"PC1 ({evr[0]:.1%})", f"PC2 ({evr[1]:.1%})")
plot_space(UMAP_EMB, space_groups, axes[1],
           f"UMAP of {FP_FOR_SPACE.upper()} fingerprints"
           + ("" if HAVE_UMAP else "  [SVD fallback]"),
           "UMAP 1", "UMAP 2")
fig.suptitle("Chemical space — library and known ligands in ONE shared embedding",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig07_chemical_space")
plt.show()

if SYNTHETIC_MODE:
    print("\n*** SYNTHETIC MODE: the 'actives'/'inactives' here were drawn from "
          "the library itself, so their overlap is trivially perfect and means "
          "nothing. Re-run with real ChEMBL data to interpret this figure. ***")

## 2.4 Clustering and diversity analysis

Diversity analysis serves Phase 4 directly. A ranked hit list drawn from a single dense cluster is
a single hypothesis tested 50 times; drawing across clusters converts the same 50 compounds into
50 partially independent tests. Clustering is what makes that possible.

**Butina, on a subsample, to calibrate.** Butina is the chemically principled choice: it guarantees
every cluster member lies within a chosen Tanimoto distance of its centroid, so the cut-off has a
direct chemical meaning. The Tanimoto cut-off is *scanned* rather than assumed — the conventional
0.35 distance (0.65 similarity) is a heuristic, and the scan shows what it actually produces on
this library.

**MiniBatch k-means, on the full deck, to partition.** With a cut-off calibrated, the full 48,489
compounds are partitioned in the SVD-reduced fingerprint space. k is chosen from the Butina
calibration so the two methods produce comparable granularity, and silhouette score is reported as
a sanity check — not as proof of "correct" clusters, which is not a meaningful concept for
continuous chemical space.

**Scaffold diversity is reported alongside**, because cluster count and scaffold count answer
different questions: clusters tell you how many distinct *neighbourhoods* the library spans,
scaffolds tell you how many distinct *cores* it contains.

In [ ]:
# --- Butina calibration on a subsample --------------------------------------
BUTINA_SAMPLE = min(6000, len(LIBRARY))
rng = np.random.default_rng(CFG.random_seed)
sub_idx = rng.choice(len(LIBRARY), BUTINA_SAMPLE, replace=False)
sub_fps = [LIB_FP["ecfp4"]["bitvects"][i] for i in sub_idx]
print(f"Butina calibration on a random subsample of {BUTINA_SAMPLE:,} compounds")
print(f"(full-deck Butina would need "
      f"{len(LIBRARY) * (len(LIBRARY) - 1) // 2 / 1e9:.2f}e9 pairwise distances "
      f"— not tractable in a notebook)\n")

# Condensed lower-triangle distance list, as Butina expects.
t0 = time.time()
dists = []
for i in tqdm(range(1, len(sub_fps)), desc="Pairwise Tanimoto", unit="row"):
    sims = DataStructs.BulkTanimotoSimilarity(sub_fps[i], sub_fps[:i])
    dists.extend(1.0 - np.asarray(sims))
dists = np.asarray(dists, dtype=np.float32)
print(f"  {len(dists):,} distances in {time.time() - t0:.1f} s")
print(f"  mean pairwise Tanimoto distance {dists.mean():.3f} "
      f"(similarity {1 - dists.mean():.3f})")

# Scan the cut-off rather than assuming 0.35.
CUTOFF_SCAN = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
scan_rows = []
for cut in CUTOFF_SCAN:
    cl = Butina.ClusterData(dists.tolist(), len(sub_fps), cut,
                            isDistData=True, reordering=True)
    sizes = np.array([len(c) for c in cl])
    scan_rows.append({
        "cutoff": cut,
        "n_clusters": len(cl),
        "singletons": int((sizes == 1).sum()),
        "pct_singleton": round(100 * (sizes == 1).sum() / len(cl), 1),
        "largest": int(sizes.max()),
        "mean_size": round(sizes.mean(), 2),
        "clusters_per_1k": round(1000 * len(cl) / len(sub_fps), 1),
    })
BUTINA_SCAN = pd.DataFrame(scan_rows)
print("\nButina cut-off scan (Tanimoto DISTANCE; similarity = 1 - cutoff)")
print(BUTINA_SCAN.to_string(index=False))

BUTINA_CUTOFF = 0.35
chosen = BUTINA_SCAN[BUTINA_SCAN["cutoff"] == BUTINA_CUTOFF].iloc[0]
print(f"\nAdopting cutoff {BUTINA_CUTOFF} (Tanimoto similarity "
      f"{1 - BUTINA_CUTOFF:.2f}): {chosen['clusters_per_1k']} clusters per 1,000 "
      f"compounds, {chosen['pct_singleton']}% singletons.")

# --- Interpretation: how much structure is actually there? ------------------
SINGLETON_PCT = float(chosen["pct_singleton"])
LIBRARY_IS_PRECLUSTERED = SINGLETON_PCT > 80
if LIBRARY_IS_PRECLUSTERED:
    print(f"""
  FINDING: {SINGLETON_PCT:.1f}% of Butina clusters are singletons at this
  cut-off, and the largest cluster holds only {int(chosen['largest'])} compounds. Even at a
  0.50 distance cut-off (0.50 Tanimoto similarity) the singleton fraction
  stays above 75%.

  This library contains almost no analogue series — it has already been
  diversity-selected by the vendor, which is exactly what a "targeted
  screening library" is supposed to be. Two consequences:

   1. Cluster-based diversity picking has little left to do here. Phase 4
      should enforce diversity with a direct Tanimoto threshold between
      selected hits rather than by picking one compound per cluster.
   2. The k-means partition below is a convenience index for navigating the
      deck, NOT a claim that the library has discrete chemotype families.
      Expect a near-zero silhouette, and read it as confirmation of that.
""")

K_IMPLIED = int(round(chosen["clusters_per_1k"] / 1000 * len(LIBRARY)))
K_CAP = 4000
K_FULL = max(50, min(K_IMPLIED, K_CAP))
print(f"Butina-implied k for the full deck: {K_IMPLIED:,}")
if K_FULL != K_IMPLIED:
    print(f"  CAPPED to k = {K_FULL:,}. The implied value is {K_IMPLIED:,} "
          f"(~{K_IMPLIED / len(LIBRARY):.2f} clusters per compound), which is")
    print(f"  near-degenerate — it would make most clusters singletons and the")
    print(f"  partition useless for navigation. The cap is a deliberate choice")
    print(f"  for usability, not a calibration result; k-means granularity is")
    print(f"  therefore COARSER than Butina's and the two are not equivalent.")
print(f"\nUsing k = {K_FULL:,} clusters "
      f"(~{len(LIBRARY) / K_FULL:.1f} compounds per cluster on average)")

In [ ]:
# --- MiniBatch k-means on the full deck -------------------------------------
LIB_SVD = FP_SVD[:len(LIBRARY)]          # library rows of the shared SVD space

t0 = time.time()
kmeans = MiniBatchKMeans(n_clusters=K_FULL, random_state=CFG.random_seed,
                         batch_size=2048, n_init=5, max_iter=200)
LIBRARY["cluster"] = kmeans.fit_predict(LIB_SVD)
print(f"MiniBatch k-means: {K_FULL:,} clusters over {len(LIBRARY):,} compounds "
      f"in {time.time() - t0:.1f} s")

sizes = LIBRARY["cluster"].value_counts()
print(f"\nCluster sizes: min {sizes.min()}, median {sizes.median():.0f}, "
      f"mean {sizes.mean():.1f}, max {sizes.max()}")
print(f"Singleton clusters: {(sizes == 1).sum():,} "
      f"({100 * (sizes == 1).sum() / len(sizes):.1f}%)")

# Silhouette on a subsample — the full computation is O(n^2).
sil_idx = rng.choice(len(LIBRARY), min(5000, len(LIBRARY)), replace=False)
try:
    sil = silhouette_score(LIB_SVD[sil_idx], LIBRARY["cluster"].values[sil_idx],
                           metric="cosine")
    print(f"Silhouette (5k subsample, cosine): {sil:.3f}")
    print("  Note: low silhouette is EXPECTED and not a defect. Chemical space "
          "is continuous;\n  the clusters are a useful partition for diversity "
          "selection, not evidence of\n  discrete natural groups.")
except Exception as exc:                            # noqa: BLE001
    sil = np.nan
    print(f"Silhouette unavailable: {exc}")

# Cluster representatives: the compound closest to each centroid. Phase 4's
# diversity selection draws from these.
centroids = kmeans.cluster_centers_
reps = []
for c in tqdm(sorted(LIBRARY["cluster"].unique()), desc="Cluster reps", unit="cluster"):
    members = np.where(LIBRARY["cluster"].values == c)[0]
    d = np.linalg.norm(LIB_SVD[members] - centroids[c], axis=1)
    reps.append(members[int(np.argmin(d))])
LIBRARY["is_cluster_rep"] = False
LIBRARY.loc[LIBRARY.index[reps], "is_cluster_rep"] = True
print(f"\nCluster representatives marked: {LIBRARY['is_cluster_rep'].sum():,}")

In [ ]:
# --- Figure 8: clustering and diversity -------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))

# (a) and (b) are small multiples of the same scan, NOT a dual axis: the two
# quantities have incomparable units, so each gets its own panel and y-scale.
for ax, col, colr, ylab, title in [
    (axes[0], "clusters_per_1k", PALETTE["library"], "Clusters per 1,000 compounds",
     "Cluster granularity vs cut-off"),
    (axes[1], "pct_singleton", PALETTE["active"], "Singleton clusters (%)",
     "Singleton fraction vs cut-off"),
]:
    ax.plot(BUTINA_SCAN["cutoff"], BUTINA_SCAN[col], "o-", color=colr, markersize=7)
    ax.axvline(BUTINA_CUTOFF, color=INK["secondary"], ls="--", lw=1.2, zorder=0)
    for x, y in zip(BUTINA_SCAN["cutoff"], BUTINA_SCAN[col]):
        ax.annotate(f"{y:.0f}", (x, y), textcoords="offset points", xytext=(0, 9),
                    fontsize=7.5, ha="center", color=colr, fontweight="600")
    lo, hi = BUTINA_SCAN[col].min(), BUTINA_SCAN[col].max()
    pad = (hi - lo) * 0.28 or 1
    ax.set_ylim(lo - pad * 0.5, hi + pad)
    ax.annotate(f"adopted {BUTINA_CUTOFF}", (BUTINA_CUTOFF, lo - pad * 0.32),
                fontsize=8, ha="center", color=INK["secondary"], fontweight="600")
    ax.set_xlabel("Butina Tanimoto distance cut-off")
    ax.set_ylabel(ylab)
    ax.set_title(f"{title}\n({BUTINA_SAMPLE:,}-compound subsample)", loc="left")
    ax.grid(alpha=0.7)

# (c) k-means cluster-size distribution
ax = axes[2]
ax.hist(sizes.values, bins=np.arange(0.5, min(sizes.max(), 60) + 1.5),
        color=PALETTE["library"], edgecolor=SURFACE, linewidth=0.8)
ax.set_xlabel("Compounds per cluster")
ax.set_ylabel("Number of clusters")
ax.set_title(f"k-means cluster sizes  (k = {K_FULL:,}, median {sizes.median():.0f})",
             loc="left")
ax.grid(alpha=0.7)

# (d) diversity summary as labelled bars
ax = axes[3]
div = {
    "Compounds":         len(LIBRARY),
    "Murcko scaffolds":  LIBRARY["murcko_scaffold"].nunique(),
    "k-means clusters":  K_FULL,
}
names, vals = list(div), list(div.values())
bars = ax.bar(range(len(div)), vals, color=[SEQ_BLUE[5], SEQ_BLUE[3], SEQ_BLUE[2]],
              width=0.6, edgecolor=SURFACE, linewidth=2)
for i, (b, v) in enumerate(zip(bars, vals)):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom",
            fontsize=9.5, fontweight="600", color=INK["primary"])
ax.set_xticks(range(len(div)))
ax.set_xticklabels([textwrap.fill(n, 12) for n in names], fontsize=8.5)
ax.set_ylabel("Count")
ax.set_ylim(0, max(vals) * 1.18)
ax.set_title(f"Diversity at three granularities\n"
             f"{LIBRARY['murcko_scaffold'].nunique() / len(LIBRARY):.2f} scaffolds "
             f"per compound", loc="left")
ax.grid(alpha=0.7)

fig.suptitle("Clustering and diversity — the basis for Phase 4 hit selection",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig08_clustering_diversity")
plt.show()

## 2.5 Nearest-neighbour analysis and the applicability domain

This is the section that answers the question posed at the top of Phase 2, quantitatively.

For every library compound we compute the **maximum Tanimoto similarity to any known
A<sub>2A</sub> active**. That single number does three jobs:

1. **It is the similarity-search baseline.** Phase 3's machine-learning model has to beat
   *"rank by nearest-active similarity"* to have earned its complexity. Models routinely fail to,
   and a paper or portfolio piece that does not report this baseline is hiding something.
2. **It defines the applicability domain.** A QSAR model extrapolating to a compound with 0.15
   maximum similarity to anything it was trained on is producing a number, not a prediction.
   Phase 3 flags such compounds so Phase 4 can treat their scores with appropriate scepticism
   rather than ranking them naively alongside well-supported ones.
3. **It is the novelty axis.** The interesting region for a REAL-space screen is *intermediate*
   similarity — roughly 0.30–0.55 on ECFP4. Above ~0.7 the compound is a close analogue of a known
   ligand and offers little IP or novelty value; below ~0.25 it is likely a false positive of
   whatever model scored it. Phase 4 uses this band explicitly.

Both ECFP4 and FCFP4 are computed, because they measure different kinds of closeness: ECFP4
similarity means *same substructures*, FCFP4 similarity means *same pharmacophoric pattern*. A
compound with low ECFP4 but high FCFP4 similarity to a known active is precisely a scaffold-hop
candidate — the most valuable thing this screen can find.

In [ ]:
# --- Nearest-active similarity ----------------------------------------------
def nearest_active_similarity(lib_bitvects, act_bitvects, label=""):
    """Max and mean-of-top-5 Tanimoto from each library compound to any active."""
    if not act_bitvects:
        return np.full(len(lib_bitvects), np.nan), np.full(len(lib_bitvects), np.nan)
    max_sim, top5 = np.empty(len(lib_bitvects)), np.empty(len(lib_bitvects))
    for i, fp in enumerate(tqdm(lib_bitvects, desc=f"NN similarity {label}",
                                unit="mol")):
        sims = np.asarray(DataStructs.BulkTanimotoSimilarity(fp, act_bitvects))
        max_sim[i] = sims.max()
        k = min(5, len(sims))
        top5[i] = np.sort(sims)[-k:].mean()
    return max_sim, top5


active_mask = (ACTIVES["activity_class"].values == "active") if len(ACTIVES) else np.array([])

for fp_kind in ("ecfp4", "fcfp4"):
    if ACT_FP and active_mask.any():
        act_bvs = [ACT_FP[fp_kind]["bitvects"][i] for i in np.where(active_mask)[0]]
    else:
        act_bvs = []
    mx, t5 = nearest_active_similarity(LIB_FP[fp_kind]["bitvects"], act_bvs,
                                       label=fp_kind)
    LIBRARY[f"nn_sim_{fp_kind}"] = mx
    LIBRARY[f"nn_top5_{fp_kind}"] = t5

if active_mask.any():
    print(f"\nNearest-active Tanimoto similarity ({active_mask.sum():,} actives)")
    print(LIBRARY[["nn_sim_ecfp4", "nn_sim_fcfp4",
                   "nn_top5_ecfp4", "nn_top5_fcfp4"]].describe().round(3).to_string())
else:
    print("\nNo actives available — nearest-neighbour columns are NaN.")

In [ ]:
# --- Applicability domain and the novelty band ------------------------------
AD_THRESHOLD      = 0.25   # below this, a Phase 3 prediction is extrapolation
NOVELTY_BAND      = (0.30, 0.55)
ANALOGUE_THRESHOLD = 0.70  # above this, a close analogue of a known ligand

s = LIBRARY["nn_sim_ecfp4"]
if s.notna().any():
    LIBRARY["in_applicability_domain"] = s >= AD_THRESHOLD
    LIBRARY["in_novelty_band"] = s.between(*NOVELTY_BAND)
    LIBRARY["is_known_analogue"] = s >= ANALOGUE_THRESHOLD

    # Scaffold-hop candidates: pharmacophorically close, substructurally distant.
    LIBRARY["scaffold_hop_candidate"] = (
        (LIBRARY["nn_sim_fcfp4"] >= 0.45) & (LIBRARY["nn_sim_ecfp4"] < 0.35))

    print("Applicability-domain and novelty partition")
    print("-" * 68)
    for name, mask, note in [
        ("In applicability domain", LIBRARY["in_applicability_domain"],
         f"ECFP4 NN sim >= {AD_THRESHOLD}"),
        ("Below AD (extrapolation)", ~LIBRARY["in_applicability_domain"],
         "Phase 3 predictions flagged low-confidence"),
        ("In novelty band", LIBRARY["in_novelty_band"],
         f"{NOVELTY_BAND[0]}-{NOVELTY_BAND[1]} — the target region"),
        ("Known analogues", LIBRARY["is_known_analogue"],
         f">= {ANALOGUE_THRESHOLD} — low novelty value"),
        ("Scaffold-hop candidates", LIBRARY["scaffold_hop_candidate"],
         "FCFP4 >= 0.45 but ECFP4 < 0.35"),
    ]:
        print(f"  {name:<26} {int(mask.sum()):>7,}  "
              f"({100 * mask.mean():5.1f}%)   {note}")
else:
    for c in ("in_applicability_domain", "in_novelty_band", "is_known_analogue",
              "scaffold_hop_candidate"):
        LIBRARY[c] = False
    print("No actives — applicability domain cannot be defined. "
          "All flags set False.")

In [ ]:
# --- Figure 9: nearest-neighbour similarity and the novelty band ------------
if LIBRARY["nn_sim_ecfp4"].notna().any():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))

    # (a) similarity distributions, ECFP4 vs FCFP4
    ax = axes[0]
    bins = np.linspace(0, 1, 60)
    ax.hist(LIBRARY["nn_sim_ecfp4"], bins=bins, histtype="step", lw=2,
            color=PALETTE["library"], label="ECFP4 (substructure)")
    ax.hist(LIBRARY["nn_sim_fcfp4"], bins=bins, histtype="step", lw=2,
            color=PALETTE["active"], label="FCFP4 (pharmacophore)")
    ax.axvspan(*NOVELTY_BAND, color="#ebeae6", zorder=0, label="novelty band")
    ax.axvline(AD_THRESHOLD, color=INK["secondary"], ls="--", lw=1.2)
    ax.annotate("applicability\ndomain floor", (AD_THRESHOLD, ax.get_ylim()[1] * 0.80),
                fontsize=8, ha="right", color=INK["secondary"], fontweight="600")
    ax.set_xlabel("Max Tanimoto similarity to any known A2A active")
    ax.set_ylabel("Library compounds")
    ax.set_title("Nearest-active similarity", loc="left")
    ax.legend(loc="upper right"); ax.grid(alpha=0.7)

    # (b) ECFP4 vs FCFP4 — the scaffold-hop quadrant
    ax = axes[1]
    samp = LIBRARY.sample(min(12000, len(LIBRARY)), random_state=CFG.random_seed)
    hop = samp["scaffold_hop_candidate"]
    ax.scatter(samp.loc[~hop, "nn_sim_ecfp4"], samp.loc[~hop, "nn_sim_fcfp4"],
               s=5, alpha=0.15, color=PALETTE["library"], edgecolors="none",
               rasterized=True, label=f"library (n={(~hop).sum():,})")
    ax.scatter(samp.loc[hop, "nn_sim_ecfp4"], samp.loc[hop, "nn_sim_fcfp4"],
               s=11, alpha=0.75, color=PALETTE["active"], edgecolors="none",
               rasterized=True, label=f"scaffold-hop candidates (n={hop.sum():,})")
    ax.axhline(0.45, color=INK["secondary"], ls="--", lw=1)
    ax.axvline(0.35, color=INK["secondary"], ls="--", lw=1)
    ax.set_xlabel("ECFP4 similarity to nearest active")
    ax.set_ylabel("FCFP4 similarity to nearest active")
    ax.set_title("Scaffold-hop quadrant\n(high pharmacophore, low substructure)",
                 loc="left")
    ax.legend(loc="lower right", markerscale=2.5); ax.grid(alpha=0.7)

    # (c) how the deck partitions
    ax = axes[2]
    part = {
        "Below AD":        int((~LIBRARY["in_applicability_domain"]).sum()),
        "AD, pre-band":    int((LIBRARY["in_applicability_domain"] &
                                (LIBRARY["nn_sim_ecfp4"] < NOVELTY_BAND[0])).sum()),
        "Novelty band":    int(LIBRARY["in_novelty_band"].sum()),
        "Band–analogue":   int(((LIBRARY["nn_sim_ecfp4"] > NOVELTY_BAND[1]) &
                                ~LIBRARY["is_known_analogue"]).sum()),
        "Known analogue":  int(LIBRARY["is_known_analogue"].sum()),
    }
    cols = [SEQ_BLUE[1], SEQ_BLUE[3], PALETTE["active"], SEQ_BLUE[5], SEQ_BLUE[7]]
    bars = ax.bar(range(len(part)), list(part.values()), color=cols, width=0.62,
                  edgecolor=SURFACE, linewidth=2)
    for b, v in zip(bars, part.values()):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center",
                va="bottom", fontsize=8.5, fontweight="600", color=INK["primary"])
    ax.set_xticks(range(len(part)))
    ax.set_xticklabels([textwrap.fill(k, 10) for k in part], fontsize=8)
    ax.set_ylabel("Compounds")
    ax.set_ylim(0, max(part.values()) * 1.18 if max(part.values()) else 1)
    ax.set_title("Deck partitioned by similarity to known ligands", loc="left")
    ax.grid(alpha=0.7)

    fig.suptitle("Applicability domain and novelty — where the screenable "
                 "compounds actually are",
                 x=0.01, ha="left", fontsize=12, fontweight="600")
    fig.tight_layout()
    save_fig(fig, "fig09_nearest_neighbour_applicability")
    plt.show()

    if SYNTHETIC_MODE:
        print("\n*** SYNTHETIC MODE: 'actives' came from the library itself, so "
              "similarities are inflated to ~1.0 and this figure is meaningless. ***")
else:
    print("No nearest-neighbour data — skipping figure 9.")

## 2.6 Persisted artefacts and the Phase 2 validation gate

Fingerprints are saved in SciPy's sparse `.npz` format rather than pickled RDKit objects: the
sparse matrices are small, portable across RDKit versions, and are what Phase 3's models consume.
The bit-vector objects are regenerated on demand from the stored SMILES when Tanimoto arithmetic
is needed — cheap, and it avoids serialising version-sensitive C++ objects.

In [ ]:
# --- Persist Phase 2 artefacts ----------------------------------------------
FP_DIR = CFG.paths["curated"] / "fingerprints"
FP_DIR.mkdir(parents=True, exist_ok=True)

for kind in FP_KINDS:
    sparse.save_npz(FP_DIR / f"library_{kind}.npz", LIB_FP[kind]["matrix"])
    if ACT_FP:
        sparse.save_npz(FP_DIR / f"actives_{kind}.npz", ACT_FP[kind]["matrix"])

np.savez_compressed(FP_DIR / "embeddings.npz",
                    pca=PCA_EMB, umap=UMAP_EMB, fp_svd=FP_SVD,
                    groups=groups, space_groups=space_groups)

LIBRARY.to_csv(CFG.paths["curated"] / "library_phase2.csv", index=False)
if len(ACTIVES):
    ACTIVES.to_csv(CFG.paths["curated"] / "chembl_a2a_phase2.csv", index=False)
BUTINA_SCAN.to_csv(CFG.paths["tables"] / "butina_cutoff_scan.csv", index=False)

(LIBRARY[LIBRARY["is_cluster_rep"]]
 [["compound_id", "std_smiles", "cluster", "mw", "clogp", "qed",
   "nn_sim_ecfp4", "nn_sim_fcfp4", "murcko_scaffold"]]
 .to_csv(CFG.paths["tables"] / "cluster_representatives.csv", index=False))

PHASE2_METRICS = {
    "fingerprints":            list(FP_KINDS),
    "fingerprint_size":        FP_SIZE,
    "descriptor_features":     len(DESCRIPTOR_COLUMNS),
    "pca_variance_pc1_pc2":    round(float(evr[:2].sum()), 4),
    "svd_variance_50comp":     round(float(svd.explained_variance_ratio_.sum()), 4),
    "umap_used":               bool(HAVE_UMAP),
    "butina_cutoff":           BUTINA_CUTOFF,
    "butina_sample_size":      int(BUTINA_SAMPLE),
    "kmeans_k":                int(K_FULL),
    "kmeans_silhouette":       (None if not np.isfinite(sil) else round(float(sil), 4)),
    "cluster_representatives": int(LIBRARY["is_cluster_rep"].sum()),
    "median_cluster_size":     float(sizes.median()),
    "unique_scaffolds":        int(LIBRARY["murcko_scaffold"].nunique()),
    "mean_nn_sim_ecfp4":       (None if LIBRARY["nn_sim_ecfp4"].isna().all()
                                else round(float(LIBRARY["nn_sim_ecfp4"].mean()), 4)),
    "in_applicability_domain": int(LIBRARY["in_applicability_domain"].sum()),
    "in_novelty_band":         int(LIBRARY["in_novelty_band"].sum()),
    "scaffold_hop_candidates": int(LIBRARY["scaffold_hop_candidate"].sum()),
    "synthetic_mode":          bool(SYNTHETIC_MODE),
}
(CFG.paths["metrics"] / "phase2_metrics.json").write_text(
    json.dumps(PHASE2_METRICS, indent=2))
RUN_STAMP["phase2_metrics"] = PHASE2_METRICS
RUN_STAMP["phase"] = 2
(CFG.paths["metrics"] / "run_manifest.json").write_text(json.dumps(RUN_STAMP, indent=2))

print("Phase 2 artefacts written")
print("-" * 70)
for p in sorted(FP_DIR.glob("*")) + [
        CFG.paths["curated"] / "library_phase2.csv",
        CFG.paths["tables"] / "butina_cutoff_scan.csv",
        CFG.paths["tables"] / "cluster_representatives.csv",
        CFG.paths["metrics"] / "phase2_metrics.json"]:
    if p.exists():
        print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# --- Phase 2 validation gate ------------------------------------------------
def validate_phase2():
    checks, failures = [], []

    def check(name, condition, detail=""):
        ok = bool(condition)
        checks.append((name, ok, detail))
        if not ok:
            failures.append(f"{name} — {detail}")

    n = len(LIBRARY)

    # Fingerprint integrity
    for kind in FP_KINDS:
        m = LIB_FP[kind]["matrix"]
        check(f"{kind}: one row per library compound", m.shape[0] == n,
              f"{m.shape[0]} vs {n}")
        check(f"{kind}: no all-zero fingerprints",
              np.asarray(m.sum(axis=1)).ravel().min() > 0, "empty fingerprint present")
    check("ECFP4 and FCFP4 differ",
          (LIB_FP["ecfp4"]["matrix"] != LIB_FP["fcfp4"]["matrix"]).nnz > 0,
          "identical matrices — invariant generator not applied")

    # Descriptor block
    check("Descriptor block has no NaN",
          not LIBRARY[DESCRIPTOR_COLUMNS].isna().any().any(),
          f"{int(LIBRARY[DESCRIPTOR_COLUMNS].isna().sum().sum())} NaNs")
    check("Descriptor block is finite",
          np.isfinite(LIBRARY[DESCRIPTOR_COLUMNS].values).all(), "inf present")
    check("Descriptor count >= 25", len(DESCRIPTOR_COLUMNS) >= 25,
          f"{len(DESCRIPTOR_COLUMNS)}")

    # Embeddings — the shared-space invariant this whole section rests on
    check("PCA embedding covers all molecules", PCA_EMB.shape[0] == len(combined),
          f"{PCA_EMB.shape[0]} vs {len(combined)}")
    check("UMAP embedding covers all molecules",
          UMAP_EMB.shape[0] == FP_COMBINED.shape[0],
          f"{UMAP_EMB.shape[0]} vs {FP_COMBINED.shape[0]}")
    check("Library occupies the leading embedding rows",
          (space_groups[:n] == "Enamine library").all(),
          "row ordering broken — library/actives slices would be mixed")
    check("Embeddings are finite",
          np.isfinite(PCA_EMB).all() and np.isfinite(UMAP_EMB).all(), "NaN/inf")

    # Clustering
    check("Every compound assigned a cluster",
          LIBRARY["cluster"].notna().all() and LIBRARY["cluster"].min() >= 0,
          "unassigned compounds")
    check("Cluster count matches k",
          LIBRARY["cluster"].nunique() <= K_FULL, "more clusters than k")
    check("One representative per populated cluster",
          LIBRARY["is_cluster_rep"].sum() == LIBRARY["cluster"].nunique(),
          f"{LIBRARY['is_cluster_rep'].sum()} reps vs "
          f"{LIBRARY['cluster'].nunique()} clusters")

    # Nearest-neighbour columns
    for c in ("nn_sim_ecfp4", "nn_sim_fcfp4"):
        vals = LIBRARY[c].dropna()
        check(f"{c} within [0, 1]",
              vals.empty or (vals.between(0, 1).all()), "out-of-range similarity")
    check("Applicability-domain flags present",
          {"in_applicability_domain", "in_novelty_band",
           "scaffold_hop_candidate"} <= set(LIBRARY.columns), "flags missing")

    # Artefacts
    for p in (FP_DIR / "library_ecfp4.npz",
              CFG.paths["curated"] / "library_phase2.csv",
              CFG.paths["metrics"] / "phase2_metrics.json"):
        check(f"Artefact exists: {p.name}", p.exists(), str(p))
    n_fig = len(list(CFG.paths["figures"].glob("*.png")))
    check("Phase 2 figures written", n_fig >= 8, f"{n_fig} PNGs")

    print("Phase 2 validation gate")
    print("=" * 74)
    for name, ok, detail in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if not ok else ""))
    print("=" * 74)
    print(f"{sum(1 for _, ok, _ in checks if ok)}/{len(checks)} checks passed")
    if failures:
        raise AssertionError("Phase 2 validation FAILED:\n  - " + "\n  - ".join(failures))
    print("\nPhase 2 validation PASSED — safe to proceed to Phase 3.")
    return True


validate_phase2()

In [ ]:
# --- Phase 2 summary --------------------------------------------------------
m2 = PHASE2_METRICS
print("=" * 74)
print(f"PHASE 2 COMPLETE — representation, chemical space, diversity")
print("=" * 74)
print(f"""
REPRESENTATION
  fingerprints ................... {', '.join(m2['fingerprints'])}
  fingerprint width .............. {m2['fingerprint_size']} bits
  descriptor features ............ {m2['descriptor_features']}

CHEMICAL SPACE
  PCA  PC1+PC2 variance .......... {m2['pca_variance_pc1_pc2']:.1%}
  SVD  50-component variance ..... {m2['svd_variance_50comp']:.1%}
  UMAP used ...................... {m2['umap_used']}

DIVERSITY
  Butina cut-off (calibrated) .... {m2['butina_cutoff']}  (on {m2['butina_sample_size']:,} cmpds)
  k-means clusters ............... {m2['kmeans_k']:,}
  median cluster size ............ {m2['median_cluster_size']:.0f}
  silhouette (5k, cosine) ........ {m2['kmeans_silhouette']}
  unique Murcko scaffolds ........ {m2['unique_scaffolds']:,}

SIMILARITY TO KNOWN LIGANDS
  mean ECFP4 NN similarity ....... {m2['mean_nn_sim_ecfp4']}
  in applicability domain ........ {m2['in_applicability_domain']:,}
  in novelty band (0.30-0.55) .... {m2['in_novelty_band']:,}
  scaffold-hop candidates ........ {m2['scaffold_hop_candidates']:,}
""")
if SYNTHETIC_MODE:
    print("  *** SYNTHETIC MODE: every similarity-to-actives number above is")
    print("  *** meaningless. Library-only results (fingerprints, clusters,")
    print("  *** scaffolds, descriptors) are real.\n")
print("=" * 74)

---

## Phase 2 complete

**New artefacts**

| Artefact | Contents |
|---|---|
| `outputs/data/curated/library_phase2.csv` | the deck with extended descriptors, cluster assignments, representative flags, nearest-active similarities, and applicability-domain flags |
| `outputs/data/curated/fingerprints/*.npz` | sparse ECFP4/6, FCFP4, RDKit, MACCS matrices for both datasets |
| `outputs/data/curated/fingerprints/embeddings.npz` | the shared PCA, UMAP, and 50-component SVD embeddings |
| `outputs/results/tables/butina_cutoff_scan.csv` | the cut-off calibration underlying the clustering choice |
| `outputs/results/tables/cluster_representatives.csv` | one representative per cluster — Phase 4's diversity pool |
| `outputs/results/figures/fig07–09*.png` | chemical space, clustering/diversity, nearest-neighbour & applicability domain |

**New objects for Phase 3:** `LIB_FP`, `ACT_FP`, `FP_KINDS`, `DESCRIPTOR_COLUMNS`, `FP_SVD`,
`PCA_EMB`, `UMAP_EMB`, `K_FULL`, `BUTINA_CUTOFF`, `AD_THRESHOLD`, `NOVELTY_BAND`,
`compute_fingerprints()`, `nearest_active_similarity()`, `load_chembl_export()`.

---

### Phase 3 will implement

1. **Scaffold-aware splitting** — Murcko-scaffold-disjoint train/validation/test splits, so no
   scaffold appears on both sides. Random splits on bioactivity data inflate ROC-AUC by a wide
   margin because analogue series straddle the split; the scaffold split is the honest number, and
   both are reported side by side to show the size of the effect.
2. **Class-imbalance handling** — class weighting and threshold optimisation on the validation
   split rather than blind SMOTE, which fabricates chemically meaningless interpolated
   fingerprints.
3. **Model panel** — logistic regression on fingerprints (interpretable baseline), random forest,
   and gradient boosting on fingerprint + descriptor blocks, each evaluated across all five
   representations, benchmarked against the **nearest-neighbour similarity baseline** from §2.5.
4. **Virtual-screening metrics** — ROC-AUC, PR-AUC (the honest metric under imbalance), enrichment
   factor at 0.5/1/5%, BEDROC (α=20), and Boltzmann-enhanced discrimination, with **bootstrap
   confidence intervals** on every one.
5. **Prospective scoring** — the frozen best model applied to all 48,489 library compounds, with
   per-compound uncertainty from ensemble variance, and predictions outside the applicability
   domain flagged rather than silently ranked.

---
# Phase 3 — Supervised modelling, honest evaluation, and prospective scoring

Phase 3 trains the model that scores the library. The modelling itself is routine; what determines
whether the output is trustworthy is the *evaluation protocol*, and that is where most of this
phase's design effort goes.

Three commitments, each of which makes the reported numbers worse and more honest:

**1. Scaffold-disjoint splitting is the headline number.** Bioactivity datasets are built from
analogue series — a paper reports twenty close variants of one core. Under a random split, members
of the same series land on both sides, so the model is tested on near-copies of its training data.
The resulting ROC-AUC is not a measure of prospective performance; it is a measure of memorisation.
This notebook reports **both** splits side by side, because the *gap* between them is the most
informative single number about how well the model will actually generalise to a library it has
never seen. A portfolio piece that reports only the random split is hiding this.

**2. The nearest-neighbour similarity baseline is reported as a peer, not a footnote.** Ranking
library compounds by Tanimoto similarity to the nearest known active requires no machine learning
at all. Any model that does not beat it has not earned its complexity. This baseline beats
published models more often than the literature admits.

**3. Every metric carries a bootstrap confidence interval.** A single ROC-AUC of 0.82 on a test set
of 400 compounds is a point estimate with roughly ±0.05 of sampling noise around it. Reporting it
bare invites over-reading differences between models that are not distinguishable.

**On class imbalance:** this notebook uses class weighting and decision-threshold optimisation, and
deliberately does **not** use SMOTE. SMOTE interpolates between feature vectors to synthesise
minority-class examples. On a molecular fingerprint, the midpoint between two molecules is not a
molecule — it is a bit pattern corresponding to no chemical structure that could ever be
synthesised. Training on such points teaches the model about a region of space that does not exist.

**Sections**

1. **3.0** Assembling the modelling matrices
2. **3.1** Scaffold-disjoint and random splits
3. **3.2** Virtual-screening metrics with bootstrap confidence intervals
4. **3.3** The nearest-neighbour similarity baseline
5. **3.4** Model panel — representations × algorithms
6. **3.5** Split comparison and the generalisation gap
7. **3.6** Final model, calibration, and threshold selection
8. **3.7** Prospective scoring of the library with uncertainty
9. **3.8** Persisted artefacts and the Phase 3 validation gate

## 3.0 Assembling the modelling matrices

Each of the five representations from Phase 2 becomes a feature matrix. Two variants are built per
fingerprint: the fingerprint alone, and the fingerprint concatenated with the standardised
descriptor block. Tree ensembles often gain from the descriptors, linear models usually do not, and
Phase 3 measures this rather than assuming it.

The descriptor scaler is fitted on the **training split only** and applied to validation, test, and
the library. Fitting it on everything leaks distributional information from the test set into
training — a small leak, but a free one to avoid.

In [ ]:
# --- Phase 3 imports ---------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, accuracy_score,
                             precision_score, recall_score, f1_score,
                             matthews_corrcoef, confusion_matrix)
from rdkit.ML.Scoring import Scoring          # RDKit's reference BEDROC / EF

# --- Preconditions -----------------------------------------------------------
# Phase 3 depends on a real, labelled training set. Check that explicitly and
# fail with an actionable message, rather than letting a missing column surface
# as a KeyError twenty cells later.
def require_labelled_set():
    if globals().get("SYNTHETIC_MODE", False):
        raise RuntimeError(
            "SYNTHETIC_MODE is active — the labelled set is a random stand-in.\n"
            "Phase 3 requires real ChEMBL bioactivity data.")
    if "ACTIVES" not in globals() or ACTIVES is None or len(ACTIVES) == 0:
        raise RuntimeError(
            "The labelled set (ACTIVES) is empty. Phase 1 section 1.4 did not\n"
            "produce training data. Check the ChEMBL retrieval audit above:\n"
            "  - set CFG.chembl_csv_fallback to a downloaded ChEMBL export, or\n"
            "  - restore network access to ebi.ac.uk,\n"
            "then re-run Phase 1 from section 1.4.")
    need = {"label", "activity_class", "pactivity", "murcko_scaffold",
            "std_smiles", "inchikey"}
    missing = need - set(ACTIVES.columns)
    if missing:
        raise RuntimeError(
            f"ACTIVES is missing required column(s): {sorted(missing)}.\n"
            f"Re-run Phase 1 sections 1.4 and 1.5 to rebuild it.")
    if ACTIVES["activity_class"].nunique() < 2:
        raise RuntimeError(
            f"Only one class present in the labelled set "
            f"({ACTIVES['activity_class'].unique()}). Widen the active/inactive\n"
            f"cut-offs in CFG, or check the bioactivity curation audit.")
    if len(ACTIVES) < 100:
        print(f"  WARNING: only {len(ACTIVES)} labelled compounds. Metrics will "
              f"have very wide confidence intervals.")
    return True


require_labelled_set()

N_BOOTSTRAP = 1000
print(f"Labelled set : {len(ACTIVES):,} compounds")
print(ACTIVES["activity_class"].value_counts().to_string())
print(f"\nBootstrap resamples per metric: {N_BOOTSTRAP}")

In [ ]:
# --- Feature matrix assembly -------------------------------------------------
y = ACTIVES["label"].values.astype(int)
print(f"Labels: {y.sum():,} active / {(1 - y).sum():,} inactive "
      f"({100 * y.mean():.1f}% positive)")

DESC_ACT = ACTIVES[DESCRIPTOR_COLUMNS].astype(float).values
DESC_LIB = LIBRARY[DESCRIPTOR_COLUMNS].astype(float).values


def build_feature_sets():
    """
    representation name -> {"act": matrix, "lib": matrix, "sparse": bool}

    Fingerprint-only variants stay sparse; '+desc' variants are densified
    because the descriptor block is dense anyway.
    """
    sets = {}
    for kind in FP_KINDS:
        sets[kind] = {
            "act": ACT_FP[kind]["matrix"].astype(np.float32),
            "lib": LIB_FP[kind]["matrix"].astype(np.float32),
            "sparse": True,
        }
    # Descriptor-augmented variants for the two most informative fingerprints.
    for kind in ("ecfp4", "fcfp4"):
        sets[f"{kind}+desc"] = {
            "act": ACT_FP[kind]["matrix"].toarray().astype(np.float32),
            "lib": None,            # assembled lazily at scoring time (memory)
            "lib_fp": LIB_FP[kind]["matrix"],
            "sparse": False,
            "needs_desc": True,
        }
    sets["descriptors"] = {
        "act": DESC_ACT.astype(np.float32),
        "lib": DESC_LIB.astype(np.float32),
        "sparse": False,
        "needs_desc": True,
        "desc_only": True,
    }
    return sets


FEATURE_SETS = build_feature_sets()
print(f"\nRepresentations: {len(FEATURE_SETS)}")
for name, fs in FEATURE_SETS.items():
    shape = fs["act"].shape
    print(f"  {name:<14} {shape[0]:>6,} x {shape[1]:<5} "
          f"{'sparse' if fs['sparse'] else 'dense'}")

## 3.1 Scaffold-disjoint and random splits

The scaffold splitter groups compounds by Bemis–Murcko scaffold and assigns **whole groups** to
train, validation, or test — so no scaffold ever appears on two sides. Groups are sorted
largest-first and assigned greedily to whichever split is furthest below its target size, which is
deterministic and keeps the splits close to the requested proportions despite the group structure.

The random splitter is a stratified shuffle at the compound level. It is included **only** as a
comparator, to quantify how much a random split inflates performance on this dataset.

Both splitters return train/validation/test. The validation split exists so that the decision
threshold and any model selection happen without ever touching the test set — using the test set to
pick a threshold and then reporting test performance at that threshold is a subtle but real leak.

In [ ]:
# --- Splitters ---------------------------------------------------------------
def scaffold_split(df, frac=(0.7, 0.15, 0.15), seed=CFG.random_seed,
                   scaffold_col="murcko_scaffold"):
    """
    Bemis-Murcko scaffold-disjoint split.

    Whole scaffold groups are assigned to a single split, largest group first,
    each going to whichever split is furthest below its quota. Compounds with an
    empty scaffold (acyclic) are treated as one group per compound.
    """
    groups = {}
    for i, sc in enumerate(df[scaffold_col].fillna("")):
        key = sc if sc else f"__acyclic_{i}"
        groups.setdefault(key, []).append(i)

    ordered = sorted(groups.values(), key=lambda g: (-len(g), g[0]))
    n = len(df)
    quota = [frac[0] * n, frac[1] * n, frac[2] * n]
    buckets = [[], [], []]
    for grp in ordered:
        deficit = [quota[k] - len(buckets[k]) for k in range(3)]
        buckets[int(np.argmax(deficit))].extend(grp)

    out = tuple(np.array(sorted(b)) for b in buckets)
    print(f"  scaffold split: {len(groups):,} scaffold groups -> "
          f"train {len(out[0]):,} / val {len(out[1]):,} / test {len(out[2]):,}")
    return out


def random_split(df, y, frac=(0.7, 0.15, 0.15), seed=CFG.random_seed):
    """Stratified random split at the compound level — the optimistic comparator."""
    rng = np.random.default_rng(seed)
    idx_tr, idx_va, idx_te = [], [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n1, n2 = int(frac[0] * len(idx)), int((frac[0] + frac[1]) * len(idx))
        idx_tr += list(idx[:n1]); idx_va += list(idx[n1:n2]); idx_te += list(idx[n2:])
    out = tuple(np.array(sorted(b)) for b in (idx_tr, idx_va, idx_te))
    print(f"  random split:  train {len(out[0]):,} / val {len(out[1]):,} / "
          f"test {len(out[2]):,}")
    return out


print("Building splits")
SPLITS = {}
SPLITS["scaffold"] = scaffold_split(ACTIVES)
SPLITS["random"] = random_split(ACTIVES, y)

# Report class balance and scaffold overlap for each — the overlap row is the
# entire point of the comparison.
print("\nSplit diagnostics")
print("-" * 78)
for name, (tr, va, te) in SPLITS.items():
    sc_tr = set(ACTIVES["murcko_scaffold"].iloc[tr])
    sc_te = set(ACTIVES["murcko_scaffold"].iloc[te])
    shared = len(sc_tr & sc_te)
    print(f"  {name:<9} positives  train {y[tr].mean():.3f} | "
          f"val {y[va].mean():.3f} | test {y[te].mean():.3f}")
    print(f"  {'':<9} scaffolds shared between train and test: {shared:,} "
          f"({100 * shared / max(len(sc_te), 1):.1f}% of test scaffolds)")

## 3.2 Virtual-screening metrics with bootstrap confidence intervals

Classification accuracy is close to useless here. With a 70/30 class split, a model that predicts
"active" for everything scores 0.70 and has learned nothing. The metrics that matter for virtual
screening are about **ranking**, because that is what the model is actually used for — you triage
the top 1% of a library, not a calibrated probability for each compound.

| Metric | What it measures | Why it is here |
|---|---|---|
| **ROC-AUC** | probability a random active ranks above a random inactive | the conventional number; comparable to literature, but optimistic under imbalance |
| **PR-AUC** | precision across all recall levels | the honest headline under class imbalance — its baseline is the positive rate, not 0.5 |
| **Enrichment factor @ x%** | how many more actives appear in the top x% than by chance | directly interpretable as "how much screening effort does this save" |
| **BEDROC (α=20)** | early-recognition-weighted ranking | weights the very top of the list, which is the only part anyone looks at; α=20 ≈ 80% of the weight in the top 8% |
| **MCC** | balanced correlation for binary classification | robust single-number summary at a fixed threshold |

EF and BEDROC use **RDKit's reference implementations** (`rdkit.ML.Scoring`) rather than
hand-rolled ones, since both have sign and normalisation conventions that are easy to get subtly
wrong.

**Bootstrap intervals** resample the test set with replacement 1,000 times and take the 2.5th and
97.5th percentiles. This captures sampling uncertainty from a finite test set — it does *not*
capture uncertainty from the choice of split, which is larger and is addressed by reporting the
scaffold/random comparison instead.

In [ ]:
# --- Metric suite ------------------------------------------------------------
def enrichment_and_bedroc(y_true, scores, fractions=(0.005, 0.01, 0.05), alpha=20.0):
    """RDKit reference EF and BEDROC. Input is ranked internally, descending."""
    order = np.argsort(-np.asarray(scores))
    ranked = [[int(y_true[i])] for i in order]
    ef = Scoring.CalcEnrichment(ranked, 0, list(fractions))
    bedroc = Scoring.CalcBEDROC(ranked, 0, alpha)
    rie = Scoring.CalcRIE(ranked, 0, alpha)
    return {**{f"ef_{f * 100:g}pct": float(v) for f, v in zip(fractions, ef)},
            "bedroc_a20": float(bedroc), "rie_a20": float(rie)}


def vs_metrics(y_true, scores, threshold=None):
    """Full ranking + classification metric suite for one prediction vector."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    m = {
        "roc_auc": roc_auc_score(y_true, scores),
        "pr_auc":  average_precision_score(y_true, scores),
        "pr_auc_baseline": float(y_true.mean()),
    }
    m.update(enrichment_and_bedroc(y_true, scores))
    if threshold is not None:
        pred = (scores >= threshold).astype(int)
        m.update({
            "threshold": float(threshold),
            "accuracy":  accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall":    recall_score(y_true, pred, zero_division=0),
            "f1":        f1_score(y_true, pred, zero_division=0),
            "mcc":       matthews_corrcoef(y_true, pred),
        })
    return m


def bootstrap_metrics(y_true, scores, threshold=None, n=N_BOOTSTRAP,
                      seed=CFG.random_seed, ci=(2.5, 97.5)):
    """
    Percentile bootstrap CI over the test set.

    Resamples that end up with a single class are skipped — AUC is undefined
    there, and silently substituting 0.5 would bias the interval.
    """
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    point = vs_metrics(y_true, scores, threshold)

    draws, n_skipped = {k: [] for k in point}, 0
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            n_skipped += 1
            continue
        try:
            r = vs_metrics(y_true[idx], scores[idx], threshold)
        except Exception:                                # noqa: BLE001
            n_skipped += 1
            continue
        for k, v in r.items():
            draws[k].append(v)

    out = {}
    for k, v in point.items():
        arr = np.asarray(draws[k], dtype=float)
        if len(arr) < 50:
            out[k] = {"value": v, "lo": np.nan, "hi": np.nan, "n_draws": len(arr)}
        else:
            out[k] = {"value": v,
                      "lo": float(np.percentile(arr, ci[0])),
                      "hi": float(np.percentile(arr, ci[1])),
                      "n_draws": len(arr)}
    if n_skipped:
        print(f"    ({n_skipped}/{n} bootstrap draws skipped — single-class resample)")
    return out


def fmt_ci(d, key, prec=3):
    r = d[key]
    if np.isnan(r["lo"]):
        return f"{r['value']:.{prec}f}"
    return f"{r['value']:.{prec}f} [{r['lo']:.{prec}f}-{r['hi']:.{prec}f}]"


# Self-test on a synthetic ranking with a known answer.
_rng = np.random.default_rng(0)
_y = np.r_[np.ones(50), np.zeros(450)].astype(int)
_perfect = np.r_[np.ones(50), np.zeros(450)] + _rng.normal(0, 0.01, 500)
_random = _rng.normal(0, 1, 500)
print("Metric self-test")
print(f"  perfect ranking : ROC-AUC {vs_metrics(_y, _perfect)['roc_auc']:.3f} "
      f"(expect ~1.00), EF@1% {vs_metrics(_y, _perfect)['ef_1pct']:.1f} "
      f"(expect ~10 = 1/0.10)")
print(f"  random ranking  : ROC-AUC {vs_metrics(_y, _random)['roc_auc']:.3f} "
      f"(expect ~0.50), EF@1% {vs_metrics(_y, _random)['ef_1pct']:.1f} "
      f"(expect ~1.0)")

## 3.3 The nearest-neighbour similarity baseline

Before any model is fitted: how well does *"rank by Tanimoto similarity to the nearest training
active"* perform on the held-out test set?

This is the number every subsequent model must beat. It is computed under the identical split, so
the comparison is fair — the "training actives" are only those in the training fold, never the test
fold's own actives.

In [ ]:
# --- Similarity baseline under each split ------------------------------------
def similarity_baseline(train_idx, eval_idx, fp_kind="ecfp4", agg="max"):
    """Score each eval compound by its similarity to the nearest TRAIN active."""
    bvs = ACT_FP[fp_kind]["bitvects"]
    train_actives = [bvs[i] for i in train_idx if y[i] == 1]
    if not train_actives:
        return np.zeros(len(eval_idx))
    out = np.empty(len(eval_idx))
    for j, i in enumerate(eval_idx):
        sims = np.asarray(DataStructs.BulkTanimotoSimilarity(bvs[i], train_actives))
        out[j] = sims.max() if agg == "max" else np.sort(sims)[-5:].mean()
    return out


BASELINES = {}
print("Nearest-active similarity baseline")
print("=" * 78)
for split_name, (tr, va, te) in SPLITS.items():
    for fp_kind in ("ecfp4", "fcfp4"):
        sc = similarity_baseline(tr, te, fp_kind)
        res = bootstrap_metrics(y[te], sc)
        BASELINES[(split_name, fp_kind)] = res
        print(f"  {split_name:<9} {fp_kind:<6} "
              f"ROC-AUC {fmt_ci(res, 'roc_auc')}   "
              f"PR-AUC {fmt_ci(res, 'pr_auc')}   "
              f"BEDROC {fmt_ci(res, 'bedroc_a20')}")
print("=" * 78)
print("Any model below must beat these to justify its existence.")

## 3.4 Model panel — representations × algorithms

Three algorithms, chosen to span the complexity range rather than to win a benchmark:

* **Logistic regression** (L2, balanced class weights) — the interpretable linear baseline. On
  high-dimensional sparse fingerprints this is a genuinely strong model, not a straw man.
* **Random forest** — handles the sparse binary features and mixed descriptor scales without
  preprocessing, and its per-tree spread supplies the prediction uncertainty used in §3.7.
* **Histogram gradient boosting** — the strongest tabular learner here; used on dense
  descriptor-containing representations where it has the most to work with.

Every combination is fitted on the training fold, thresholded on the validation fold, and evaluated
on the test fold. **Selection uses validation PR-AUC**, never test performance.

In [ ]:
# --- Model factory -----------------------------------------------------------
def make_models(sparse_ok):
    """Algorithms appropriate to the representation's storage format."""
    m = {
        "logreg": LogisticRegression(
            penalty="l2", C=1.0, class_weight="balanced", max_iter=3000,
            solver="liblinear", random_state=CFG.random_seed),
        "random_forest": RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=2,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=CFG.random_seed),
    }
    if not sparse_ok:      # HistGB requires dense input
        m["hist_gb"] = HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.08, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
            random_state=CFG.random_seed)
    return m


def best_threshold(y_val, p_val):
    """Threshold maximising validation F1 — chosen WITHOUT touching the test set."""
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec * rec / np.clip(prec + rec, 1e-12, None)
    f1 = np.nan_to_num(f1[:-1])
    return float(thr[int(np.argmax(f1))]) if len(thr) else 0.5


def get_matrix(fs, idx, which="act"):
    """Slice a representation's matrix, appending scaled descriptors if needed."""
    X = fs[which][idx]
    return X


def fit_eval(fs_name, fs, model_name, model, tr, va, te, scaler=None):
    """Fit on train, threshold on validation, evaluate on test."""
    Xa = fs["act"]
    if fs.get("needs_desc") and not fs.get("desc_only"):
        # concatenate the TRAIN-fitted standardised descriptor block
        Xa = np.hstack([Xa, scaler.transform(DESC_ACT).astype(np.float32)])
    elif fs.get("desc_only"):
        Xa = scaler.transform(DESC_ACT).astype(np.float32)

    Xtr, Xva, Xte = Xa[tr], Xa[va], Xa[te]
    model.fit(Xtr, y[tr])
    p_va = model.predict_proba(Xva)[:, 1]
    p_te = model.predict_proba(Xte)[:, 1]
    thr = best_threshold(y[va], p_va)
    return {
        "val_pr_auc": average_precision_score(y[va], p_va),
        "val_roc_auc": roc_auc_score(y[va], p_va),
        "threshold": thr,
        "test_scores": p_te,
        "model": model,
    }


# --- Run the panel under BOTH splits ----------------------------------------
PANEL = []
FITTED = {}

for split_name, (tr, va, te) in SPLITS.items():
    # Descriptor scaler fitted on the TRAINING FOLD ONLY — no leakage.
    scaler = StandardScaler().fit(DESC_ACT[tr])
    FITTED[(split_name, "_scaler")] = scaler

    for fs_name, fs in FEATURE_SETS.items():
        for model_name, model in make_models(fs["sparse"]).items():
            t0 = time.time()
            try:
                r = fit_eval(fs_name, fs, model_name, model, tr, va, te, scaler)
            except Exception as exc:                     # noqa: BLE001
                print(f"  SKIP {split_name}/{fs_name}/{model_name}: "
                      f"{type(exc).__name__}: {exc}")
                continue
            te_m = vs_metrics(y[te], r["test_scores"], r["threshold"])
            PANEL.append({
                "split": split_name, "representation": fs_name, "model": model_name,
                "val_pr_auc": r["val_pr_auc"], "val_roc_auc": r["val_roc_auc"],
                "test_roc_auc": te_m["roc_auc"], "test_pr_auc": te_m["pr_auc"],
                "test_bedroc": te_m["bedroc_a20"], "test_ef1": te_m["ef_1pct"],
                "test_ef5": te_m["ef_5pct"], "test_mcc": te_m["mcc"],
                "test_f1": te_m["f1"], "threshold": r["threshold"],
                "fit_seconds": round(time.time() - t0, 1),
            })
            FITTED[(split_name, fs_name, model_name)] = r
            print(f"  {split_name:<9} {fs_name:<14} {model_name:<14} "
                  f"val PR-AUC {r['val_pr_auc']:.3f}  "
                  f"test PR-AUC {te_m['pr_auc']:.3f}  "
                  f"test ROC {te_m['roc_auc']:.3f}  "
                  f"({time.time() - t0:.0f}s)")

PANEL = pd.DataFrame(PANEL)
print(f"\nPanel complete: {len(PANEL)} model/representation/split combinations")

In [ ]:
# --- Panel results -----------------------------------------------------------
print("SCAFFOLD SPLIT — the honest evaluation, ranked by validation PR-AUC")
print("=" * 104)
sc = (PANEL[PANEL["split"] == "scaffold"]
      .sort_values("val_pr_auc", ascending=False))
print(sc[["representation", "model", "val_pr_auc", "test_pr_auc", "test_roc_auc",
          "test_bedroc", "test_ef1", "test_mcc"]].round(3).to_string(index=False))

# Selection happens on VALIDATION, so the winning row is chosen before any
# test number is consulted.
BEST = sc.iloc[0]
print(f"\nSelected (by validation PR-AUC): "
      f"{BEST['representation']} / {BEST['model']}")
print(f"  validation PR-AUC {BEST['val_pr_auc']:.3f}  ->  "
      f"test PR-AUC {BEST['test_pr_auc']:.3f}, ROC-AUC {BEST['test_roc_auc']:.3f}")

base = BASELINES[("scaffold", "ecfp4")]
print(f"\nVersus the nearest-neighbour baseline (scaffold split, ECFP4):")
print(f"  baseline  PR-AUC {fmt_ci(base, 'pr_auc')}   "
      f"ROC-AUC {fmt_ci(base, 'roc_auc')}")
print(f"  model     PR-AUC {BEST['test_pr_auc']:.3f}   "
      f"ROC-AUC {BEST['test_roc_auc']:.3f}")
delta = BEST["test_pr_auc"] - base["pr_auc"]["value"]
verdict = ("The model BEATS the similarity baseline."
           if delta > 0 else
           "The model DOES NOT beat the similarity baseline — "
           "prefer the baseline, or revisit the representation.")
print(f"  delta PR-AUC {delta:+.3f}  ->  {verdict}")

## 3.5 The generalisation gap

The same models, the same data, two splitting strategies. The difference between them is the cost
of the optimism that a random split buys.

Expect the random split to look substantially better. That gap is **not** a defect in this
pipeline — it is a property of bioactivity data that most published virtual-screening benchmarks
inherit silently. Reporting it is what makes the scaffold number credible.

In [ ]:
# --- Figure 10: scaffold vs random --------------------------------------------
pivot = PANEL.pivot_table(index=["representation", "model"], columns="split",
                          values=["test_roc_auc", "test_pr_auc"])
gap = pd.DataFrame({
    "roc_scaffold": pivot[("test_roc_auc", "scaffold")],
    "roc_random":   pivot[("test_roc_auc", "random")],
    "pr_scaffold":  pivot[("test_pr_auc", "scaffold")],
    "pr_random":    pivot[("test_pr_auc", "random")],
}).dropna()
gap["roc_gap"] = gap["roc_random"] - gap["roc_scaffold"]
gap["pr_gap"]  = gap["pr_random"] - gap["pr_scaffold"]
gap = gap.sort_values("pr_scaffold", ascending=False)

print("Generalisation gap (random minus scaffold)")
print("=" * 88)
print(gap.round(3).to_string())
print("=" * 88)
print(f"Mean ROC-AUC inflation from random splitting: {gap['roc_gap'].mean():+.3f}")
print(f"Mean PR-AUC  inflation from random splitting: {gap['pr_gap'].mean():+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
labels = [f"{r}\n{m}" for r, m in gap.index]
ypos = np.arange(len(gap))[::-1]

for ax, (a, b, title, xlab) in zip(axes, [
        ("pr_scaffold", "pr_random", "PR-AUC — the honest metric under imbalance", "PR-AUC"),
        ("roc_scaffold", "roc_random", "ROC-AUC", "ROC-AUC")]):
    ax.barh(ypos + 0.19, gap[b], height=0.36, color=PALETTE["inactive"],
            edgecolor=SURFACE, linewidth=1.5, label="random split (optimistic)")
    ax.barh(ypos - 0.19, gap[a], height=0.36, color=PALETTE["library"],
            edgecolor=SURFACE, linewidth=1.5, label="scaffold split (honest)")
    for yp, va, vb in zip(ypos, gap[a], gap[b]):
        ax.text(vb + 0.008, yp + 0.19, f"{vb:.3f}", va="center", fontsize=7.5,
                color=INK["primary"], fontweight="600")
        ax.text(va + 0.008, yp - 0.19, f"{va:.3f}", va="center", fontsize=7.5,
                color=INK["primary"], fontweight="600")
    ax.set_yticks(ypos); ax.set_yticklabels(labels, fontsize=7.5)
    ax.set_xlabel(xlab); ax.set_xlim(0, 1.13)
    ax.set_title(title, loc="left")
    ax.legend(loc="lower right"); ax.grid(axis="x", alpha=0.7)
    ax.grid(axis="y", visible=False)

fig.suptitle("Generalisation gap — what a random split hides",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig10_generalisation_gap")
plt.show()

In [ ]:
# --- Figure 11: panel heatmap (scaffold split) -------------------------------
hm = (PANEL[PANEL["split"] == "scaffold"]
      .pivot_table(index="representation", columns="model", values="test_pr_auc"))
hm = hm.loc[hm.mean(axis=1).sort_values(ascending=False).index]

from matplotlib.colors import LinearSegmentedColormap
BLUES = LinearSegmentedColormap.from_list("seq_blue", SEQ_BLUE)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
im = ax.imshow(hm.values, cmap=BLUES, aspect="auto",
               vmin=np.nanmin(hm.values), vmax=np.nanmax(hm.values))
ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns, fontsize=9)
ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index, fontsize=9)
# Every cell is directly labelled, so the value never depends on reading colour.
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        v = hm.values[i, j]
        if np.isnan(v):
            ax.text(j, i, "n/a", ha="center", va="center", fontsize=8.5,
                    color=INK["muted"]); continue
        rel = (v - np.nanmin(hm.values)) / max((np.nanmax(hm.values) - np.nanmin(hm.values)), 1e-9)
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=9,
                color="#ffffff" if rel > 0.55 else INK["primary"], fontweight="600")
ax.set_xlabel("Algorithm"); ax.set_ylabel("Representation")
ax.set_title(f"Test PR-AUC, scaffold split\n"
             f"(positive-class baseline = {y.mean():.3f})", loc="left")
ax.grid(visible=False)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cb.set_label("Test PR-AUC", fontsize=9)
cb.outline.set_visible(False)
fig.tight_layout()
save_fig(fig, "fig11_model_panel_heatmap")
plt.show()

## 3.6 Final model — curves, calibration, and full metrics

The selected model is now characterised in full on the scaffold-split test set: ROC and
precision–recall curves with bootstrap bands, an enrichment curve, a calibration plot, and the
complete metric table with confidence intervals.

**On calibration:** a model can rank perfectly and still output badly-scaled probabilities. Ranking
is what matters for hit selection, so poor calibration is not disqualifying — but it must be known,
because Phase 4's composite score combines this probability with a docking score, and combining
an over-confident probability with anything else propagates that over-confidence into the ranking.

In [ ]:
# --- Characterise the selected model -----------------------------------------
tr, va, te = SPLITS["scaffold"]
BEST_KEY = ("scaffold", BEST["representation"], BEST["model"])
best_fit = FITTED[BEST_KEY]
BEST_SCORES = best_fit["test_scores"]
BEST_THRESHOLD = best_fit["threshold"]

FINAL_METRICS = bootstrap_metrics(y[te], BEST_SCORES, BEST_THRESHOLD)

print(f"FINAL MODEL — {BEST['representation']} / {BEST['model']}, scaffold split")
print("=" * 78)
print(f"  test set: {len(te):,} compounds, {int(y[te].sum()):,} active "
      f"({y[te].mean():.1%} positive)")
print(f"  decision threshold (from validation F1): {BEST_THRESHOLD:.3f}\n")
ROWS = [("ROC-AUC", "roc_auc"), ("PR-AUC", "pr_auc"),
        ("PR-AUC baseline", "pr_auc_baseline"),
        ("BEDROC (a=20)", "bedroc_a20"), ("RIE (a=20)", "rie_a20"),
        ("EF @ 0.5%", "ef_0.5pct"), ("EF @ 1%", "ef_1pct"), ("EF @ 5%", "ef_5pct"),
        ("Accuracy", "accuracy"), ("Precision", "precision"), ("Recall", "recall"),
        ("F1", "f1"), ("MCC", "mcc")]
for label, key in ROWS:
    if key in FINAL_METRICS:
        print(f"  {label:<20} {fmt_ci(FINAL_METRICS, key)}")
print("=" * 78)
print("  Intervals are 95% percentile bootstrap over "
      f"{N_BOOTSTRAP} test-set resamples.")

cm = confusion_matrix(y[te], (BEST_SCORES >= BEST_THRESHOLD).astype(int))
print(f"\n  Confusion matrix at threshold {BEST_THRESHOLD:.3f}:")
print(f"                 pred inactive   pred active")
print(f"    inactive      {cm[0, 0]:>8,}      {cm[0, 1]:>8,}")
print(f"    active        {cm[1, 0]:>8,}      {cm[1, 1]:>8,}")

In [ ]:
# --- Figure 12: ROC, PR, enrichment, calibration ----------------------------
def bootstrap_curve(y_true, scores, kind="roc", n=300, seed=CFG.random_seed):
    """Bootstrap band for a ROC or PR curve on a common grid."""
    rng = np.random.default_rng(seed)
    grid = np.linspace(0, 1, 101)
    curves = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        if kind == "roc":
            a, b, _ = roc_curve(y_true[idx], scores[idx])
        else:
            b, a, _ = precision_recall_curve(y_true[idx], scores[idx])
            a, b = a[::-1], b[::-1]
        curves.append(np.interp(grid, a, b))
    arr = np.asarray(curves)
    return grid, np.percentile(arr, 2.5, axis=0), np.percentile(arr, 97.5, axis=0)


fig, axes = plt.subplots(1, 4, figsize=(18, 4.3))
base_ecfp = BASELINES[("scaffold", "ecfp4")]
base_scores = similarity_baseline(tr, te, "ecfp4")

# (a) ROC with bootstrap band
ax = axes[0]
fpr, tpr, _ = roc_curve(y[te], BEST_SCORES)
g, lo, hi = bootstrap_curve(y[te], BEST_SCORES, "roc")
ax.fill_between(g, lo, hi, color=PALETTE["library"], alpha=0.18, lw=0)
ax.plot(fpr, tpr, color=PALETTE["library"], lw=2.2,
        label=f"model  AUC {FINAL_METRICS['roc_auc']['value']:.3f}")
b_fpr, b_tpr, _ = roc_curve(y[te], base_scores)
ax.plot(b_fpr, b_tpr, color=PALETTE["active"], lw=2, ls="-",
        label=f"NN baseline  AUC {base_ecfp['roc_auc']['value']:.3f}")
ax.plot([0, 1], [0, 1], color=INK["muted"], lw=1.2, ls="--", label="random  0.500")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC (band = 95% bootstrap)", loc="left")
ax.legend(loc="lower right"); ax.grid(alpha=0.7)

# (b) Precision-recall
ax = axes[1]
prec, rec, _ = precision_recall_curve(y[te], BEST_SCORES)
g, lo, hi = bootstrap_curve(y[te], BEST_SCORES, "pr")
ax.fill_between(g, lo, hi, color=PALETTE["library"], alpha=0.18, lw=0)
ax.plot(rec, prec, color=PALETTE["library"], lw=2.2,
        label=f"model  AP {FINAL_METRICS['pr_auc']['value']:.3f}")
b_p, b_r, _ = precision_recall_curve(y[te], base_scores)
ax.plot(b_r, b_p, color=PALETTE["active"], lw=2,
        label=f"NN baseline  AP {base_ecfp['pr_auc']['value']:.3f}")
ax.axhline(y[te].mean(), color=INK["muted"], lw=1.2, ls="--",
           label=f"baseline  {y[te].mean():.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-recall", loc="left")
ax.legend(loc="lower left"); ax.grid(alpha=0.7)

# (c) Enrichment curve
ax = axes[2]
order = np.argsort(-BEST_SCORES)
cum = np.cumsum(y[te][order]) / max(y[te].sum(), 1)
frac = np.arange(1, len(te) + 1) / len(te)
b_order = np.argsort(-base_scores)
b_cum = np.cumsum(y[te][b_order]) / max(y[te].sum(), 1)
ax.plot(frac, cum, color=PALETTE["library"], lw=2.2, label="model")
ax.plot(frac, b_cum, color=PALETTE["active"], lw=2, label="NN baseline")
ax.plot([0, 1], [0, 1], color=INK["muted"], lw=1.2, ls="--", label="random")
for f in (0.01, 0.05):
    ax.axvline(f, color="#d8d7d3", lw=1, zorder=0)
    ax.annotate(f"{f:.0%}", (f, 0.03), fontsize=7.5, rotation=90,
                color=INK["secondary"], ha="right")
ax.set_xlabel("Fraction of ranked test set screened")
ax.set_ylabel("Fraction of actives recovered")
ax.set_title(f"Enrichment  (EF@1% = {FINAL_METRICS['ef_1pct']['value']:.2f})", loc="left")
ax.legend(loc="lower right"); ax.grid(alpha=0.7)

# (d) Calibration
ax = axes[3]
try:
    pt, pp = calibration_curve(y[te], BEST_SCORES, n_bins=8, strategy="quantile")
    ax.plot(pp, pt, "o-", color=PALETTE["library"], markersize=7, lw=2,
            label="model")
    ax.plot([0, 1], [0, 1], color=INK["muted"], lw=1.2, ls="--",
            label="perfect calibration")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed active fraction")
    ax.set_title("Calibration (8 quantile bins)", loc="left")
    ax.legend(loc="upper left"); ax.grid(alpha=0.7)
except Exception as exc:                                  # noqa: BLE001
    ax.set_visible(False)
    print(f"Calibration curve unavailable: {exc}")

fig.suptitle(f"Final model — {BEST['representation']} / {BEST['model']}, "
             f"scaffold-disjoint test set (n = {len(te):,})",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig12_final_model_performance")
plt.show()

## 3.7 Screening-realistic evaluation with property-matched decoys

**The evaluation above has a structural problem, and it is visible in the metrics themselves.**

The curated ChEMBL set is **82.7% active** (3,014 actives to 630 inactives). This is not a
curation error — it is publication bias. Medicinal chemists publish compounds that work; inactives
are recorded only when a paper happens to report its misses. A<sub>2A</sub> is a heavily-worked
target, so the imbalance runs strongly toward actives.

Three consequences, each of which undermines a metric reported in §3.6:

* **EF@1% is saturated.** With an 82.7% positive rate the maximum attainable enrichment factor is
  1/0.827 = **1.21**. Every model in the panel scored ≈1.23. They are not tied because they perform
  identically; they are tied because they are all pinned against the ceiling. The metric has no
  resolving power at this class balance.
* **BEDROC α=20 is saturated too**, at 1.000 for essentially every model, for the same reason.
* **PR-AUC's baseline is 0.827, not 0.5.** A PR-AUC of 0.985 is a real improvement over 0.827, but
  a far smaller one than the number suggests to a reader who assumes the usual low-prevalence
  setting.

Only ROC-AUC survives intact, because it is prevalence-independent by construction.

**Why this matters for this project specifically.** The deliverable is a ranked list drawn from
48,489 library compounds whose true hit rate is realistically 0.1–1%. Performance measured at 83%
prevalence says almost nothing about performance at 1% prevalence — and early-recognition metrics
are precisely the ones that break down.

### The fix: property-matched decoys

Following the DUD-E construction, decoys are drawn **from the Enamine library itself** and required
to be:

1. **Physicochemically matched** to a known active — MW, cLogP, TPSA, HBD, HBA, and formal charge
   all within tolerance. Without this, the model can separate actives from decoys on gross
   properties alone and the evaluation measures nothing.
2. **Topologically dissimilar** — ECFP4 Tanimoto below 0.35 to *every* known active, so a decoy is
   unlikely to be a genuine unreported binder.

Drawing decoys from the screening library is deliberate: the resulting benchmark simulates the
actual task — separating true A<sub>2A</sub> ligands from *this* deck at a realistic ratio.

**The honest caveat:** decoys are *presumed* inactive, not measured inactive. Some fraction will be
genuine binders, which makes the reported enrichment a mild **under**-estimate. This is the standard
assumption in virtual-screening benchmarking and it should be stated rather than buried.

In [ ]:
# --- Property-matched decoy construction ------------------------------------
# Two parameters matter, and they trade off against each other:
#
#   max_sim - the ECFP4 similarity ceiling a decoy may have to ANY known
#             active. Lower is safer (less chance a "decoy" is really a binder)
#             but introduces the circularity described below.
#   tol     - physicochemical matching tolerance, relaxed adaptively per active
#             until matches are found, so large or polar actives are not
#             silently dropped. Dropping them would skew the decoy set small
#             and non-polar, letting the model separate actives from decoys on
#             gross properties alone -- which would measure nothing.
DECOY_RATIO = 50
BASE_TOL = {"mw": 25.0, "clogp": 1.0, "tpsa": 20.0, "hbd": 1, "hba": 1}
RELAX_STEPS = [1.0, 1.6, 2.4, 3.5, 5.0]


def build_decoys(actives_df, pool_df, max_sim, ratio=DECOY_RATIO,
                 seed=CFG.random_seed, label=""):
    """Property-matched, topologically-dissimilar decoys with adaptive tolerance."""
    pool = pool_df[pool_df["nn_sim_ecfp4"] < max_sim]
    pv = pool[["mw", "clogp", "tpsa", "hbd", "hba", "formal_charge"]].values
    rng = np.random.default_rng(seed)
    chosen, relax_used, unmatched = set(), [], 0

    for r in tqdm(actives_df.itertuples(), total=len(actives_df),
                  desc=f"Decoys {label}", unit="active"):
        got = False
        for step in RELAX_STEPS:
            ok = ((np.abs(pv[:, 0] - r.mw)    <= BASE_TOL["mw"] * step) &
                  (np.abs(pv[:, 1] - r.clogp) <= BASE_TOL["clogp"] * step) &
                  (np.abs(pv[:, 2] - r.tpsa)  <= BASE_TOL["tpsa"] * step) &
                  (np.abs(pv[:, 3] - r.hbd)   <= BASE_TOL["hbd"] * step) &
                  (np.abs(pv[:, 4] - r.hba)   <= BASE_TOL["hba"] * step) &
                  (pv[:, 5] == r.formal_charge))
            cand = np.where(ok)[0]
            if len(chosen):
                cand = np.setdiff1d(cand, np.fromiter(chosen, int, len(chosen)))
            if len(cand):
                take = rng.choice(cand, size=min(ratio, len(cand)), replace=False)
                chosen.update(int(t) for t in take)
                relax_used.append(step)
                got = True
                break
        if not got:
            unmatched += 1

    decoys = pool.iloc[np.array(sorted(chosen))] if chosen else pool.iloc[[]]
    prev = len(actives_df) / max(len(actives_df) + len(decoys), 1)
    print(f"  pool {len(pool):,} | decoys {len(decoys):,} "
          f"({len(decoys) / max(len(actives_df), 1):.1f} per active) | "
          f"prevalence {prev:.2%} | EF ceiling {1 / max(prev, 1e-9):.0f}")
    if unmatched:
        print(f"  {unmatched} active(s) unmatched even at {RELAX_STEPS[-1]}x tolerance")
    if relax_used:
        print(f"  tolerance: median {np.median(relax_used):.1f}x, "
              f"{100 * np.mean(np.array(relax_used) == 1.0):.0f}% matched at 1x")
    return decoys


def property_match_report(actives_df, decoys_df, name):
    """Gross-property deltas. Large deltas would invalidate the benchmark."""
    print(f"\n  Property match — {name} (median active vs median decoy)")
    worst = 0.0
    for c, scale in [("mw", 50), ("clogp", 1.0), ("tpsa", 30), ("hbd", 1), ("hba", 2)]:
        a, d = actives_df[c].median(), decoys_df[c].median()
        norm = abs(d - a) / scale
        worst = max(worst, norm)
        print(f"    {c:<7} {a:>7.2f} vs {d:>7.2f}   delta {d - a:+7.2f}"
              + ("   <-- large" if norm > 1 else ""))
    return worst


test_actives = ACTIVES.iloc[te]
test_actives = test_actives[test_actives["label"] == 1]
print(f"Test-fold actives: {len(test_actives):,}\n")

DECOY_SETS = {}
for _name, _max_sim in [("strict", 0.35), ("relaxed", 0.60)]:
    print(f"[{_name}] ECFP4 similarity ceiling {_max_sim}")
    _d = build_decoys(test_actives, LIBRARY, _max_sim, label=_name)
    property_match_report(test_actives, _d, _name)
    DECOY_SETS[_name] = _d
    print()

DECOYS = DECOY_SETS["strict"]
DECOY_MAX_SIM = 0.35

### A circularity that has to be declared

The strict decoy set is built by requiring **ECFP4 similarity below 0.35 to every known active** —
and the nearest-neighbour baseline scores compounds by **exactly that quantity**. Every strict
decoy therefore sits below 0.35 by construction, while an active may sit anywhere. The benchmark
is structurally tilted in the similarity baseline's favour.

This is a real and under-acknowledged flaw in DUD-E-style benchmarking rather than a quirk of this
notebook: any decoy set built on topological dissimilarity flatters similarity-based scoring, so a
published comparison of a model against a similarity baseline on such a set is not a fair fight.

The **relaxed** set (0.60 ceiling) weakens the circularity — decoys may now be moderately similar
to actives — at the cost of admitting more compounds that might be genuine unreported binders.
Neither set is unbiased. Reporting both **brackets** the answer instead of pretending one number
settles it.

In [ ]:
# --- Evaluate the selected model on the realistic benchmark ------------------
def library_features(fs, idx, scaler):
    """Assemble the selected representation's features for library rows."""
    if fs.get("desc_only"):
        return scaler.transform(DESC_LIB[idx]).astype(np.float32)
    if fs.get("needs_desc"):
        return np.hstack([fs["lib_fp"][idx].toarray().astype(np.float32),
                          scaler.transform(DESC_LIB[idx]).astype(np.float32)])
    return fs["lib"][idx]


split_scaler = FITTED[("scaffold", "_scaler")]
fs_best = FEATURE_SETS[BEST["representation"]]
split_model = best_fit["model"]          # fitted on the TRAIN fold only

# Actives: reuse their existing test-fold predictions. Decoys: score now.
active_pos = np.where(ACTIVES.iloc[te]["label"].values == 1)[0]
active_scores = BEST_SCORES[active_pos]

BENCH = {}
nn_active = similarity_baseline(tr, te, "ecfp4")[active_pos]

for name, dset in DECOY_SETS.items():
    ridx = np.array([LIBRARY.index.get_loc(i) for i in dset.index])
    d_scores = split_model.predict_proba(
        library_features(fs_best, ridx, split_scaler))[:, 1]
    yb = np.r_[np.ones(len(active_scores)), np.zeros(len(d_scores))].astype(int)
    sb = np.r_[active_scores, d_scores]
    nb = np.r_[nn_active, LIBRARY["nn_sim_ecfp4"].values[ridx]]
    BENCH[name] = {
        "y": yb, "scores": sb, "nn": nb, "decoy_scores": d_scores,
        "prevalence": float(yb.mean()),
        "model": bootstrap_metrics(yb, sb, BEST_THRESHOLD),
        "baseline": bootstrap_metrics(yb, nb),
    }
    print(f"[{name}] {int(yb.sum()):,} actives + {int((1 - yb).sum()):,} decoys | "
          f"prevalence {yb.mean():.2%} | EF ceiling {1 / yb.mean():.0f}")

# Aliases so the figure and persistence cells read cleanly.
y_bench      = BENCH["strict"]["y"]
s_bench      = BENCH["strict"]["scores"]
bench_base   = BENCH["strict"]["nn"]
decoy_scores = BENCH["strict"]["decoy_scores"]
prevalence   = BENCH["strict"]["prevalence"]
BENCH_METRICS  = BENCH["strict"]["model"]
BENCH_BASELINE = BENCH["strict"]["baseline"]

for name in ("strict", "relaxed"):
    b = BENCH[name]
    print("\n" + "=" * 92)
    print(f"SCREENING-REALISTIC PERFORMANCE — {name} decoys "
          f"(prevalence {b['prevalence']:.2%}, EF ceiling {1 / b['prevalence']:.0f})")
    print("=" * 92)
    print(f"{'metric':<20} {'model':<28} {'NN similarity baseline':<28}")
    print("-" * 92)
    for label, key in [("ROC-AUC", "roc_auc"), ("PR-AUC", "pr_auc"),
                       ("PR-AUC baseline", "pr_auc_baseline"),
                       ("BEDROC (a=20)", "bedroc_a20"),
                       ("EF @ 1%", "ef_1pct"), ("EF @ 5%", "ef_5pct")]:
        if key in b["model"]:
            print(f"{label:<20} {fmt_ci(b['model'], key):<28} "
                  f"{fmt_ci(b['baseline'], key):<28}")
    dd = b["model"]["pr_auc"]["value"] - b["baseline"]["pr_auc"]["value"]
    print("-" * 92)
    print(f"  model minus baseline, PR-AUC: {dd:+.3f}  -> "
          f"{'model wins' if dd > 0 else 'BASELINE WINS'}")

ds = (BENCH["strict"]["model"]["pr_auc"]["value"]
      - BENCH["strict"]["baseline"]["pr_auc"]["value"])
dr = (BENCH["relaxed"]["model"]["pr_auc"]["value"]
      - BENCH["relaxed"]["baseline"]["pr_auc"]["value"])
print("\n" + "=" * 92)
print("READING THESE NUMBERS")
print("=" * 92)
print(f"""
  On the ChEMBL class balance (82.7% active) the model appeared to beat the
  similarity baseline. At screening prevalence that does not hold: the
  baseline leads by {-ds:+.3f} PR-AUC on strict decoys and {-dr:+.3f} on relaxed ones.

  The strict figure is confounded -- decoys were SELECTED for low similarity
  to actives, which is the baseline's own scoring function. The relaxed set
  removes much of that advantage, so the two together bracket the truth.

  Defensible conclusion: on this target and this library, a random forest on
  ECFP4 + descriptors does NOT clearly outperform nearest-active Tanimoto
  similarity. Phase 4 therefore carries BOTH signals into the composite score
  rather than trusting the model alone.

  Decoys are PRESUMED inactive, so every enrichment here is a mild
  UNDER-estimate -- equally for both methods.
""")

In [ ]:
# --- Figure 14: the two evaluation regimes side by side ---------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))

# (a) why EF collapsed: the prevalence ceiling
ax = axes[0]
regimes = [f"ChEMBL balance\n({y.mean():.1%} active)",
           f"Decoy benchmark\n({prevalence:.1%} active)"]
ceilings = [1 / y.mean(), 1 / prevalence]
observed = [FINAL_METRICS["ef_1pct"]["value"], BENCH_METRICS["ef_1pct"]["value"]]
xp = np.arange(2)
ax.bar(xp - 0.19, ceilings, width=0.36, color=SEQ_BLUE[2],
       edgecolor=SURFACE, linewidth=2, label="theoretical EF@1% ceiling")
ax.bar(xp + 0.19, observed, width=0.36, color=PALETTE["library"],
       edgecolor=SURFACE, linewidth=2, label="observed EF@1%")
for x, c, o in zip(xp, ceilings, observed):
    ax.text(x - 0.19, c, f"{c:.1f}", ha="center", va="bottom", fontsize=9,
            fontweight="600", color=INK["primary"])
    ax.text(x + 0.19, o, f"{o:.1f}", ha="center", va="bottom", fontsize=9,
            fontweight="600", color=INK["primary"])
ax.set_xticks(xp); ax.set_xticklabels(regimes, fontsize=8.5)
ax.set_ylabel("Enrichment factor @ 1%")
ax.set_yscale("log")
ax.set_title("Why EF was uninformative\n(observed pinned to ceiling, left)", loc="left")
ax.legend(loc="upper left"); ax.grid(alpha=0.7)

# (b) enrichment curve on the realistic benchmark
ax = axes[1]
o_m = np.argsort(-s_bench); o_b = np.argsort(-bench_base)
frac = np.arange(1, len(y_bench) + 1) / len(y_bench)
ax.plot(frac, np.cumsum(y_bench[o_m]) / y_bench.sum(), color=PALETTE["library"],
        lw=2.2, label=f"model (EF@1% {BENCH_METRICS['ef_1pct']['value']:.1f})")
ax.plot(frac, np.cumsum(y_bench[o_b]) / y_bench.sum(), color=PALETTE["active"],
        lw=2, label=f"NN baseline (EF@1% {BENCH_BASELINE['ef_1pct']['value']:.1f})")
ax.plot([0, 1], [0, 1], color=INK["muted"], lw=1.2, ls="--", label="random")
ax.set_xscale("log"); ax.set_xlim(1 / len(y_bench), 1)
ax.set_xlabel("Fraction screened (log scale)")
ax.set_ylabel("Fraction of actives recovered")
ax.set_title("Early recognition on the realistic benchmark", loc="left")
ax.legend(loc="upper left"); ax.grid(alpha=0.7)

# (c) score separation
ax = axes[2]
bins = np.linspace(0, 1, 45)
ax.hist(decoy_scores, bins=bins, density=True, histtype="step", lw=2,
        color=PALETTE["inactive"], label=f"decoys (n={len(decoy_scores):,})")
ax.hist(active_scores, bins=bins, density=True, histtype="step", lw=2,
        color=PALETTE["active"], label=f"known actives (n={len(active_scores):,})")
ax.axvline(BEST_THRESHOLD, color=INK["secondary"], ls="--", lw=1.4)
ax.annotate(f"threshold\n{BEST_THRESHOLD:.2f}", (BEST_THRESHOLD, ax.get_ylim()[1] * 0.75),
            fontsize=8, ha="left", color=INK["secondary"], fontweight="600")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Density")
ax.set_title("Score separation, actives vs matched decoys", loc="left")
ax.legend(loc="upper center"); ax.grid(alpha=0.7)

fig.suptitle("Realistic evaluation — what the model does at screening prevalence",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig14_decoy_benchmark")
plt.show()

DECOYS[["compound_id", "std_smiles", "mw", "clogp", "tpsa", "hbd", "hba",
        "nn_sim_ecfp4"]].to_csv(CFG.paths["tables"] / "matched_decoys.csv", index=False)

## 3.8 Prospective scoring of the library

The selected configuration is refitted on **all** labelled data (train + validation + test) — once
the protocol is fixed and the honest performance estimate is banked, discarding 30% of the training
signal costs accuracy for no benefit — and applied to all curated library compounds.

Two numbers accompany every prediction:

* **Ensemble uncertainty.** For the random forest, the spread of per-tree votes. A compound at
  probability 0.8 with tree standard deviation 0.05 is a different proposition from one at 0.8 with
  standard deviation 0.35, and the ranked list should not treat them identically.
* **Applicability-domain status**, from Phase 2's nearest-active similarity. Predictions below the
  AD threshold are extrapolation and are flagged. They are **not** deleted — a genuinely novel
  chemotype necessarily sits far from the training set, so blanket deletion would discard the most
  interesting compounds — but Phase 4 weights them accordingly rather than ranking them naively.

In [ ]:
# --- Refit on all labelled data and score the library ------------------------
final_scaler = StandardScaler().fit(DESC_ACT)
rep_name, model_name = BEST["representation"], BEST["model"]
fs = FEATURE_SETS[rep_name]

def assemble(which):
    """Build the full feature matrix for 'act' (labelled) or 'lib' (library)."""
    if fs.get("desc_only"):
        src = DESC_ACT if which == "act" else DESC_LIB
        return final_scaler.transform(src).astype(np.float32)
    if fs.get("needs_desc"):
        fp = (fs["act"] if which == "act"
              else fs["lib_fp"].toarray().astype(np.float32))
        desc = final_scaler.transform(DESC_ACT if which == "act" else DESC_LIB)
        return np.hstack([fp, desc.astype(np.float32)])
    return fs["act"] if which == "act" else fs["lib"]

X_all = assemble("act")
final_model = make_models(fs["sparse"])[model_name]
t0 = time.time()
final_model.fit(X_all, y)
print(f"Refitted {rep_name}/{model_name} on all {len(y):,} labelled compounds "
      f"({time.time() - t0:.1f} s)")

# Score the library in chunks to bound peak memory on a free Colab instance.
CHUNK = 8000
scores, unc = [], []
for start in tqdm(range(0, len(LIBRARY), CHUNK), desc="Scoring library", unit="chunk"):
    sl = slice(start, min(start + CHUNK, len(LIBRARY)))
    if fs.get("desc_only"):
        Xc = final_scaler.transform(DESC_LIB[sl]).astype(np.float32)
    elif fs.get("needs_desc"):
        Xc = np.hstack([fs["lib_fp"][sl].toarray().astype(np.float32),
                        final_scaler.transform(DESC_LIB[sl]).astype(np.float32)])
    else:
        Xc = fs["lib"][sl]
    scores.append(final_model.predict_proba(Xc)[:, 1])
    if hasattr(final_model, "estimators_"):
        per_tree = np.stack([t.predict_proba(
            Xc.toarray() if sparse.issparse(Xc) else Xc)[:, 1]
            for t in final_model.estimators_])
        unc.append(per_tree.std(axis=0))
    else:
        unc.append(np.full(Xc.shape[0], np.nan))

LIBRARY["ml_score"] = np.concatenate(scores)
LIBRARY["ml_uncertainty"] = np.concatenate(unc)
LIBRARY["ml_predicted_active"] = LIBRARY["ml_score"] >= BEST_THRESHOLD

print(f"\nProspective scores for {len(LIBRARY):,} library compounds")
print(LIBRARY["ml_score"].describe().round(4).to_string())
print(f"\nPredicted active at threshold {BEST_THRESHOLD:.3f}: "
      f"{int(LIBRARY['ml_predicted_active'].sum()):,} "
      f"({100 * LIBRARY['ml_predicted_active'].mean():.2f}%)")
if LIBRARY["ml_uncertainty"].notna().any():
    print(f"Ensemble uncertainty: mean {LIBRARY['ml_uncertainty'].mean():.4f}, "
          f"max {LIBRARY['ml_uncertainty'].max():.4f}")

# A composite Phase-3 confidence flag for Phase 4 to consume.
LIBRARY["prediction_confidence"] = np.select(
    [(~LIBRARY["in_applicability_domain"]),
     (LIBRARY["ml_uncertainty"] > LIBRARY["ml_uncertainty"].quantile(0.75))],
    ["out_of_domain", "high_variance"], default="supported")
print("\nPrediction confidence:")
print(LIBRARY["prediction_confidence"].value_counts().to_string())

In [ ]:
# --- Figure 13: prospective score landscape ---------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))

# (a) score distribution, in and out of the applicability domain
ax = axes[0]
bins = np.linspace(0, 1, 50)
inn = LIBRARY[LIBRARY["in_applicability_domain"]]["ml_score"]
out = LIBRARY[~LIBRARY["in_applicability_domain"]]["ml_score"]
ax.hist(inn, bins=bins, histtype="step", lw=2, color=PALETTE["library"],
        label=f"in applicability domain (n={len(inn):,})")
if len(out):
    ax.hist(out, bins=bins, histtype="step", lw=2, color=PALETTE["active"],
            label=f"out of domain (n={len(out):,})")
ax.axvline(BEST_THRESHOLD, color=INK["secondary"], ls="--", lw=1.4)
ax.annotate(f"threshold {BEST_THRESHOLD:.2f}", (BEST_THRESHOLD, ax.get_ylim()[1] * 0.88),
            fontsize=8, ha="left", color=INK["secondary"], fontweight="600")
ax.set_yscale("log")
ax.set_xlabel("Predicted probability of A2A activity")
ax.set_ylabel("Library compounds (log scale)")
ax.set_title("Prospective score distribution", loc="left")
ax.legend(loc="upper right"); ax.grid(alpha=0.7)

# (b) score vs nearest-active similarity — the novelty/confidence trade-off
ax = axes[1]
samp = LIBRARY.sample(min(15000, len(LIBRARY)), random_state=CFG.random_seed)
sc_ = ax.scatter(samp["nn_sim_ecfp4"], samp["ml_score"], s=5, alpha=0.25,
                 c=samp["ml_uncertainty"], cmap=BLUES, edgecolors="none",
                 rasterized=True)
ax.axvline(AD_THRESHOLD, color=INK["secondary"], ls="--", lw=1.2)
ax.axvspan(*NOVELTY_BAND, color="#ebeae6", zorder=0)
ax.axhline(BEST_THRESHOLD, color=INK["secondary"], ls=":", lw=1.2)
ax.annotate("novelty band", (np.mean(NOVELTY_BAND), 0.02), fontsize=8,
            ha="center", color=INK["secondary"], fontweight="600")
ax.set_xlabel("ECFP4 similarity to nearest known active")
ax.set_ylabel("Predicted probability")
ax.set_title("Score vs novelty\n(colour = ensemble uncertainty)", loc="left")
cb = fig.colorbar(sc_, ax=ax, fraction=0.046, pad=0.03)
cb.set_label("tree-vote SD", fontsize=8); cb.outline.set_visible(False)
ax.grid(alpha=0.7)

# (c) how the top-ranked slice breaks down by confidence
ax = axes[2]
top = LIBRARY.nlargest(2000, "ml_score")
counts = top["prediction_confidence"].value_counts()
names = [n for n in ("supported", "high_variance", "out_of_domain") if n in counts]
vals = [int(counts[n]) for n in names]
cols = {"supported": PALETTE["library"], "high_variance": SEQ_BLUE[2],
        "out_of_domain": PALETTE["active"]}
bars = ax.bar(range(len(names)), vals, color=[cols[n] for n in names],
              width=0.58, edgecolor=SURFACE, linewidth=2)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,}\n({100 * v / len(top):.0f}%)",
            ha="center", va="bottom", fontsize=9, fontweight="600",
            color=INK["primary"])
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace("_", "\n") for n in names], fontsize=8.5)
ax.set_ylabel("Compounds")
ax.set_ylim(0, max(vals) * 1.22)
ax.set_title("Top 2,000 by score — how much is trustworthy?", loc="left")
ax.grid(alpha=0.7)

fig.suptitle("Prospective library scoring — score, novelty, and confidence together",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig13_prospective_scoring")
plt.show()

## 3.9 Persisted artefacts and the Phase 3 validation gate

In [ ]:
# --- Persist Phase 3 ---------------------------------------------------------
import pickle

PANEL.to_csv(CFG.paths["tables"] / "model_panel.csv", index=False)
gap.to_csv(CFG.paths["tables"] / "generalisation_gap.csv")

metrics_out = {
    "selected_representation": rep_name,
    "selected_model": model_name,
    "decision_threshold": float(BEST_THRESHOLD),
    "n_labelled": int(len(ACTIVES)),
    "n_train": int(len(tr)), "n_val": int(len(va)), "n_test": int(len(te)),
    "positive_rate": float(y.mean()),
    "final_metrics": {k: {kk: (None if not np.isfinite(vv) else float(vv))
                          for kk, vv in v.items()}
                      for k, v in FINAL_METRICS.items()},
    "nn_baseline_scaffold_ecfp4": {
        k: {kk: (None if not np.isfinite(vv) else float(vv))
            for kk, vv in v.items()}
        for k, v in BASELINES[("scaffold", "ecfp4")].items()},
    "mean_roc_inflation_random_split": float(gap["roc_gap"].mean()),
    "mean_pr_inflation_random_split": float(gap["pr_gap"].mean()),
    "library_predicted_active": int(LIBRARY["ml_predicted_active"].sum()),
    "confidence_breakdown": LIBRARY["prediction_confidence"]
                            .value_counts().to_dict(),
    "n_bootstrap": N_BOOTSTRAP,
    "chembl_positive_rate": float(y.mean()),
    "ef_ceiling_chembl_balance": float(1 / y.mean()),
    "decoy_benchmark": {
        "n_actives": int(y_bench.sum()),
        "n_decoys": int((1 - y_bench).sum()),
        "prevalence": float(prevalence),
        "ef_ceiling": float(1 / prevalence),
        "decoy_ratio_target": DECOY_RATIO,
        "decoy_max_similarity": DECOY_MAX_SIM,
        "both_sets": {n: {"prevalence": BENCH[n]["prevalence"],
                          "n_decoys": int((1 - BENCH[n]["y"]).sum()),
                          "model_pr_auc": BENCH[n]["model"]["pr_auc"]["value"],
                          "baseline_pr_auc": BENCH[n]["baseline"]["pr_auc"]["value"],
                          "model_roc_auc": BENCH[n]["model"]["roc_auc"]["value"],
                          "baseline_roc_auc": BENCH[n]["baseline"]["roc_auc"]["value"],
                          "model_ef1": BENCH[n]["model"]["ef_1pct"]["value"],
                          "baseline_ef1": BENCH[n]["baseline"]["ef_1pct"]["value"]}
                      for n in BENCH},
        "metrics": {k: {kk: (None if not np.isfinite(vv) else float(vv))
                        for kk, vv in v.items()} for k, v in BENCH_METRICS.items()},
        "nn_baseline": {k: {kk: (None if not np.isfinite(vv) else float(vv))
                            for kk, vv in v.items()}
                        for k, v in BENCH_BASELINE.items()},
    },
}
(CFG.paths["metrics"] / "phase3_metrics.json").write_text(
    json.dumps(metrics_out, indent=2))

with open(CFG.paths["models"] / "final_model.pkl", "wb") as fh:
    pickle.dump({"model": final_model, "scaler": final_scaler,
                 "representation": rep_name, "algorithm": model_name,
                 "threshold": BEST_THRESHOLD,
                 "descriptor_columns": DESCRIPTOR_COLUMNS,
                 "sklearn_version": __import__("sklearn").__version__}, fh)

LIBRARY.to_csv(CFG.paths["curated"] / "library_phase3_scored.csv", index=False)
(LIBRARY.nlargest(5000, "ml_score")
 [["compound_id", "std_smiles", "ml_score", "ml_uncertainty",
   "prediction_confidence", "nn_sim_ecfp4", "nn_sim_fcfp4", "cluster",
   "mw", "clogp", "tpsa", "qed", "murcko_scaffold"]]
 .to_csv(CFG.paths["tables"] / "top5000_ml_ranked.csv", index=False))

RUN_STAMP["phase3_metrics"] = metrics_out
RUN_STAMP["phase"] = 3
(CFG.paths["metrics"] / "run_manifest.json").write_text(json.dumps(RUN_STAMP, indent=2))
print("Phase 3 artefacts written.")

In [ ]:
# --- Phase 3 validation gate -------------------------------------------------
def validate_phase3():
    checks, failures = [], []

    def check(name, cond, detail=""):
        ok = bool(cond); checks.append((name, ok, detail))
        if not ok: failures.append(f"{name} — {detail}")

    # The split invariant this entire phase depends on
    sc_tr = set(ACTIVES["murcko_scaffold"].iloc[SPLITS["scaffold"][0]])
    sc_va = set(ACTIVES["murcko_scaffold"].iloc[SPLITS["scaffold"][1]])
    sc_te = set(ACTIVES["murcko_scaffold"].iloc[SPLITS["scaffold"][2]])
    check("Scaffold split: train/test disjoint", not (sc_tr & sc_te),
          f"{len(sc_tr & sc_te)} shared scaffolds")
    check("Scaffold split: train/val disjoint", not (sc_tr & sc_va),
          f"{len(sc_tr & sc_va)} shared scaffolds")
    all_idx = np.concatenate(SPLITS["scaffold"])
    check("Scaffold split covers every compound exactly once",
          len(all_idx) == len(ACTIVES) and len(set(all_idx)) == len(ACTIVES),
          f"{len(all_idx)} indices for {len(ACTIVES)} compounds")
    check("Both classes present in every fold",
          all(len(np.unique(y[s])) == 2 for s in SPLITS["scaffold"]),
          "a fold is single-class")

    # Metric sanity
    check("Test ROC-AUC in [0, 1]", 0 <= FINAL_METRICS["roc_auc"]["value"] <= 1)
    check("Test ROC-AUC beats chance", FINAL_METRICS["roc_auc"]["value"] > 0.5,
          f"{FINAL_METRICS['roc_auc']['value']:.3f}")
    check("PR-AUC exceeds the positive-rate baseline",
          FINAL_METRICS["pr_auc"]["value"] > FINAL_METRICS["pr_auc_baseline"]["value"],
          f"{FINAL_METRICS['pr_auc']['value']:.3f} vs "
          f"{FINAL_METRICS['pr_auc_baseline']['value']:.3f}")
    check("Bootstrap CI brackets the point estimate",
          (FINAL_METRICS["roc_auc"]["lo"] <= FINAL_METRICS["roc_auc"]["value"]
           <= FINAL_METRICS["roc_auc"]["hi"]), "point estimate outside its CI")
    check("Bootstrap drew enough valid resamples",
          FINAL_METRICS["roc_auc"]["n_draws"] > N_BOOTSTRAP * 0.5,
          f"{FINAL_METRICS['roc_auc']['n_draws']} draws")
    check("EF@1% >= 1 (better than random)",
          FINAL_METRICS["ef_1pct"]["value"] >= 1.0,
          f"{FINAL_METRICS['ef_1pct']['value']:.2f}")

    # Model selection happened on validation, not test
    sc_panel = PANEL[PANEL["split"] == "scaffold"]
    check("Selected model is the validation-PR-AUC maximum",
          np.isclose(BEST["val_pr_auc"], sc_panel["val_pr_auc"].max()),
          "selection did not use validation")

    # Prospective scoring
    check("Every library compound scored", LIBRARY["ml_score"].notna().all(),
          f"{int(LIBRARY['ml_score'].isna().sum())} missing")
    check("Scores are probabilities in [0, 1]",
          LIBRARY["ml_score"].between(0, 1).all(), "out-of-range score")
    check("Confidence flag assigned to every compound",
          LIBRARY["prediction_confidence"].notna().all(), "missing flags")
    check("Scores are not degenerate", LIBRARY["ml_score"].std() > 1e-6,
          "zero variance — model predicts a constant")

    # Decoy benchmark integrity
    check("Decoy benchmark prevalence is realistic", prevalence < 0.10,
          f"{prevalence:.3f} — decoy ratio too low to be screening-like")
    check("Decoys are topologically dissimilar to all actives",
          DECOYS["nn_sim_ecfp4"].max() < DECOY_MAX_SIM,
          f"max similarity {DECOYS['nn_sim_ecfp4'].max():.3f}")
    check("Decoy EF ceiling exceeds the ChEMBL-balance ceiling",
          (1 / prevalence) > (1 / y.mean()) * 3,
          "decoy set does not materially widen the EF range")
    check("Benchmark EF@1% is below its ceiling (not saturated)",
          BENCH_METRICS["ef_1pct"]["value"] <= 1 / prevalence + 1e-6,
          f"EF {BENCH_METRICS['ef_1pct']['value']:.2f} exceeds ceiling "
          f"{1 / prevalence:.2f}")
    check("No decoy is also a labelled ChEMBL compound",
          not (set(DECOYS["inchikey"]) & set(ACTIVES["inchikey"])),
          "decoy set overlaps the labelled data")

    for p in (CFG.paths["metrics"] / "phase3_metrics.json",
              CFG.paths["models"] / "final_model.pkl",
              CFG.paths["tables"] / "top5000_ml_ranked.csv"):
        check(f"Artefact exists: {p.name}", p.exists(), str(p))
    n_fig = len(list(CFG.paths["figures"].glob("*.png")))
    check("Phase 3 figures written", n_fig >= 14, f"{n_fig} PNGs")

    print("Phase 3 validation gate")
    print("=" * 76)
    for name, ok, detail in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if not ok else ""))
    print("=" * 76)
    print(f"{sum(1 for _, ok, _ in checks if ok)}/{len(checks)} checks passed")
    if failures:
        raise AssertionError("Phase 3 validation FAILED:\n  - " + "\n  - ".join(failures))
    print("\nPhase 3 validation PASSED — safe to proceed to Phase 4.")
    return True


validate_phase3()

In [ ]:
# --- Phase 3 summary ---------------------------------------------------------
base_pr = BASELINES[("scaffold", "ecfp4")]["pr_auc"]["value"]
print("=" * 76)
print("PHASE 3 COMPLETE — supervised modelling and prospective scoring")
print("=" * 76)
print(f"""
SELECTED MODEL
  representation ................. {rep_name}
  algorithm ...................... {model_name}
  decision threshold ............. {BEST_THRESHOLD:.3f}   (validation F1 optimum)

HONEST PERFORMANCE  (scaffold-disjoint test set, n = {len(te):,})
  ROC-AUC ........................ {fmt_ci(FINAL_METRICS, 'roc_auc')}
  PR-AUC ......................... {fmt_ci(FINAL_METRICS, 'pr_auc')}
    (positive-rate baseline) ..... {FINAL_METRICS['pr_auc_baseline']['value']:.3f}
  BEDROC (a=20) .................. {fmt_ci(FINAL_METRICS, 'bedroc_a20')}
  EF @ 1% ........................ {fmt_ci(FINAL_METRICS, 'ef_1pct', 2)}
  MCC ............................ {fmt_ci(FINAL_METRICS, 'mcc')}

SCREENING-REALISTIC PERFORMANCE  (property-matched decoys, {prevalence:.2%} active)
  ROC-AUC ........................ {fmt_ci(BENCH_METRICS, 'roc_auc')}
  PR-AUC ......................... {fmt_ci(BENCH_METRICS, 'pr_auc')}
    (positive-rate baseline) ..... {prevalence:.4f}
  BEDROC (a=20) .................. {fmt_ci(BENCH_METRICS, 'bedroc_a20')}
  EF @ 1% ........................ {fmt_ci(BENCH_METRICS, 'ef_1pct', 1)}  (ceiling {1 / prevalence:.0f})
  EF @ 5% ........................ {fmt_ci(BENCH_METRICS, 'ef_5pct', 1)}

VERSUS THE NO-ML BASELINE  (nearest-active Tanimoto similarity)
                                   model     baseline     delta
  ChEMBL balance, PR-AUC ....... {FINAL_METRICS['pr_auc']['value']:>8.3f}  {base_pr:>8.3f}   {FINAL_METRICS['pr_auc']['value'] - base_pr:>+7.3f}
  Strict decoys,  PR-AUC ....... {BENCH['strict']['model']['pr_auc']['value']:>8.3f}  {BENCH['strict']['baseline']['pr_auc']['value']:>8.3f}   {BENCH['strict']['model']['pr_auc']['value'] - BENCH['strict']['baseline']['pr_auc']['value']:>+7.3f}
  Relaxed decoys, PR-AUC ....... {BENCH['relaxed']['model']['pr_auc']['value']:>8.3f}  {BENCH['relaxed']['baseline']['pr_auc']['value']:>8.3f}   {BENCH['relaxed']['model']['pr_auc']['value'] - BENCH['relaxed']['baseline']['pr_auc']['value']:>+7.3f}

  The first row is NOT the operative comparison. At the 82.7% active rate of
  the ChEMBL set the model appears to lead; at screening prevalence it does
  not. The decoy rows are what predicts library performance, and on both of
  them the no-ML similarity baseline WINS. Phase 4 carries both signals.

COST OF A RANDOM SPLIT
  mean ROC-AUC inflation ......... {gap['roc_gap'].mean():+.3f}
  mean PR-AUC inflation .......... {gap['pr_gap'].mean():+.3f}

PROSPECTIVE LIBRARY SCORING
  compounds scored ............... {len(LIBRARY):,}
  predicted active ............... {int(LIBRARY['ml_predicted_active'].sum()):,}
  well-supported predictions ..... {int((LIBRARY['prediction_confidence'] == 'supported').sum()):,}
  out of applicability domain .... {int((LIBRARY['prediction_confidence'] == 'out_of_domain').sum()):,}
""")
print("=" * 76)

---

## Phase 3 complete

**New artefacts**

| Artefact | Contents |
|---|---|
| `outputs/data/curated/library_phase3_scored.csv` | the full deck with ML scores, ensemble uncertainty, and confidence flags |
| `outputs/results/tables/top5000_ml_ranked.csv` | the top 5,000 by predicted probability, with novelty and confidence columns |
| `outputs/results/tables/model_panel.csv` | every model × representation × split combination evaluated |
| `outputs/results/tables/generalisation_gap.csv` | scaffold-vs-random comparison per model |
| `outputs/results/models/final_model.pkl` | the refitted model, scaler, threshold, and feature spec |
| `outputs/results/metrics/phase3_metrics.json` | all metrics with bootstrap intervals |
| `outputs/results/figures/fig10–13*.png` | generalisation gap, panel heatmap, final-model curves, prospective scoring |

**New objects for Phase 4:** `final_model`, `final_scaler`, `BEST_THRESHOLD`, `FINAL_METRICS`,
`PANEL`, `BASELINES`, `vs_metrics()`, `bootstrap_metrics()`, `enrichment_and_bedroc()`.

---

### Phase 4 will implement

1. **Optional structure-based arm** — AutoDock Vina against PDB 4EIY, with receptor preparation,
   a box defined from the co-crystallised ZM-241385, and a **redocking control** that reports RMSD
   to the crystallographic pose. If the RMSD control fails, the docking scores are reported as
   unreliable rather than used. The whole arm is skipped cleanly if Vina or the structure is
   unavailable — docking is never a hard requirement.
2. **Transparent composite ranking** — an explicit, weighted, fully documented score combining ML
   probability, nearest-active similarity, docking score (if present), and ligand efficiency, with
   every weight stated and a sensitivity analysis showing how the top-N changes as weights move.
3. **Diversity-aware selection** — given Phase 2's finding that this library is already
   near-maximally diverse, selection enforces a direct Tanimoto ceiling between chosen hits rather
   than one-per-cluster picking.
4. **Final deliverables** — the prioritised hit table with identifiers, SMILES, descriptors, all
   component scores, uncertainty, and rank; 2D structure grids of the top hits annotated with their
   scores; and a per-hit rationale column stating *why* each compound was selected.

---
# Phase 4 — Structure-based scoring, composite ranking, and hit selection

Phase 4 turns 48,489 scored compounds into a defensible shortlist.

Three findings from earlier phases set the design, and none of them was assumed at the outset:

**From Phase 2 — the library is already diversity-selected.** At the conventional Butina cut-off,
92% of clusters are singletons and the largest holds nine compounds. Textbook "pick one compound
per cluster" diversity selection therefore has almost nothing to do. Selection here enforces a
**direct Tanimoto ceiling between chosen hits** instead.

**From Phase 3 — the ML model does not beat nearest-active similarity at screening prevalence.**
On property-matched decoys the similarity baseline led by 0.14–0.15 PR-AUC. The composite score
below therefore weights similarity **at least as heavily as** the model, rather than treating the
ML probability as the primary signal. This is the evidence dictating the weights, not taste.

**From Phase 3 — 9,562 library compounds sit outside the applicability domain.** Their predictions
are extrapolation. They are down-weighted rather than deleted, because a genuinely novel chemotype
necessarily sits far from the training set.

**Sections**

1. **4.0** Optional structure-based arm — receptor prep, box definition, redocking control
2. **4.1** Docking the top-ranked slice
3. **4.2** The composite score — every weight stated
4. **4.3** Weight sensitivity analysis
5. **4.4** Diversity-aware selection
6. **4.5** The final prioritised hit list
7. **4.6** Structures, rationales, and persisted deliverables
8. **4.7** Phase 4 validation gate and project summary

## 4.0 The structure-based arm (optional)

**PDB 4EIY** is a 1.8 Å thermostabilised A<sub>2A</sub> structure with the high-affinity antagonist
**ZM-241385** (ligand code `ZMA`) bound — one of the best-resolved class-A GPCR structures
available, which is a large part of why A<sub>2A</sub> was chosen as the target.

**This entire section is optional and skips cleanly.** If the structure cannot be downloaded, or
AutoDock Vina is not installed, `DOCKING_AVAILABLE` is set `False`, every downstream cell adapts,
and the composite score renormalises its weights over the remaining components. Docking is never a
hard requirement — a virtual screen that silently produces no output because a PDB fetch failed is
worse than one that says so and carries on.

### The redocking control, and why it gates everything

Before any library compound is docked, **ZM-241385 is removed from the structure and docked back
into its own site.** If Vina cannot reproduce the crystallographic pose to within ~2.0 Å RMSD, then
the receptor preparation, the box, or the scoring function is not working for this site — and every
library docking score that follows is noise dressed up as a number.

Most published docking workflows omit this control. It costs one docking run and it is the only
thing standing between a plausible-looking score column and a meaningless one. If the control
fails, this notebook **reports the docking scores as unreliable and excludes them from the
composite**, rather than quietly ranking on them.

### Receptor preparation

Preparation is deliberately explicit rather than hidden in a one-line helper:

* waters, ions, and cryoprotectants stripped (the structure contains several);
* the fusion partner / thermostabilising construct chains removed, keeping only the receptor chain;
* hydrogens added and Gasteiger charges assigned;
* the search box centred on the ZM-241385 centroid with a 22 Å edge — large enough to admit the
  orthosteric pocket and its vestibule, small enough to keep the search tractable.

In [ ]:
# --- Optional docking dependencies ------------------------------------------
# Installed here rather than in the Phase 1 setup cell because the whole
# structure-based arm is optional: a failure to install must not break Phases
# 1-3, so this is deliberately non-fatal.
import importlib, subprocess, sys
for _mod, _pkg in [("vina", "vina"), ("gemmi", "gemmi"), ("meeko", "meeko")]:
    if importlib.util.find_spec(_mod) is None:
        try:
            _cmd = [sys.executable, "-m", "pip", "install", "-q", _pkg]
            try:
                subprocess.check_call(_cmd)
            except subprocess.CalledProcessError:
                subprocess.check_call(_cmd + ["--break-system-packages"])
            print(f"installed {_pkg}")
        except Exception as _e:                              # noqa: BLE001
            print(f"could not install {_pkg}: {type(_e).__name__} "
                  f"(the docking arm will skip if it is required)")

# --- Phase 4 configuration ---------------------------------------------------
DOCK_CFG = {
    "pdb_id":          CFG.pdb_id,            # 4EIY
    "ligand_code":     "ZMA",                 # ZM-241385 in 4EIY
    "receptor_chain":  "A",
    "box_padding":     5.0,                   # per-side padding around the ligand extent
    "box_min":         20.0,                  # clip the box to a sane range: a 37 A
    "box_max":         28.0,                  # box wastes search effort and accuracy
    "exhaustiveness":  8,
    "n_poses":         5,
    "seed":            CFG.random_seed,
    "n_dock":          120,                   # top-N to dock (~8 s/ligand measured)
    "redock_rmsd_max": 2.0,                   # control must pass this
    # Drop a local 4EIY.pdb next to the notebook if the RCSB download is blocked.
    "local_pdb":       f"{CFG.pdb_id}.pdb",
}

# Heuristics for stripping a crystal structure down to the receptor.
CRYSTAL_JUNK = {
    "HOH", "DOD", "SO4", "PO4", "GOL", "EDO", "PEG", "MPD", "ACT", "CL", "NA",
    "K", "MG", "CA", "ZN", "NI", "IOD", "BR", "TRS", "IMD", "DMS", "FMT",
    "OLC", "OLA", "OLB", "PLM", "CLR", "1PE", "P6G", "PGE", "BOG",
    "LMT", "NAG", "Y01", "HEM",
}

DOCKING_AVAILABLE = False
DOCKING_RELIABLE  = False
DOCK_SKIP_REASON  = None
print(f"Docking target: PDB {DOCK_CFG['pdb_id']}, ligand {DOCK_CFG['ligand_code']} "
      f"({CFG.reference_ligand})")

In [ ]:
# --- Acquire the structure ---------------------------------------------------
def fetch_structure(pdb_id, local_fallback=None, out_dir=None):
    """
    Obtain a PDB file: cached copy, then a user-supplied local file, then RCSB.
    Returns (path, source) or (None, reason).
    """
    out_dir = out_dir or CFG.paths["raw"]
    cached = Path(out_dir) / f"{pdb_id}.pdb"
    if cached.exists() and cached.stat().st_size > 1000:
        print(f"Using cached structure: {cached}")
        return cached, "cache"

    if local_fallback and Path(local_fallback).exists():
        import shutil
        shutil.copy(local_fallback, cached)
        print(f"Using local structure: {local_fallback} -> {cached}")
        return cached, "local"

    for url in (f"https://files.rcsb.org/download/{pdb_id}.pdb",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pdb_id.lower()}.ent"):
        try:
            import urllib.request
            print(f"Downloading {url} ...")
            urllib.request.urlretrieve(url, cached)
            if cached.stat().st_size > 1000:
                print(f"  downloaded -> {cached} ({cached.stat().st_size / 1024:.0f} KB)")
                return cached, "download"
        except Exception as exc:                        # noqa: BLE001
            print(f"  failed: {type(exc).__name__}: {exc}")

    return None, ("Structure unavailable. Download "
                  f"https://files.rcsb.org/download/{pdb_id}.pdb manually and place "
                  f"it beside this notebook as '{local_fallback}', then re-run.")


PDB_PATH, PDB_SOURCE = fetch_structure(DOCK_CFG["pdb_id"], DOCK_CFG["local_pdb"])
if PDB_PATH is None:
    DOCK_SKIP_REASON = PDB_SOURCE
    print(f"\n{'!' * 76}\n!!  STRUCTURE-BASED ARM SKIPPED\n!!  {PDB_SOURCE}\n{'!' * 76}")

In [ ]:
# --- Check the docking toolchain --------------------------------------------
def check_docking_toolchain():
    """Report which docking components are present. Returns (ok, detail)."""
    have = {}
    try:
        from vina import Vina                                   # noqa: F401
        have["vina"] = True
    except Exception as exc:                                    # noqa: BLE001
        have["vina"] = False
        print(f"  AutoDock Vina Python bindings unavailable: {type(exc).__name__}")
        print("    install with:  pip install vina")
    try:
        from meeko import MoleculePreparation, PDBQTWriterLegacy  # noqa: F401
        have["meeko"] = True
    except Exception:                                           # noqa: BLE001
        have["meeko"] = False
        print("  Meeko unavailable — falling back to the built-in PDBQT writer")
        print("    Meeko needs gemmi:  pip install meeko gemmi")
        print("    Without it, ligands dock RIGIDLY (TORSDOF 0), which is a")
        print("    materially weaker search. Install it for production work.")

    for k, v in have.items():
        print(f"  {k:<8} {'available' if v else 'MISSING'}")
    return have["vina"], have


if PDB_PATH is not None:
    VINA_OK, TOOLCHAIN = check_docking_toolchain()
    if not VINA_OK:
        DOCK_SKIP_REASON = ("AutoDock Vina is not installed. "
                            "Run `pip install vina` and re-run this section.")
        print(f"\n{'!' * 76}\n!!  STRUCTURE-BASED ARM SKIPPED\n"
              f"!!  {DOCK_SKIP_REASON}\n{'!' * 76}")
else:
    VINA_OK, TOOLCHAIN = False, {}

In [ ]:
# --- Receptor preparation and box definition --------------------------------
def parse_pdb(path, chain="A", ligand_code=None):
    """
    Split a PDB into receptor atoms and the reference ligand.

    Deliberately a plain text parse rather than a structural-biology dependency:
    it is auditable, has no extra install, and 4EIY needs only chain selection
    and HETATM triage.
    """
    receptor, ligand, het_seen = [], [], {}
    for line in Path(path).read_text().splitlines():
        rec = line[:6].strip()
        if rec not in ("ATOM", "HETATM"):
            continue
        resn = line[17:20].strip()
        ch = line[21].strip()
        alt = line[16].strip()
        if alt not in ("", "A"):           # keep only the first altloc
            continue
        if rec == "HETATM":
            het_seen[resn] = het_seen.get(resn, 0) + 1
            if ligand_code and resn == ligand_code:
                ligand.append(line)
            continue
        if ch and chain and ch != chain:
            continue
        if resn in CRYSTAL_JUNK:
            continue
        receptor.append(line)
    return receptor, ligand, het_seen


def atom_coords(lines):
    return np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                     for l in lines])


BOX = None
if PDB_PATH is not None:
    rec_lines, lig_lines, het = parse_pdb(
        PDB_PATH, DOCK_CFG["receptor_chain"], DOCK_CFG["ligand_code"])
    print(f"Receptor atoms (chain {DOCK_CFG['receptor_chain']}, cleaned): "
          f"{len(rec_lines):,}")
    print(f"Reference ligand {DOCK_CFG['ligand_code']} atoms: {len(lig_lines)}")
    print(f"\nHETATM residues present in the file (top 12):")
    for k, v in sorted(het.items(), key=lambda kv: -kv[1])[:12]:
        tag = ("  <- reference ligand" if k == DOCK_CFG["ligand_code"]
               else ("  (stripped)" if k in CRYSTAL_JUNK else ""))
        print(f"  {k:<5} {v:>5} atoms{tag}")

    if not lig_lines:
        DOCK_SKIP_REASON = (f"Ligand {DOCK_CFG['ligand_code']} not found in "
                            f"{PDB_PATH.name}; cannot define a search box or run "
                            f"the redocking control.")
        print(f"\n!! {DOCK_SKIP_REASON}")
    elif not rec_lines:
        DOCK_SKIP_REASON = f"No receptor atoms parsed from {PDB_PATH.name}."
        print(f"\n!! {DOCK_SKIP_REASON}")
    else:
        lig_xyz = atom_coords(lig_lines)
        centre = lig_xyz.mean(axis=0)
        extent = lig_xyz.max(axis=0) - lig_xyz.min(axis=0)
        size = np.clip(extent + 2 * DOCK_CFG["box_padding"],
                       DOCK_CFG["box_min"], DOCK_CFG["box_max"])
        BOX = {"center": centre.tolist(), "size": size.tolist()}
        print(f"\nSearch box from the {CFG.reference_ligand} centroid")
        print(f"  centre  ({centre[0]:7.2f}, {centre[1]:7.2f}, {centre[2]:7.2f})")
        print(f"  size    ({size[0]:5.1f}, {size[1]:5.1f}, {size[2]:5.1f}) A")
        print(f"  ligand extent ({extent[0]:.1f}, {extent[1]:.1f}, {extent[2]:.1f}) A")
        print(f"  (clipped to [{DOCK_CFG['box_min']:.0f}, {DOCK_CFG['box_max']:.0f}] A per side — "
              f"an oversized box wastes search effort and loses accuracy)")
        REC_CLEAN = CFG.paths["raw"] / f"{DOCK_CFG['pdb_id']}_receptor.pdb"
        REC_CLEAN.write_text("\n".join(rec_lines) + "\nEND\n")
        LIG_REF = CFG.paths["raw"] / f"{DOCK_CFG['pdb_id']}_{DOCK_CFG['ligand_code']}.pdb"
        LIG_REF.write_text("\n".join(lig_lines) + "\nEND\n")
        print(f"\n  receptor -> {REC_CLEAN}")
        print(f"  ligand   -> {LIG_REF}")

### PDBQT conversion

Vina needs PDBQT: coordinates plus atom types, partial charges, and — for ligands — a rotatable-bond
torsion tree. Meeko is the modern tool for this and is used when available. The fallback writer
below covers the receptor (rigid, so no torsion tree is needed) and simple ligands, and is honest
about its limits: it does not handle flexible receptor side chains or exotic atom types, and it
prints that rather than failing silently.

In [ ]:
# --- PDBQT writers -----------------------------------------------------------
GASTEIGER_FALLBACK = {"C": 0.0, "N": -0.3, "O": -0.4, "S": -0.1, "H": 0.1,
                      "F": -0.2, "Cl": -0.1, "Br": -0.1, "I": -0.1, "P": 0.2}
AD_TYPE = {"C": "C", "N": "NA", "O": "OA", "S": "SA", "H": "HD", "F": "F",
           "CL": "Cl", "BR": "Br", "I": "I", "P": "P"}


def receptor_to_pdbqt(pdb_lines, out_path):
    """Minimal rigid-receptor PDBQT. Polar-H typing is approximate."""
    out = []
    for l in pdb_lines:
        name = l[12:16].strip()
        elem = (l[76:78].strip() or name[0]).upper()
        if elem == "H":
            continue                      # implicit non-polar hydrogens
        t = AD_TYPE.get(elem, "C")
        q = GASTEIGER_FALLBACK.get(elem.capitalize(), 0.0)
        out.append(f"{l[:54]}  0.00  0.00    {q:>6.3f} {t:<2}")
    Path(out_path).write_text("\n".join(out) + "\nTER\nEND\n")
    return out_path


def mol_to_pdbqt(mol, out_path, name="LIG"):
    """
    Ligand PDBQT via Meeko when present; otherwise a rigid single-torsion
    fallback (poses will be less accurate — stated, not hidden).
    """
    if TOOLCHAIN.get("meeko"):
        from meeko import MoleculePreparation, PDBQTWriterLegacy
        prep = MoleculePreparation()
        setups = prep.prepare(mol)
        pdbqt, ok, err = PDBQTWriterLegacy.write_string(setups[0])
        if not ok:
            raise RuntimeError(f"Meeko failed: {err}")
        Path(out_path).write_text(pdbqt)
        return out_path

    # Vina's PDBQT parser is strict about PDB column positions. Build the
    # record field by field with the altLoc column present -- omitting it
    # shifts every coordinate by one character and the parser rejects the file.
    conf = mol.GetConformer()
    lines = ["ROOT"]
    serial = 0
    for i, atom in enumerate(mol.GetAtoms()):
        if atom.GetSymbol() == "H" and not any(
                n.GetSymbol() in ("N", "O") for n in atom.GetNeighbors()):
            continue
        serial += 1
        p = conf.GetAtomPosition(i)
        sym = atom.GetSymbol()
        t = AD_TYPE.get(sym.upper(), "C")
        q = GASTEIGER_FALLBACK.get(sym, 0.0)
        aname = f"{sym}{serial}"[:4]
        lines.append(
            f"ATOM  "            # 1-6   record
            f"{serial:>5}"       # 7-11  serial
            f" "                 # 12    space
            f"{aname:<4}"        # 13-16 atom name
            f" "                 # 17    altLoc
            f"{name:>3}"         # 18-20 resName
            f" "                 # 21    space
            f"A"                 # 22    chain
            f"{1:>4}"            # 23-26 resSeq
            f" "                 # 27    iCode
            f"   "               # 28-30 spaces
            f"{p.x:8.3f}{p.y:8.3f}{p.z:8.3f}"
            f"  1.00  0.00    {q:>6.3f} {t:<2}")
    lines += ["ENDROOT", "TORSDOF 0"]
    Path(out_path).write_text("\n".join(lines) + "\n")
    return out_path


def crystal_ligand_to_pdbqt(pdb_lines, out_path):
    """
    Write the crystallographic ligand using ITS OWN coordinates.

    This exists because bond orders in a PDB HETATM block are inferred from
    atom proximity and are frequently wrong -- for ZM-241385 in 4EIY, RDKit
    perceives the triazolotriazine as fully saturated and the furan as a
    tetrahydrofuran. Re-embedding from that mis-perceived SMILES would produce
    a different molecule, and the redocking RMSD would be meaningless.

    Taking the crystal coordinates directly sidesteps the problem entirely:
    element assignments are read correctly from the PDB, and the RMSD is
    computed against the same atoms in the same order.
    """
    out, kept = ["ROOT"], []
    for l in pdb_lines:
        elem = (l[76:78].strip() or l[12:16].strip()[0]).upper()
        if elem == "H":
            continue
        q = GASTEIGER_FALLBACK.get(elem.capitalize(), 0.0)
        t = AD_TYPE.get(elem, "C")
        # Columns 7-54 are reused verbatim, preserving PDB alignment exactly.
        out.append(f"ATOM  {l[6:54]}  0.00  0.00    {q:>6.3f} {t:<2}")
        kept.append(l)
    out += ["ENDROOT", "TORSDOF 0"]
    Path(out_path).write_text("\n".join(out) + "\n")
    return out_path, kept


def embed_3d(smiles, seed=CFG.random_seed):
    """Generate a single MMFF-minimised 3D conformer."""
    from rdkit.Chem import AllChem
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
    except Exception:                                   # noqa: BLE001
        pass
    return mol


print("PDBQT writers ready.",
      "Meeko" if TOOLCHAIN.get("meeko") else "built-in fallback (lower pose accuracy)")

In [ ]:
# --- The redocking control ---------------------------------------------------
def heavy_atom_rmsd(mol_a, mol_b):
    """Symmetry-corrected heavy-atom RMSD between two poses of one molecule."""
    from rdkit.Chem import rdMolAlign
    a = Chem.RemoveHs(mol_a); b = Chem.RemoveHs(mol_b)
    try:
        return rdMolAlign.CalcRMS(a, b)
    except Exception:                                   # noqa: BLE001
        ca = a.GetConformer().GetPositions()
        cb = b.GetConformer().GetPositions()
        n = min(len(ca), len(cb))
        return float(np.sqrt(((ca[:n] - cb[:n]) ** 2).sum(axis=1).mean()))


REDOCK_RMSD = None
if VINA_OK and BOX is not None and DOCK_SKIP_REASON is None:
    try:
        from vina import Vina
        REC_PDBQT = CFG.paths["raw"] / f"{DOCK_CFG['pdb_id']}_receptor.pdbqt"
        receptor_to_pdbqt(rec_lines, REC_PDBQT)
        print(f"Receptor PDBQT -> {REC_PDBQT}")

        # The crystallographic ligand is docked using its OWN coordinates,
        # rigidly. See crystal_ligand_to_pdbqt() for why re-embedding from the
        # PDB-perceived SMILES would invalidate the control.
        LIG_PDBQT = CFG.paths["raw"] / "redock_probe.pdbqt"
        _, kept_lig = crystal_ligand_to_pdbqt(lig_lines, LIG_PDBQT)
        ref_xyz = atom_coords(kept_lig)
        print(f"Redock probe: {len(kept_lig)} heavy atoms from the crystal pose")

        v = Vina(sf_name="vina", seed=DOCK_CFG["seed"], verbosity=0)
        v.set_receptor(str(REC_PDBQT))
        v.set_ligand_from_file(str(LIG_PDBQT))
        v.compute_vina_maps(center=BOX["center"], box_size=BOX["size"])
        v.dock(exhaustiveness=DOCK_CFG["exhaustiveness"], n_poses=DOCK_CFG["n_poses"])
        best_e = float(v.energies()[0][0])

        pose_path = CFG.paths["raw"] / "redock_pose.pdbqt"
        v.write_poses(str(pose_path), n_poses=1, overwrite=True)
        pose_xyz = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                             for l in pose_path.read_text().splitlines()
                             if l.startswith(("ATOM", "HETATM"))])
        n = min(len(pose_xyz), len(ref_xyz))
        REDOCK_RMSD = float(np.sqrt(
            ((pose_xyz[:n] - ref_xyz[:n]) ** 2).sum(axis=1).mean()))

        print(f"\nREDOCKING CONTROL  (rigid-body)")
        print(f"  best Vina score : {best_e:.2f} kcal/mol")
        print(f"  pose RMSD       : {REDOCK_RMSD:.2f} A "
              f"(threshold {DOCK_CFG['redock_rmsd_max']:.1f} A)")
        print(f"  atoms compared  : {n} of {len(ref_xyz)} crystal heavy atoms")
        DOCKING_RELIABLE = (np.isfinite(REDOCK_RMSD)
                            and REDOCK_RMSD <= DOCK_CFG["redock_rmsd_max"])
        DOCKING_AVAILABLE = True
        if DOCKING_RELIABLE:
            print("  VERDICT: PASS — the receptor preparation, box placement and")
            print("  scoring function reproduce the crystallographic pose.")
            print("  Docking scores are admissible for the composite score.")
            print("\n  Scope of this control: it is a RIGID-BODY redock, so it")
            print("  validates the receptor, the box and the scoring function, but")
            print("  not conformer search. Library ligands are docked flexibly")
            print("  (Meeko) or rigidly (fallback) — see the toolchain report above.")
        else:
            print("  VERDICT: FAIL — the setup does NOT reproduce the known pose.")
            print("  Docking scores will be COMPUTED BUT EXCLUDED from the "
                  "composite score.")
            print("  Likely causes: receptor protonation, box placement, or the "
                  "fallback PDBQT writer.\n  Install Meeko and re-run before "
                  "trusting any docking number.")
    except Exception as exc:                            # noqa: BLE001
        DOCK_SKIP_REASON = f"Redocking control failed: {type(exc).__name__}: {exc}"
        print(f"\n!! {DOCK_SKIP_REASON}")
        DOCKING_AVAILABLE = False

if not DOCKING_AVAILABLE:
    print(f"\n{'=' * 76}")
    print("STRUCTURE-BASED ARM: NOT ACTIVE")
    print(f"  reason: {DOCK_SKIP_REASON}")
    print("  The composite score in 4.2 will renormalise over the remaining")
    print("  components. This is a supported configuration, not a failure —")
    print("  the ligand-based screen is complete on its own.")
    print(f"{'=' * 76}")

## 4.1 Docking the top-ranked slice

Docking all 48,489 compounds is neither necessary nor affordable — at roughly 10–30 s per ligand
that is days of compute. The standard and defensible approach is a **funnel**: the cheap
ligand-based score ranks everything, and the expensive structure-based score is spent only on the
slice that survives.

`DOCK_CFG["n_dock"]` (default 200) controls the slice. The pre-dock ranking used to choose it is
the ligand-based composite from §4.2 computed **without** the docking term, so the funnel is
internally consistent.

In [ ]:
# --- Ligand-based pre-rank (the funnel's first stage) -----------------------
def minmax(s, lo=None, hi=None):
    """Scale to [0, 1], robust to constant input."""
    s = pd.to_numeric(s, errors="coerce")
    lo = s.min() if lo is None else lo
    hi = s.max() if hi is None else hi
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    return ((s - lo) / (hi - lo)).clip(0, 1)


def novelty_score(sim, band=NOVELTY_BAND, analogue=0.70):
    """
    Reward intermediate similarity to known actives.

    Below the band a compound is likely a model false positive; above the
    analogue threshold it is a close relative of a published ligand with little
    novelty or IP value. The peak sits inside the band.
    """
    sim = pd.to_numeric(sim, errors="coerce").fillna(0)
    lo, hi = band
    out = np.where(
        sim < lo, (sim / max(lo, 1e-9)) * 0.7,
        np.where(sim <= hi, 1.0,
                 np.clip(1.0 - (sim - hi) / max(analogue - hi, 1e-9) * 0.6, 0.3, 1.0)))
    return pd.Series(out, index=sim.index)


LIBRARY["s_ml"]       = LIBRARY["ml_score"]
LIBRARY["s_sim"]      = minmax(LIBRARY["nn_sim_ecfp4"])
LIBRARY["s_novelty"]  = novelty_score(LIBRARY["nn_sim_ecfp4"])
LIBRARY["s_quality"]  = minmax(LIBRARY["qed"])
LIBRARY["s_confidence"] = np.select(
    [LIBRARY["prediction_confidence"] == "supported",
     LIBRARY["prediction_confidence"] == "high_variance"],
    [1.0, 0.75], default=0.5)

PRE_WEIGHTS = {"s_ml": 0.35, "s_sim": 0.35, "s_novelty": 0.15, "s_quality": 0.15}
LIBRARY["prescore"] = sum(LIBRARY[k] * w for k, w in PRE_WEIGHTS.items())
LIBRARY["prescore"] *= LIBRARY["s_confidence"]

print("Ligand-based pre-rank components")
for k, w in PRE_WEIGHTS.items():
    print(f"  {k:<12} weight {w:.2f}   mean {LIBRARY[k].mean():.3f}")
print(f"  x confidence multiplier (supported 1.00 / high-variance 0.75 / "
      f"out-of-domain 0.50)")

DOCK_SLICE = LIBRARY.nlargest(DOCK_CFG["n_dock"], "prescore")
print(f"\nDocking slice: top {len(DOCK_SLICE):,} of {len(LIBRARY):,} by pre-rank")
print(f"  prescore range {DOCK_SLICE['prescore'].min():.3f} - "
      f"{DOCK_SLICE['prescore'].max():.3f}")
print(DOCK_SLICE["prediction_confidence"].value_counts().to_string())

In [ ]:
# --- Dock the slice ----------------------------------------------------------
LIBRARY["vina_score"] = np.nan
LIBRARY["vina_le"] = np.nan

# Docking is by far the most expensive step in the project, and its result is
# deterministic given a fixed seed. Cache it so re-runs of later sections cost
# nothing. Delete this file to force a re-dock.
DOCK_CACHE = CFG.paths["raw"] / "docking_scores.csv"
_cached = {}
if DOCK_CACHE.exists():
    _c = pd.read_csv(DOCK_CACHE)
    _cached = dict(zip(_c["compound_id"].astype(str), _c["vina_score"]))
    print(f"Docking cache: {len(_cached):,} scores loaded from {DOCK_CACHE}")
    print("  (delete that file to force a re-dock)")

if DOCKING_AVAILABLE:
    from vina import Vina
    v = Vina(sf_name="vina", seed=DOCK_CFG["seed"], verbosity=0)
    v.set_receptor(str(REC_PDBQT))
    v.compute_vina_maps(center=BOX["center"], box_size=BOX["size"])

    # Per-ligand PDBQT scratch. Kept out of the output tree: these are
    # transient, numerous, and of no interest once a score exists.
    import tempfile
    tmp = Path(tempfile.mkdtemp(prefix="dock_"))
    results, failures = {}, 0
    t0 = time.time()

    for r in tqdm(DOCK_SLICE.itertuples(), total=len(DOCK_SLICE),
                  desc="Docking", unit="lig"):
        if str(r.compound_id) in _cached:
            results[r.compound_id] = float(_cached[str(r.compound_id)])
            continue
        try:
            mol = embed_3d(r.std_smiles)
            if mol is None:
                failures += 1
                continue
            lp = tmp / f"{r.compound_id}.pdbqt"
            mol_to_pdbqt(mol, lp, name="LIG")
            v.set_ligand_from_file(str(lp))
            v.dock(exhaustiveness=DOCK_CFG["exhaustiveness"],
                   n_poses=DOCK_CFG["n_poses"])
            results[r.compound_id] = float(v.energies()[0][0])
            lp.unlink(missing_ok=True)
        except Exception:                               # noqa: BLE001
            failures += 1
    import shutil as _sh
    _sh.rmtree(tmp, ignore_errors=True)

    elapsed = time.time() - t0
    n_new = len(results) - sum(1 for k in results if str(k) in _cached)
    print(f"\n{len(results):,} ligands scored ({n_new:,} newly docked, "
          f"{len(results) - n_new:,} from cache) in {elapsed / 60:.1f} min; "
          f"{failures} failed")
    if n_new:
        print(f"  {elapsed / max(n_new, 1):.1f} s per newly docked ligand")
    pd.DataFrame({"compound_id": list(results), "vina_score": list(results.values())}) \
        .to_csv(DOCK_CACHE, index=False)
    print(f"  cache written -> {DOCK_CACHE}")

    LIBRARY["vina_score"] = LIBRARY["compound_id"].map(results)
    ok = LIBRARY["vina_score"].notna()
    LIBRARY.loc[ok, "vina_le"] = (-LIBRARY.loc[ok, "vina_score"]
                                  / LIBRARY.loc[ok, "heavy_atoms"])
    print(f"\nVina score (kcal/mol): "
          f"{LIBRARY['vina_score'].describe().round(2).to_dict()}")
    print(f"Ligand efficiency: mean {LIBRARY['vina_le'].mean():.3f} kcal/mol/HA")
    if not DOCKING_RELIABLE:
        print("\n!! Redocking control FAILED — these scores are reported for "
              "inspection\n!! but are EXCLUDED from the composite score.")
else:
    print("Docking not active — vina_score / vina_le remain NaN and are "
          "excluded from the composite.")

## 4.2 The composite score

Every weight is stated, every component is scaled to [0, 1] with higher meaning better, and the
rationale for each weight is given. There is no hidden normalisation and no tuned magic number.

| Component | Weight | What it contributes | Why this weight |
|---|---|---|---|
| `s_sim` — nearest-active similarity | **0.30** | ligand-based evidence of A<sub>2A</sub> pharmacology | Phase 3 showed this **outperforms** the ML model at screening prevalence, so it is weighted at least as heavily |
| `s_ml` — ML probability | **0.25** | learned SAR beyond simple similarity | real but unproven signal; deliberately *not* the largest weight, because the decoy benchmark did not support that |
| `s_dock` — docking score | **0.20** | structure-based complementarity to the orthosteric site | orthogonal evidence, but only if the redocking control passed; otherwise **0** and redistributed |
| `s_novelty` — novelty band | **0.15** | preference for intermediate similarity | the point of screening REAL space is new chemotypes, not re-finding known ligands |
| `s_quality` — QED | **0.10** | general drug-likeness | a tiebreaker, not a driver |

The composite is then **multiplied by a confidence factor** (1.00 supported / 0.75 high-variance /
0.50 out-of-domain) rather than having confidence added as a weighted term. Multiplication is the
right operator here: low confidence should *discount* an otherwise attractive compound
proportionally, not offset it with a fixed bonus.

**Weights renormalise automatically** when a component is unavailable, so switching docking off
does not silently shrink every score toward zero.

In [ ]:
# --- Composite scoring -------------------------------------------------------
BASE_WEIGHTS = {
    "s_sim":     0.30,
    "s_ml":      0.25,
    "s_dock":    0.20,
    "s_novelty": 0.15,
    "s_quality": 0.10,
}

# Docking contributes only if it ran AND passed the redocking control.
USE_DOCKING = bool(DOCKING_AVAILABLE and DOCKING_RELIABLE
                   and LIBRARY["vina_score"].notna().any())

if USE_DOCKING:
    # More negative Vina energy is better, so invert before scaling. Clip at the
    # 1st percentile so a single outlier pose cannot compress the whole scale.
    lo = LIBRARY["vina_score"].quantile(0.01)
    hi = LIBRARY["vina_score"].quantile(0.99)
    LIBRARY["s_dock"] = 1.0 - minmax(LIBRARY["vina_score"].clip(lo, hi), lo, hi)
    LIBRARY["s_dock"] = LIBRARY["s_dock"].fillna(0.0)
else:
    LIBRARY["s_dock"] = np.nan


def composite_score(df, weights=BASE_WEIGHTS, use_docking=USE_DOCKING,
                    apply_confidence=True):
    """Weighted sum over available components, renormalised, times confidence."""
    w = {k: v for k, v in weights.items()
         if not (k == "s_dock" and not use_docking)}
    total = sum(w.values())
    w = {k: v / total for k, v in w.items()}
    score = sum(df[k].fillna(0) * wt for k, wt in w.items())
    if apply_confidence:
        score = score * df["s_confidence"]
    return score, w


LIBRARY["composite"], ACTIVE_WEIGHTS = composite_score(LIBRARY)
LIBRARY["rank"] = LIBRARY["composite"].rank(ascending=False, method="first").astype(int)

print("Composite score — active weights (renormalised)")
print("-" * 56)
for k, wt in ACTIVE_WEIGHTS.items():
    print(f"  {k:<12} {wt:.4f}   (base {BASE_WEIGHTS[k]:.2f})")
print("-" * 56)
print(f"  docking included: {USE_DOCKING}"
      + ("" if USE_DOCKING else f"   [{DOCK_SKIP_REASON or 'redock control failed'}]"))
print(f"  x confidence multiplier applied")
print(f"\nComposite distribution:")
print(LIBRARY["composite"].describe().round(4).to_string())

## 4.3 Weight sensitivity analysis

A weighted score is only as trustworthy as its stability. If moving a weight by ±50% reshuffles the
top 200, then the specific weights are doing the work and the "prioritisation" is an artefact of
choices nobody can justify to three decimal places.

The analysis below perturbs each weight in turn and measures how much of the top-200 set survives,
plus the Spearman rank correlation across the full library. **High overlap means the ranking is
driven by the data rather than by the weights** — which is the claim a composite score needs to
earn before anyone orders compounds against it.

In [ ]:
# --- Sensitivity to the weights ---------------------------------------------
from scipy.stats import spearmanr

TOP_N = 200
base_top = set(LIBRARY.nlargest(TOP_N, "composite")["compound_id"])
base_scores = LIBRARY["composite"].values

scen = []
for comp in ACTIVE_WEIGHTS:
    for mult, tag in [(0.5, "-50%"), (1.5, "+50%")]:
        w = dict(BASE_WEIGHTS)
        w[comp] = BASE_WEIGHTS[comp] * mult
        s, _ = composite_score(LIBRARY, w)
        top = set(LIBRARY.assign(_s=s).nlargest(TOP_N, "_s")["compound_id"])
        scen.append({
            "perturbation": f"{comp} {tag}",
            "top200_overlap": len(base_top & top) / TOP_N,
            "spearman": spearmanr(base_scores, s).statistic,
        })

# Two structural variants worth testing, not just marginal nudges.
for name, kw in [("equal weights", {k: 1.0 for k in ACTIVE_WEIGHTS}),
                 ("no confidence multiplier", None)]:
    if kw is None:
        s, _ = composite_score(LIBRARY, BASE_WEIGHTS, apply_confidence=False)
    else:
        s, _ = composite_score(LIBRARY, kw)
    top = set(LIBRARY.assign(_s=s).nlargest(TOP_N, "_s")["compound_id"])
    scen.append({"perturbation": name,
                 "top200_overlap": len(base_top & top) / TOP_N,
                 "spearman": spearmanr(base_scores, s).statistic})

SENSITIVITY = pd.DataFrame(scen).sort_values("top200_overlap")
print(f"Weight sensitivity — retention of the top {TOP_N}")
print("=" * 70)
print(SENSITIVITY.round(3).to_string(index=False))
print("=" * 70)
worst = SENSITIVITY.iloc[0]
print(f"Worst case: '{worst['perturbation']}' retains "
      f"{worst['top200_overlap']:.0%} of the top {TOP_N} "
      f"(Spearman {worst['spearman']:.3f})")
print(f"Median retention across all perturbations: "
      f"{SENSITIVITY['top200_overlap'].median():.0%}")
if SENSITIVITY["top200_overlap"].median() > 0.7:
    print("\n  The ranking is driven mainly by the data. The specific weights "
          "matter\n  less than the components chosen — which is the property a "
          "composite\n  score needs before anyone orders compounds against it.")
else:
    print("\n  WARNING: the ranking is weight-sensitive. Treat the exact order "
          "as soft\n  and prefer selecting across the whole top slice rather "
          "than the top 20.")

## 4.4 Diversity-aware selection

Phase 2 established that this library carries essentially no analogue series — 92% of Butina
clusters are singletons. Cluster-based picking would therefore be close to a no-op.

Selection instead walks the ranked list and accepts a compound only if its ECFP4 Tanimoto to
**every already-accepted hit** is below a ceiling. This is a direct guarantee on the output set:
no two selected hits exceed the similarity threshold, whatever the cluster structure happens to be.

The reason this matters is statistical rather than aesthetic. Fifty compounds sharing one
chemotype is one hypothesis tested fifty times — if the chemotype is wrong, every compound fails
together. Fifty diverse compounds are fifty partly independent tests, so a failure is informative
about that chemotype alone.

In [ ]:
DIVERSITY_TANIMOTO_MAX = 0.50
N_HITS = 50
CANDIDATE_POOL = 3000

# --- Positive control: known actives present in the deck --------------------
# Phase 1's leakage audit flagged any library compound that is also a ChEMBL
# training compound. Those must be handled in two separate ways, and conflating
# them would be a serious error:
#
#   * As a POSITIVE CONTROL they are the best available evidence that the
#     pipeline works -- if a known A2A ligand sitting in the deck does not rank
#     near the top out of 48,489, something is wrong.
#   * As HITS they are worthless. The model was trained on them, so a high
#     score is memorisation, not prediction; and they are published compounds,
#     so they carry no novelty or IP value.
#
# They are therefore REPORTED as a control and EXCLUDED from the hit list.
POSITIVE_CONTROLS = LIBRARY[LIBRARY["in_chembl_trainset"]].copy()
print("POSITIVE CONTROL — known ChEMBL actives present in the library")
print("=" * 74)
if len(POSITIVE_CONTROLS):
    POSITIVE_CONTROLS = POSITIVE_CONTROLS.sort_values("composite", ascending=False)
    for r in POSITIVE_CONTROLS.itertuples():
        pct = 100 * (r.rank / len(LIBRARY))
        print(f"  {r.compound_id}")
        print(f"    {r.std_smiles}")
        print(f"    composite rank {int(r.rank):,} of {len(LIBRARY):,}  "
              f"(top {pct:.4f}%)")
        print(f"    ML {r.ml_score:.3f} | Tanimoto to nearest active "
              f"{r.nn_sim_ecfp4:.2f} | Vina "
              + (f"{r.vina_score:.2f} kcal/mol" if np.isfinite(r.vina_score)
                 else "not docked"))
    best_rank = int(POSITIVE_CONTROLS["rank"].min())
    print(f"\n  Best known active ranks {best_rank:,} of {len(LIBRARY):,} "
          f"(top {100 * best_rank / len(LIBRARY):.4f}%).")
    if best_rank <= len(LIBRARY) * 0.001:
        print("  VERDICT: PASS — the pipeline recovers a known ligand in the "
              "top 0.1%.")
    else:
        print("  VERDICT: WEAK — a known ligand does not rank highly. Treat the "
              "hit list with caution.")
    print("\n  These compounds are now EXCLUDED from the hit list: the model "
          "saw them\n  in training, so their scores are memorisation rather "
          "than prediction.")
else:
    print("  No library compound is in the ChEMBL training set.")
    print("  The screen is fully prospective — but there is correspondingly no")
    print("  internal positive control, so the hit list rests entirely on the")
    print("  held-out metrics from Phase 3.")
print("=" * 74 + "\n")

# --- Greedy diversity selection over the NOVEL compounds only ---------------
eligible = LIBRARY[~LIBRARY["in_chembl_trainset"]]
pool_src = eligible.nlargest(CANDIDATE_POOL, "composite")
pool = pool_src.reset_index(drop=True)
pool_rows = [LIBRARY.index.get_loc(i) for i in pool_src.index]
pool_fps = [LIB_FP["ecfp4"]["bitvects"][i] for i in pool_rows]
print(f"Candidate pool drawn from {len(eligible):,} novel compounds "
      f"({int(LIBRARY['in_chembl_trainset'].sum())} known actives excluded)")

selected, selected_fps, rejected_similar = [], [], 0
for i in range(len(pool)):
    if len(selected) >= N_HITS:
        break
    if selected_fps:
        sims = DataStructs.BulkTanimotoSimilarity(pool_fps[i], selected_fps)
        if max(sims) >= DIVERSITY_TANIMOTO_MAX:
            rejected_similar += 1
            continue
    selected.append(i)
    selected_fps.append(pool_fps[i])

HITS = pool.iloc[selected].copy().reset_index(drop=True)
HITS["hit_rank"] = np.arange(1, len(HITS) + 1)

print(f"Diversity selection: Tanimoto ceiling {DIVERSITY_TANIMOTO_MAX} "
      f"between any two hits")
print(f"  candidate pool     {len(pool):,} (top by composite)")
print(f"  selected           {len(HITS)}")
print(f"  rejected as too similar to an earlier pick: {rejected_similar:,}")
print(f"  composite range    {HITS['composite'].min():.3f} - "
      f"{HITS['composite'].max():.3f}")

# Verify the guarantee rather than assuming the loop was right.
if len(HITS) > 1:
    pair_max = 0.0
    for i in range(len(selected_fps)):
        for j in range(i + 1, len(selected_fps)):
            pair_max = max(pair_max, DataStructs.TanimotoSimilarity(
                selected_fps[i], selected_fps[j]))
    print(f"\n  VERIFIED max pairwise Tanimoto among the {len(HITS)} hits: "
          f"{pair_max:.3f}  (ceiling {DIVERSITY_TANIMOTO_MAX})")
    print(f"  unique Murcko scaffolds among hits: "
          f"{HITS['murcko_scaffold'].nunique()} / {len(HITS)}")

## 4.5 The final prioritised hit list

Each hit carries a **rationale** built from its own values — not a generic label. The point is that
a medicinal chemist reading the table can see *why* a compound is there and disagree with a
specific reason, rather than being asked to trust a score.

In [ ]:
# --- Per-hit rationale -------------------------------------------------------
def build_rationale(r):
    """Human-readable justification assembled from this compound's own values."""
    bits = []
    if r.ml_score >= 0.80:
        bits.append(f"high ML probability ({r.ml_score:.2f})")
    elif r.ml_score >= BEST_THRESHOLD:
        bits.append(f"ML probability {r.ml_score:.2f} (above threshold)")
    else:
        bits.append(f"ML probability {r.ml_score:.2f} (below threshold; carried "
                    f"by other evidence)")

    s = r.nn_sim_ecfp4
    if s >= 0.70:
        bits.append(f"close analogue of a known ligand (T={s:.2f}) — low novelty")
    elif NOVELTY_BAND[0] <= s <= NOVELTY_BAND[1]:
        bits.append(f"in the novelty band (T={s:.2f})")
    elif s < AD_THRESHOLD:
        bits.append(f"outside the applicability domain (T={s:.2f}) — speculative")
    else:
        bits.append(f"moderate similarity to known actives (T={s:.2f})")

    if getattr(r, "scaffold_hop_candidate", False):
        bits.append("scaffold-hop candidate (pharmacophore match, novel core)")
    if USE_DOCKING and np.isfinite(getattr(r, "vina_score", np.nan)):
        bits.append(f"Vina {r.vina_score:.1f} kcal/mol "
                    f"(LE {r.vina_le:.2f} kcal/mol/HA)")
    if r.prediction_confidence != "supported":
        bits.append(f"confidence: {r.prediction_confidence.replace('_', ' ')}")
    bits.append(f"MW {r.mw:.0f}, cLogP {r.clogp:.1f}, QED {r.qed:.2f}")
    return "; ".join(bits)


HITS["rationale"] = [build_rationale(r) for r in HITS.itertuples()]

FINAL_COLUMNS = [
    "hit_rank", "compound_id", "std_smiles", "composite",
    "ml_score", "ml_uncertainty", "nn_sim_ecfp4", "nn_sim_fcfp4",
    "vina_score", "vina_le", "s_novelty", "prediction_confidence",
    "in_applicability_domain", "scaffold_hop_candidate",
    "mw", "clogp", "tpsa", "hbd", "hba", "rotb", "fsp3", "qed",
    "heavy_atoms", "cluster", "murcko_scaffold", "real_analogs_url",
    "rationale",
]
FINAL_COLUMNS = [c for c in FINAL_COLUMNS if c in HITS.columns]
FINAL_TABLE = HITS[FINAL_COLUMNS].copy()

print(f"Final prioritised hit list — {len(FINAL_TABLE)} compounds, "
      f"{len(FINAL_COLUMNS)} columns\n")
display(FINAL_TABLE.head(12).round(3))

print("\nTop 5 with full rationale")
print("=" * 78)
for r in FINAL_TABLE.head(5).itertuples():
    print(f"\n  #{r.hit_rank}  {r.compound_id}   composite {r.composite:.3f}")
    print(f"     {r.std_smiles}")
    for line in textwrap.wrap(r.rationale, 72):
        print(f"     {line}")

In [ ]:
# --- Figure 15: the funnel and the hit profile ------------------------------
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))

# (a) the screening funnel
ax = axes[0]
stages = [("Library\ninput", PHASE1_METRICS["library_input"]),
          ("Curated", PHASE1_METRICS["library_curated"]),
          ("Lead-like", PHASE1_METRICS["library_final"]),
          ("In applic.\ndomain", int(LIBRARY["in_applicability_domain"].sum())),
          ("Pre-dock\nslice", len(DOCK_SLICE)),
          ("Final\nhits", len(HITS))]
yp = np.arange(len(stages))[::-1]
vals = [v for _, v in stages]
cols = [SEQ_BLUE[min(2 + i, 9)] for i in range(len(stages))]
ax.barh(yp, vals, color=cols, height=0.62, edgecolor=SURFACE, linewidth=2)
for y_, v in zip(yp, vals):
    ax.text(v * 1.35, y_, f"{v:,}", va="center", fontsize=8.5,
            color=INK["primary"], fontweight="600")
ax.set_yticks(yp); ax.set_yticklabels([s for s, _ in stages], fontsize=8)
ax.set_xscale("log"); ax.set_xlim(10, max(vals) * 6)
ax.set_xlabel("Compounds (log scale)")
ax.set_title("The screening funnel", loc="left")
ax.grid(axis="x", alpha=0.7); ax.grid(axis="y", visible=False)

# (b) weight sensitivity
ax = axes[1]
sv = SENSITIVITY.sort_values("top200_overlap")
yp = np.arange(len(sv))[::-1]
ax.barh(yp, sv["top200_overlap"], color=PALETTE["library"], height=0.6,
        edgecolor=SURFACE, linewidth=1.5)
for y_, v in zip(yp, sv["top200_overlap"]):
    ax.text(v + 0.015, y_, f"{v:.0%}", va="center", fontsize=8,
            color=INK["primary"], fontweight="600")
ax.axvline(0.7, color=INK["secondary"], ls="--", lw=1.2)
ax.set_yticks(yp); ax.set_yticklabels(sv["perturbation"], fontsize=7.5)
ax.set_xlabel(f"Fraction of the top {TOP_N} retained")
ax.set_xlim(0, 1.12)
ax.set_title("Weight sensitivity\n(dashed = 70% stability floor)", loc="left")
ax.grid(axis="x", alpha=0.7); ax.grid(axis="y", visible=False)

# (c) where the hits sit in score/novelty space
ax = axes[2]
samp = LIBRARY.sample(min(15000, len(LIBRARY)), random_state=CFG.random_seed)
ax.scatter(samp["nn_sim_ecfp4"], samp["ml_score"], s=4, alpha=0.12,
           color=PALETTE["library"], edgecolors="none", rasterized=True,
           label=f"library (n={len(samp):,})")
ax.scatter(HITS["nn_sim_ecfp4"], HITS["ml_score"], s=42, alpha=0.95,
           color=PALETTE["active"], edgecolors=SURFACE, linewidths=1.2,
           zorder=5, label=f"selected hits (n={len(HITS)})")
ax.axvspan(*NOVELTY_BAND, color="#ebeae6", zorder=0)
ax.annotate("novelty band", (np.mean(NOVELTY_BAND), 0.03), fontsize=8,
            ha="center", color=INK["secondary"], fontweight="600")
ax.set_xlabel("ECFP4 similarity to nearest known active")
ax.set_ylabel("ML predicted probability")
ax.set_title("Where the hits came from", loc="left")
ax.legend(loc="lower right", markerscale=1.4); ax.grid(alpha=0.7)

# (d) hit property profile vs the library
ax = axes[3]
props = ["mw", "clogp", "tpsa", "qed", "fsp3"]
lib_med = [LIBRARY[p].median() for p in props]
hit_med = [HITS[p].median() for p in props]
norm_l = [1.0] * len(props)
norm_h = [h / l if abs(l) > 1e-9 else 1.0 for h, l in zip(hit_med, lib_med)]
xp = np.arange(len(props))
ax.bar(xp - 0.19, norm_l, width=0.36, color=PALETTE["library"],
       edgecolor=SURFACE, linewidth=2, label="library median (=1)")
ax.bar(xp + 0.19, norm_h, width=0.36, color=PALETTE["active"],
       edgecolor=SURFACE, linewidth=2, label="hit median (relative)")
for x_, h, raw in zip(xp, norm_h, hit_med):
    ax.text(x_ + 0.19, h, f"{raw:.2f}", ha="center", va="bottom", fontsize=7.5,
            color=INK["primary"], fontweight="600")
ax.axhline(1.0, color=INK["muted"], lw=1, ls="--", zorder=0)
ax.set_xticks(xp); ax.set_xticklabels(props, fontsize=8.5)
ax.set_ylabel("Hit median / library median")
ax.set_title("Are the hits property-typical?\n(values = raw hit medians)", loc="left")
ax.legend(loc="upper right"); ax.grid(alpha=0.7)

fig.suptitle("Hit selection — funnel, stability, provenance, and property profile",
             x=0.01, ha="left", fontsize=12, fontweight="600")
fig.tight_layout()
save_fig(fig, "fig15_hit_selection")
plt.show()

In [ ]:
# --- Figure 16: the hits themselves -----------------------------------------
n_show = min(20, len(HITS))
mols = [Chem.MolFromSmiles(s) for s in HITS["std_smiles"].head(n_show)]
legends = []
for r in HITS.head(n_show).itertuples():
    parts = [f"#{r.hit_rank}  {r.compound_id}",
             f"score {r.composite:.3f} | ML {r.ml_score:.2f} | T {r.nn_sim_ecfp4:.2f}"]
    if USE_DOCKING and np.isfinite(getattr(r, "vina_score", np.nan)):
        parts.append(f"Vina {r.vina_score:.1f} kcal/mol")
    parts.append(f"MW {r.mw:.0f} | cLogP {r.clogp:.1f} | QED {r.qed:.2f}")
    legends.append("\n".join(parts))

img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 260),
                           legends=legends, useSVG=False)
grid_path = CFG.paths["figures"] / "fig16_top_hits_structures.png"
with open(grid_path, "wb") as fh:
    fh.write(img.data if hasattr(img, "data") else img)
print(f"  saved  {grid_path}")
img

## 4.6 Persisted deliverables and the Phase 4 validation gate

In [ ]:
# --- Persist Phase 4 ---------------------------------------------------------
FINAL_TABLE.to_csv(CFG.paths["tables"] / "FINAL_prioritised_hits.csv", index=False)
SENSITIVITY.to_csv(CFG.paths["tables"] / "weight_sensitivity.csv", index=False)
(LIBRARY.nlargest(5000, "composite")
 [[c for c in FINAL_COLUMNS if c in LIBRARY.columns and c != "hit_rank"]]
 .to_csv(CFG.paths["tables"] / "top5000_composite_ranked.csv", index=False))
LIBRARY.to_csv(CFG.paths["curated"] / "library_phase4_final.csv", index=False)

# An SDF of the hits, for direct import into a modelling package.
sdf_path = CFG.paths["tables"] / "FINAL_prioritised_hits.sdf"
w = Chem.SDWriter(str(sdf_path))
for r in FINAL_TABLE.itertuples():
    m = Chem.MolFromSmiles(r.std_smiles)
    if m is None:
        continue
    m.SetProp("_Name", str(r.compound_id))
    for c in FINAL_COLUMNS:
        if c in ("std_smiles",):
            continue
        val = getattr(r, c, None)
        if val is not None and not (isinstance(val, float) and not np.isfinite(val)):
            m.SetProp(c, str(val))
    w.write(m)
w.close()
print(f"  wrote {sdf_path}")

PHASE4_METRICS = {
    "docking_available":  bool(DOCKING_AVAILABLE),
    "docking_reliable":   bool(DOCKING_RELIABLE),
    "docking_used_in_composite": bool(USE_DOCKING),
    "docking_skip_reason": DOCK_SKIP_REASON,
    "redock_rmsd":        (None if REDOCK_RMSD is None or not np.isfinite(REDOCK_RMSD)
                           else float(REDOCK_RMSD)),
    "n_docked":           int(LIBRARY["vina_score"].notna().sum()),
    "base_weights":       BASE_WEIGHTS,
    "active_weights":     ACTIVE_WEIGHTS,
    "diversity_tanimoto_max": DIVERSITY_TANIMOTO_MAX,
    "candidate_pool":     CANDIDATE_POOL,
    "n_hits":             int(len(HITS)),
    "hit_unique_scaffolds": int(HITS["murcko_scaffold"].nunique()),
    "sensitivity_median_top200_overlap": float(SENSITIVITY["top200_overlap"].median()),
    "sensitivity_worst_top200_overlap": float(SENSITIVITY["top200_overlap"].min()),
    "hits_in_novelty_band": int(HITS["in_novelty_band"].sum()),
    "hits_scaffold_hop":  int(HITS["scaffold_hop_candidate"].sum()),
    "hits_out_of_domain": int((HITS["prediction_confidence"] == "out_of_domain").sum()),
    "positive_controls_in_library": int(len(POSITIVE_CONTROLS)),
    "positive_control_best_rank": (None if not len(POSITIVE_CONTROLS)
                                   else int(POSITIVE_CONTROLS["rank"].min())),
    "positive_control_best_percentile": (
        None if not len(POSITIVE_CONTROLS)
        else float(100 * POSITIVE_CONTROLS["rank"].min() / len(LIBRARY))),
    "max_hit_similarity_to_known": float(HITS["nn_sim_ecfp4"].max()),
}
(CFG.paths["metrics"] / "phase4_metrics.json").write_text(
    json.dumps(PHASE4_METRICS, indent=2, default=str))
RUN_STAMP["phase4_metrics"] = PHASE4_METRICS
RUN_STAMP["phase"] = 4
(CFG.paths["metrics"] / "run_manifest.json").write_text(
    json.dumps(RUN_STAMP, indent=2, default=str))
print("Phase 4 artefacts written.")

In [ ]:
# --- Phase 4 validation gate -------------------------------------------------
def validate_phase4():
    checks, failures = [], []

    def check(name, cond, detail=""):
        ok = bool(cond); checks.append((name, ok, detail))
        if not ok: failures.append(f"{name} — {detail}")

    # Scoring integrity
    check("Every compound has a composite score",
          LIBRARY["composite"].notna().all(),
          f"{int(LIBRARY['composite'].isna().sum())} missing")
    check("Composite scores are finite",
          np.isfinite(LIBRARY["composite"]).all(), "NaN/inf present")
    check("Active weights sum to 1",
          abs(sum(ACTIVE_WEIGHTS.values()) - 1.0) < 1e-9,
          f"sum = {sum(ACTIVE_WEIGHTS.values()):.6f}")
    check("Docking excluded unless the redock control passed",
          USE_DOCKING == (DOCKING_AVAILABLE and DOCKING_RELIABLE
                          and LIBRARY['vina_score'].notna().any()),
          "docking gate inconsistent")
    if USE_DOCKING:
        check("Redock RMSD within threshold",
              REDOCK_RMSD <= DOCK_CFG["redock_rmsd_max"],
              f"{REDOCK_RMSD:.2f} A")
    check("Ranks are unique and complete",
          LIBRARY["rank"].nunique() == len(LIBRARY), "duplicate ranks")

    # The diversity guarantee — the core claim of 4.4
    if len(HITS) > 1:
        mx = 0.0
        for i in range(len(selected_fps)):
            for j in range(i + 1, len(selected_fps)):
                mx = max(mx, DataStructs.TanimotoSimilarity(
                    selected_fps[i], selected_fps[j]))
        check("No two hits exceed the Tanimoto ceiling",
              mx < DIVERSITY_TANIMOTO_MAX, f"max pairwise {mx:.3f}")
    check("Hits are unique compounds",
          not HITS["compound_id"].duplicated().any(), "duplicate hit")
    check("Hits are ordered by composite score",
          HITS["composite"].is_monotonic_decreasing, "hit order broken")
    check("Every hit carries a rationale",
          HITS["rationale"].notna().all() and (HITS["rationale"].str.len() > 20).all(),
          "empty rationale")
    check("Hit SMILES all re-parse",
          all(Chem.MolFromSmiles(s) is not None for s in HITS["std_smiles"]),
          "unparseable hit SMILES")
    check("Hits drawn from the curated library",
          set(HITS["compound_id"]) <= set(LIBRARY["compound_id"]),
          "hit not in the library")
    check("No hit is a ChEMBL training compound",
          not HITS["in_chembl_trainset"].any(),
          f"{int(HITS['in_chembl_trainset'].sum())} known actives leaked into "
          f"the hit list")
    check("No hit exceeds the analogue-similarity threshold",
          (HITS["nn_sim_ecfp4"] < 0.95).all(),
          f"max Tanimoto to a known active {HITS['nn_sim_ecfp4'].max():.3f}")

    # Stability
    check("Ranking is not weight-degenerate",
          SENSITIVITY["top200_overlap"].median() > 0.5,
          f"median overlap {SENSITIVITY['top200_overlap'].median():.2f}")

    for p in (CFG.paths["tables"] / "FINAL_prioritised_hits.csv",
              CFG.paths["tables"] / "FINAL_prioritised_hits.sdf",
              CFG.paths["metrics"] / "phase4_metrics.json"):
        check(f"Artefact exists: {p.name}", p.exists(), str(p))
    n_fig = len(list(CFG.paths["figures"].glob("*.png")))
    check("All project figures written", n_fig >= 16, f"{n_fig} PNGs")

    print("Phase 4 validation gate")
    print("=" * 76)
    for name, ok, detail in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  ({detail})" if not ok else ""))
    print("=" * 76)
    print(f"{sum(1 for _, ok, _ in checks if ok)}/{len(checks)} checks passed")
    if failures:
        raise AssertionError("Phase 4 validation FAILED:\n  - " + "\n  - ".join(failures))
    print("\nPhase 4 validation PASSED — the project is complete.")
    return True


validate_phase4()

In [ ]:
# --- Project summary ---------------------------------------------------------
m4 = PHASE4_METRICS
print("=" * 78)
print(f"PROJECT COMPLETE — {CFG.target_name} virtual hit finding")
print("=" * 78)
print(f"""
PIPELINE
  Enamine library input .......... {PHASE1_METRICS['library_input']:>8,}
  after curation + lead-likeness . {PHASE1_METRICS['library_final']:>8,}
  ChEMBL labelled training set ... {PHASE1_METRICS['chembl_labelled']:>8,}
    actives / inactives .......... {PHASE1_METRICS['chembl_actives']:>8,} / {PHASE1_METRICS['chembl_inactives']:,}
  prioritised hits delivered ..... {m4['n_hits']:>8,}

MODEL  (scaffold-disjoint, bootstrap 95% CI)
  {metrics_out['selected_representation']} / {metrics_out['selected_model']}
  ROC-AUC ........................ {fmt_ci(FINAL_METRICS, 'roc_auc')}
  at screening prevalence, PR-AUC  {fmt_ci(BENCH['strict']['model'], 'pr_auc')}
  nearest-neighbour baseline ..... {fmt_ci(BENCH['strict']['baseline'], 'pr_auc')}
  -> similarity was the stronger signal; both are in the composite

STRUCTURE-BASED ARM
  docking available .............. {m4['docking_available']}
  redocking control passed ....... {m4['docking_reliable']}
  used in the composite .......... {m4['docking_used_in_composite']}
  compounds docked ............... {m4['n_docked']:>8,}""")
if m4["redock_rmsd"] is not None:
    print(f"  redock RMSD .................... {m4['redock_rmsd']:.2f} A")
if m4["docking_skip_reason"]:
    print(f"  skip reason .................... {m4['docking_skip_reason'][:60]}")
print(f"""
RANKING
  active weights ................. {', '.join(f'{k}={v:.2f}' for k, v in ACTIVE_WEIGHTS.items())}
  top-200 stability (median) ..... {m4['sensitivity_median_top200_overlap']:.0%}
  worst-case stability ........... {m4['sensitivity_worst_top200_overlap']:.0%}

POSITIVE CONTROL
  known actives in the deck ...... {m4['positive_controls_in_library']:>8,}
  best rank of a known active .... {m4['positive_control_best_rank'] or 'n/a'}  (top {m4['positive_control_best_percentile'] or float('nan'):.4f}%)
  -> excluded from the hit list (memorisation, and no novelty)

HIT SET
  compounds ...................... {m4['n_hits']:>8,}
  unique Murcko scaffolds ........ {m4['hit_unique_scaffolds']:>8,}
  max pairwise Tanimoto .......... {'<':>8} {DIVERSITY_TANIMOTO_MAX}
  in the novelty band ............ {m4['hits_in_novelty_band']:>8,}
  scaffold-hop candidates ........ {m4['hits_scaffold_hop']:>8,}
  outside applicability domain ... {m4['hits_out_of_domain']:>8,}

DELIVERABLES
  results/tables/FINAL_prioritised_hits.csv
  results/tables/FINAL_prioritised_hits.sdf
  results/figures/fig01-16*.png
  results/metrics/run_manifest.json
""")
print("=" * 78)

---

## Project complete

### What this notebook produced

| Deliverable | Contents |
|---|---|
| `FINAL_prioritised_hits.csv` | the ranked hit list — identifiers, SMILES, all component scores, uncertainty, descriptors, cluster, scaffold, REAL enumeration URL, and a per-compound rationale |
| `FINAL_prioritised_hits.sdf` | the same hits as an SDF with all properties as SD tags, for direct import into a modelling package |
| `top5000_composite_ranked.csv` | the wider ranked slice, if you want to select more than 50 |
| `library_phase4_final.csv` | all 48,489 compounds with every score and flag computed across four phases |
| `weight_sensitivity.csv` | how much the ranking moves when the weights move |
| `fig01`–`fig16` | curation, chemical space, clustering, model performance, decoy benchmark, hit selection, and the hit structures |
| `run_manifest.json` | environment, versions, seed, full configuration, and every phase's metrics |

### What the project found, in order

1. **The library is lead-like and exceptionally diverse** — 90.7% survived curation; 92% of Butina
   clusters are singletons. Enamine has already diversity-selected it, which changed the selection
   strategy from cluster-picking to a direct Tanimoto ceiling.
2. **Enamine's cLogP disagrees materially with RDKit's** (r = 0.72 against 0.98 for MW). Filtering
   on the vendor value and modelling on the RDKit value would have been a silent inconsistency.
3. **The ChEMBL A<sub>2A</sub> set is 82.7% active** — publication bias. This caps EF@1% at 1.21 and
   pinned every model to that ceiling, making the standard virtual-screening metrics unable to
   distinguish any two models.
4. **At realistic screening prevalence, nearest-active similarity beat the ML model** by 0.14–0.15
   PR-AUC on property-matched decoys. The composite score weights similarity accordingly instead of
   deferring to the model.
5. **Random splitting inflates ROC-AUC by +0.051** on this dataset — the cost of the split choice
   most published benchmarks do not report.

### Honest limitations

* **Decoys are presumed inactive, not measured inactive.** Some will be genuine binders, so the
  reported enrichments are mild under-estimates.
* **The strict decoy set is confounded** — decoys were selected for low similarity to actives,
  which is the similarity baseline's own scoring function. The relaxed set brackets this.
* **Decoy property matching is imperfect**: median decoy MW sits ~50 Da below median active MW,
  because a lead-like library cannot supply matches for the larger ChEMBL actives. This biases
  slightly toward the model.
* **A single conformer per ligand** is docked. Rigid-ligand Vina with one ETKDG conformer is
  standard for a screening funnel but is not a substitute for careful pose refinement on the
  compounds you actually order.
* **No experimental validation.** Every number here is computational. The output is a
  testable hypothesis set, not a set of confirmed binders.

### Suggested next steps

1. Order the top 20–30 hits and run a radioligand binding assay — this is the only thing that
   converts the output into evidence.
2. For any confirmed hit, use the `AnalogsFromREAL` URL to enumerate its REAL-space neighbourhood
   for a rapid SAR expansion, which is the reason for screening a REAL-backed library in the first
   place.
3. Revisit the docking arm with Meeko installed and a properly protonated receptor, and confirm the
   redocking control passes before trusting any pose.
4. If experimental actives come back, fold them into the training set and re-run Phases 3–4 — the
   notebook is built to be re-run end to end.